# 10 — Raw Static-Arbitrage Diagnostics for the Empirical SPY Implied-Volatility Surface

## Purpose

This notebook takes the surface-grade SPY implied-volatility handoff from Notebook 09 and tests whether the filtered observations form a coherent raw option surface.

Notebook 09 focused on the validity of individual option observations: quote quality, implied-volatility inversion, stability filtering, and surface-grade selection.

Notebook 10 moves one level higher. It asks whether the surviving observations are jointly consistent across strike and maturity.

The goal is not to prove that the market surface is arbitrage-free. The goal is to identify, localize, and severity-rank static-arbitrage defects before any smoothing, repair, interpolation, or model calibration is attempted.

## Central Research Question

Can the filtered SPY implied-volatility observations from Notebook 09 be treated as a coherent empirical option surface, or do they still contain static-arbitrage inconsistencies across strike and maturity?

## Scope of This Notebook

This notebook diagnoses the raw empirical surface only.

It does not:

- fit SVI or SSVI;
- repair the volatility surface;
- smooth noisy observations;
- calibrate Heston or Bates models;
- perform Monte Carlo pricing;
- compute hedging strategies;
- introduce macro factors or regime conditioning.

Those steps belong to later notebooks.

The purpose here is narrower and more foundational: determine whether the raw surface is structurally usable, and identify which observations or expiry slices require exclusion, downweighting, or repair.

## Theoretical Motivation

In an arbitrage-free market, option prices across strike and maturity cannot move independently. Even if each individual quote appears valid, the collection of prices must satisfy static consistency restrictions.

This notebook focuses on four main static-arbitrage diagnostics:

1. **Pointwise bounds**

   Each option price must lie within basic no-arbitrage lower and upper bounds.

2. **Strike monotonicity**

   Call prices should not increase as strike increases. Put prices should not decrease as strike increases.

3. **Strike convexity**

   Option prices should be convex in strike. Violations of convexity correspond to negative butterfly-spread values and may imply negative risk-neutral probability mass.

4. **Calendar consistency**

   At comparable moneyness, longer-dated total variance should not materially fall below shorter-dated total variance.

The main object of interest is therefore not only implied volatility itself, but the option-price and total-variance structure implied by the filtered volatility observations.

## Input

The notebook consumes only the Notebook 09 handoff panel.

Expected input:

- surface-grade SPY implied-volatility observations;
- expiry and maturity information;
- strikes and forwards;
- log-moneyness;
- selected implied volatility;
- total variance;
- selected option price or reconstructed price inputs;
- quote-quality and IV-stability metadata.

Notebook 10 does not reload raw option chains and does not redo implied-volatility inversion.

## Output

Notebook 10 produces a diagnostic layer for Notebook 11.

Expected outputs include:

- row-level raw surface diagnostic panel;
- pointwise bounds diagnostic summary;
- strike monotonicity violation ledger;
- butterfly convexity violation ledger;
- calendar / total-variance diagnostic table;
- expiry-level surface quality table;
- moneyness-bucket violation summary;
- severity-ranked violation ledger;
- broad Notebook 11 handoff panel;
- strict Notebook 11 handoff panel;
- final validation ledger and artifact manifest.

The broad handoff is intended for smoothing and repair experiments.

The strict handoff is intended for cleaner fitting, calibration, or robustness checks.

## Research Discipline

A filtered implied-volatility observation is not automatically a valid surface observation.

Notebook 09 answers:

> Is this individual option quote usable for implied-volatility construction?

Notebook 10 answers:

> Can these usable observations coexist as one coherent empirical surface?

That distinction is the reason this notebook exists.

The final conclusion should not be a simple claim that the surface is clean or arbitrage-free. The correct conclusion should specify which regions are reliable, which regions are noisy, and which regions require repair before being passed to SVI, SSVI, Heston, Bates, or any other model-based layer.

In [1]:
# ============================================================
# Notebook 10 setup, configuration, and diagnostic tolerances
# ============================================================

from __future__ import annotations

import json
import math
import os
import platform
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.8f}")


# ------------------------------------------------------------
# Notebook identity
# ------------------------------------------------------------

NOTEBOOK_ID = "10"
NOTEBOOK_SLUG = "raw_static_arbitrage_diagnostics_for_spy_iv_surface"
NOTEBOOK_NAME = f"{NOTEBOOK_ID}_{NOTEBOOK_SLUG}"

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DATE_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%d")

SNAPSHOT_TAG = f"n{NOTEBOOK_ID}_{RUN_TIMESTAMP_UTC}"


# ------------------------------------------------------------
# Project-root discovery
# ------------------------------------------------------------

def find_project_root(start: Optional[Path] = None) -> Path:
    """
    Locate the project root without hard-coding a machine-specific path.

    Search priority:
    1. current working directory and parents containing paths.txt;
    2. current working directory and parents containing outputs/;
    3. current working directory.
    """
    start_path = Path.cwd() if start is None else Path(start).resolve()

    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if (candidate / "paths.txt").exists():
            return candidate

    for candidate in candidates:
        if (candidate / "outputs").exists():
            return candidate

    return start_path


PROJECT_ROOT = find_project_root()

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
NOTEBOOK_OUTPUT_DIR = OUTPUT_ROOT / NOTEBOOK_NAME
PLOT_DIR = NOTEBOOK_OUTPUT_DIR / "plots"
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / "tables"
HANDOFF_DIR = NOTEBOOK_OUTPUT_DIR / "handoff"
MANIFEST_DIR = NOTEBOOK_OUTPUT_DIR / "manifest"

for directory in [OUTPUT_ROOT, NOTEBOOK_OUTPUT_DIR, PLOT_DIR, TABLE_DIR, HANDOFF_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Notebook 09 handoff input candidates
# ------------------------------------------------------------

N09_OUTPUT_CANDIDATE_DIRS = [
    OUTPUT_ROOT / "09_SPY Implied Volatility Inversion and Surface-Grade Filtering",
    OUTPUT_ROOT / "09_spy_implied_volatility_inversion_and_surface_grade_filtering",
    OUTPUT_ROOT / "09_spy_iv_inversion_and_surface_grade_filtering",
    OUTPUT_ROOT / "09",
    OUTPUT_ROOT,
]

N09_HANDOFF_FILENAME_PATTERNS = [
    "*n09*n10*handoff*.parquet",
    "*N09*N10*HANDOFF*.parquet",
    "*surface*grade*handoff*.parquet",
    "*surface_grade*.parquet",
    "*iv*handoff*.parquet",
    "*n09*n10*handoff*.csv",
    "*N09*N10*HANDOFF*.csv",
    "*surface*grade*handoff*.csv",
    "*surface_grade*.csv",
    "*iv*handoff*.csv",
]


# ------------------------------------------------------------
# Diagnostic tolerances
# ------------------------------------------------------------

@dataclass(frozen=True)
class DiagnosticTolerances:
    """
    Visible numerical tolerances for raw-surface diagnostics.

    These are intentionally separated from the diagnostic functions so that
    the research assumptions are auditable from the notebook itself.
    """

    # General numerical safety
    eps: float = 1.0e-12

    # Pointwise price-bound diagnostics
    price_abs_tol: float = 1.0e-6
    price_rel_tol: float = 1.0e-5

    # Strike monotonicity diagnostics
    monotonicity_abs_tol: float = 1.0e-5
    monotonicity_rel_tol: float = 1.0e-4

    # Butterfly / convexity diagnostics
    convexity_abs_tol: float = 1.0e-6
    convexity_rel_tol: float = 1.0e-4

    # Calendar / total-variance diagnostics
    total_variance_abs_tol: float = 1.0e-6
    total_variance_rel_tol: float = 1.0e-4

    # Moneyness matching for calendar comparisons
    moneyness_match_tol: float = 0.015

    # Minimum data requirements
    min_points_per_expiry: int = 7
    min_points_for_convexity: int = 5
    min_expiries_for_calendar: int = 3

    # Region definitions in log-moneyness
    atm_abs_log_moneyness: float = 0.025
    near_atm_abs_log_moneyness: float = 0.075
    wing_abs_log_moneyness: float = 0.150

    # Severity thresholds
    mild_violation_threshold: float = 1.0e-5
    moderate_violation_threshold: float = 1.0e-4
    severe_violation_threshold: float = 1.0e-3


TOL = DiagnosticTolerances()


# ------------------------------------------------------------
# Severity labels and handoff policy labels
# ------------------------------------------------------------

SEVERITY_ORDER = [
    "clean",
    "micro_noise",
    "mild_warning",
    "moderate_warning",
    "severe_violation",
    "fatal_for_slice",
]

RECOMMENDED_ACTIONS = [
    "keep_raw",
    "keep_with_downweighting",
    "repair_before_use",
    "exclude_from_calibration",
    "diagnostic_only",
]

FINAL_STATUS_OPTIONS = [
    "READY_FOR_11_WITH_WARNINGS",
    "READY_FOR_11_STRICT_SUBSET_ONLY",
    "NOT_READY_FOR_11",
    "DIAGNOSTIC_ONLY_NO_REPAIR_INPUT",
]


# ------------------------------------------------------------
# Run metadata
# ------------------------------------------------------------

RUN_METADATA: Dict[str, Any] = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_name": NOTEBOOK_NAME,
    "snapshot_tag": SNAPSHOT_TAG,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "run_date_utc": RUN_DATE_UTC,
    "project_root": str(PROJECT_ROOT),
    "output_dir": str(NOTEBOOK_OUTPUT_DIR),
    "plot_dir": str(PLOT_DIR),
    "table_dir": str(TABLE_DIR),
    "handoff_dir": str(HANDOFF_DIR),
    "manifest_dir": str(MANIFEST_DIR),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "diagnostic_tolerances": asdict(TOL),
}


print("=" * 90)
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Snapshot: {SNAPSHOT_TAG}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {NOTEBOOK_OUTPUT_DIR}")
print("=" * 90)

pd.DataFrame(
    {
        "setting": list(asdict(TOL).keys()),
        "value": list(asdict(TOL).values()),
    }
)

Notebook: 10_raw_static_arbitrage_diagnostics_for_spy_iv_surface
Snapshot: n10_20260705T200149Z
Project root: d:\Derivative Pricing Project v1.0+\V1.1
Output directory: d:\Derivative Pricing Project v1.0+\V1.1\outputs\10_raw_static_arbitrage_diagnostics_for_spy_iv_surface


,setting,value
0,eps,0.00000000
1,price_abs_tol,0.00000100
2,price_rel_tol,0.00001000
3,monotonicity_abs_tol,0.00001000
4,monotonicity_rel_tol,0.00010000
5,convexity_abs_tol,0.00000100
6,convexity_rel_tol,0.00010000
7,total_variance_abs_tol,0.00000100
8,total_variance_rel_tol,0.00010000
9,moneyness_match_tol,0.01500000


In [2]:
# ============================================================
# Load Notebook 09 handoff and validate input schema
# ============================================================

def read_paths_txt(paths_file: Path) -> Dict[str, str]:
    """
    Read a simple paths.txt file if available.

    Supports loose key-value formats such as:
        key=value
        key: value

    Lines without a key-value separator are ignored.
    """
    parsed: Dict[str, str] = {}

    if not paths_file.exists():
        return parsed

    for raw_line in paths_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        if "=" in line:
            key, value = line.split("=", 1)
        elif ":" in line:
            key, value = line.split(":", 1)
        else:
            continue

        key = key.strip()
        value = value.strip().strip('"').strip("'")

        if key and value:
            parsed[key] = value

    return parsed


def candidate_dirs_from_paths_txt(project_root: Path) -> List[Path]:
    """
    Extract possible data/output directories from paths.txt.
    """
    paths_file = project_root / "paths.txt"
    parsed = read_paths_txt(paths_file)

    candidates: List[Path] = []

    for key, value in parsed.items():
        lower_key = key.lower()
        lower_value = value.lower()

        looks_relevant = (
            "09" in lower_key
            or "n09" in lower_key
            or "handoff" in lower_key
            or "output" in lower_key
            or "surface" in lower_key
            or "iv" in lower_key
            or "09" in lower_value
            or "n09" in lower_value
            or "handoff" in lower_value
            or "surface" in lower_value
        )

        if looks_relevant:
            candidate = Path(value)

            if not candidate.is_absolute():
                candidate = project_root / candidate

            if candidate.exists():
                candidates.append(candidate if candidate.is_dir() else candidate.parent)

    return candidates


def collect_n09_handoff_candidates() -> List[Path]:
    """
    Search likely Notebook 09 output locations for a Notebook 10 handoff file.
    """
    search_dirs: List[Path] = []

    search_dirs.extend(N09_OUTPUT_CANDIDATE_DIRS)
    search_dirs.extend(candidate_dirs_from_paths_txt(PROJECT_ROOT))

    # Add direct descendants of outputs/ that look like Notebook 09 folders.
    if OUTPUT_ROOT.exists():
        for child in OUTPUT_ROOT.iterdir():
            if child.is_dir():
                name = child.name.lower()
                if name.startswith("09") or "n09" in name or "implied" in name or "surface" in name:
                    search_dirs.append(child)

    # Deduplicate while preserving order.
    deduped_dirs: List[Path] = []
    seen_dirs = set()

    for directory in search_dirs:
        directory = Path(directory)

        try:
            resolved = directory.resolve()
        except Exception:
            resolved = directory

        if resolved not in seen_dirs and directory.exists() and directory.is_dir():
            deduped_dirs.append(directory)
            seen_dirs.add(resolved)

    candidates: List[Path] = []

    for directory in deduped_dirs:
        for pattern in N09_HANDOFF_FILENAME_PATTERNS:
            candidates.extend(directory.rglob(pattern))

    # Deduplicate files.
    deduped_files: List[Path] = []
    seen_files = set()

    for file_path in candidates:
        if not file_path.is_file():
            continue

        try:
            resolved = file_path.resolve()
        except Exception:
            resolved = file_path

        if resolved not in seen_files:
            deduped_files.append(file_path)
            seen_files.add(resolved)

    return deduped_files


def score_handoff_candidate(file_path: Path) -> Tuple[int, float]:
    """
    Rank candidate files by how likely they are to be the Notebook 09 -> 10 handoff.
    Higher score is better.
    """
    text = str(file_path).lower()
    name = file_path.name.lower()

    score = 0

    if "n09" in text or "09" in text:
        score += 20
    if "n10" in text or "10" in name:
        score += 15
    if "handoff" in text:
        score += 25
    if "surface" in text:
        score += 12
    if "grade" in text:
        score += 10
    if "iv" in text or "implied" in text:
        score += 8
    if file_path.suffix.lower() == ".parquet":
        score += 8
    if file_path.suffix.lower() == ".csv":
        score += 3
    if "raw" in text:
        score -= 3
    if "diagnostic" in text:
        score -= 5
    if NOTEBOOK_NAME.lower() in text:
        score -= 20

    try:
        mtime = file_path.stat().st_mtime
    except Exception:
        mtime = 0.0

    return score, mtime


def load_table(file_path: Path) -> pd.DataFrame:
    """
    Load a parquet or CSV handoff table.
    """
    suffix = file_path.suffix.lower()

    if suffix == ".parquet":
        return pd.read_parquet(file_path)

    if suffix == ".csv":
        return pd.read_csv(file_path)

    raise ValueError(f"Unsupported handoff file type: {file_path.suffix}")


def find_best_n09_handoff() -> Path:
    """
    Select the best available Notebook 09 handoff file.
    """
    candidates = collect_n09_handoff_candidates()

    if not candidates:
        searched_dirs = [str(p) for p in N09_OUTPUT_CANDIDATE_DIRS if Path(p).exists()]
        raise FileNotFoundError(
            "No Notebook 09 handoff file was found.\n"
            "Expected a parquet or CSV file with terms such as n09, n10, handoff, surface, grade, or iv.\n"
            f"Existing searched directories:\n{json.dumps(searched_dirs, indent=2)}"
        )

    ranked = sorted(candidates, key=score_handoff_candidate, reverse=True)

    candidate_audit = pd.DataFrame(
        {
            "rank": range(1, len(ranked) + 1),
            "path": [str(p) for p in ranked],
            "filename": [p.name for p in ranked],
            "suffix": [p.suffix.lower() for p in ranked],
            "score": [score_handoff_candidate(p)[0] for p in ranked],
            "modified_utc": [
                datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                for p in ranked
            ],
            "size_mb": [p.stat().st_size / 1_000_000 for p in ranked],
        }
    )

    print("Candidate Notebook 09 handoff files:")
    display(candidate_audit.head(15))

    return ranked[0]


def resolve_column(df: pd.DataFrame, aliases: Sequence[str], required: bool = True) -> Optional[str]:
    """
    Resolve a canonical conceptual field from a list of possible column aliases.
    """
    column_lookup = {col.lower(): col for col in df.columns}

    for alias in aliases:
        if alias.lower() in column_lookup:
            return column_lookup[alias.lower()]

    normalized_lookup = {
        col.lower().replace(" ", "_").replace("-", "_"): col
        for col in df.columns
    }

    for alias in aliases:
        normalized_alias = alias.lower().replace(" ", "_").replace("-", "_")
        if normalized_alias in normalized_lookup:
            return normalized_lookup[normalized_alias]

    if required:
        raise KeyError(
            f"Could not resolve required column from aliases: {aliases}\n"
            f"Available columns: {list(df.columns)}"
        )

    return None


# ------------------------------------------------------------
# Load best handoff file
# ------------------------------------------------------------

N09_HANDOFF_PATH = find_best_n09_handoff()
n09_handoff_raw = load_table(N09_HANDOFF_PATH)

print("=" * 90)
print("Selected Notebook 09 handoff:")
print(N09_HANDOFF_PATH)
print(f"Loaded shape: {n09_handoff_raw.shape[0]:,} rows x {n09_handoff_raw.shape[1]:,} columns")
print("=" * 90)


# ------------------------------------------------------------
# Resolve canonical schema
# ------------------------------------------------------------

COLUMN_ALIASES: Dict[str, Sequence[str]] = {
    "expiry": [
        "expiry",
        "expiration",
        "expiration_date",
        "expiry_date",
        "maturity_date",
    ],
    "tau_years": [
        "tau_years",
        "tau",
        "T",
        "ttm",
        "time_to_maturity",
        "years_to_expiry",
        "dte_years",
    ],
    "dte_calendar": [
        "dte_calendar",
        "dte",
        "days_to_expiry",
        "calendar_dte",
        "days_to_maturity",
    ],
    "strike": [
        "strike",
        "K",
        "option_strike",
    ],
    "forward": [
        "forward",
        "F",
        "forward_price",
        "synthetic_forward",
        "implied_forward",
    ],
    "spot": [
        "spot",
        "S",
        "spot_price",
        "underlying_price",
        "underlying_last",
    ],
    "log_moneyness": [
        "log_moneyness",
        "k",
        "ln_moneyness",
        "log_forward_moneyness",
        "log_strike_forward",
    ],
    "selected_iv": [
        "selected_iv",
        "surface_iv",
        "mid_iv",
        "iv",
        "implied_vol",
        "implied_volatility",
        "stable_iv",
        "chosen_iv",
    ],
    "total_variance": [
        "total_variance",
        "w",
        "variance_total",
        "iv_total_variance",
        "selected_total_variance",
    ],
    "option_type": [
        "option_type",
        "right",
        "cp_flag",
        "type",
        "selected_option_type",
    ],
    "selected_price": [
        "selected_price",
        "mid",
        "mid_price",
        "option_mid",
        "selected_mid",
        "price",
        "mark",
    ],
    "bid": [
        "bid",
        "bid_price",
        "selected_bid",
    ],
    "ask": [
        "ask",
        "ask_price",
        "selected_ask",
    ],
    "rate": [
        "rate",
        "r",
        "risk_free_rate",
        "zero_rate",
        "discount_rate",
    ],
    "dividend_yield": [
        "dividend_yield",
        "q",
        "div_yield",
        "continuous_dividend_yield",
    ],
}

required_concepts = [
    "expiry",
    "tau_years",
    "strike",
    "selected_iv",
]

optional_concepts = [
    "dte_calendar",
    "forward",
    "spot",
    "log_moneyness",
    "total_variance",
    "option_type",
    "selected_price",
    "bid",
    "ask",
    "rate",
    "dividend_yield",
]

resolved_columns: Dict[str, Optional[str]] = {}

for concept in required_concepts:
    resolved_columns[concept] = resolve_column(n09_handoff_raw, COLUMN_ALIASES[concept], required=True)

for concept in optional_concepts:
    resolved_columns[concept] = resolve_column(n09_handoff_raw, COLUMN_ALIASES[concept], required=False)

schema_resolution = pd.DataFrame(
    {
        "concept": list(resolved_columns.keys()),
        "resolved_column": list(resolved_columns.values()),
        "required": [concept in required_concepts for concept in resolved_columns.keys()],
    }
)

display(schema_resolution)


# ------------------------------------------------------------
# Create standardized working copy
# ------------------------------------------------------------

surface_raw = n09_handoff_raw.copy()

for concept, col in resolved_columns.items():
    if col is not None and concept not in surface_raw.columns:
        surface_raw[concept] = surface_raw[col]

surface_raw["expiry"] = pd.to_datetime(surface_raw["expiry"], errors="coerce")
surface_raw["tau_years"] = pd.to_numeric(surface_raw["tau_years"], errors="coerce")
surface_raw["strike"] = pd.to_numeric(surface_raw["strike"], errors="coerce")
surface_raw["selected_iv"] = pd.to_numeric(surface_raw["selected_iv"], errors="coerce")

if "dte_calendar" in surface_raw.columns:
    surface_raw["dte_calendar"] = pd.to_numeric(surface_raw["dte_calendar"], errors="coerce")
else:
    surface_raw["dte_calendar"] = surface_raw["tau_years"] * 365.25

if "forward" in surface_raw.columns:
    surface_raw["forward"] = pd.to_numeric(surface_raw["forward"], errors="coerce")

if "spot" in surface_raw.columns:
    surface_raw["spot"] = pd.to_numeric(surface_raw["spot"], errors="coerce")

if "log_moneyness" in surface_raw.columns:
    surface_raw["log_moneyness"] = pd.to_numeric(surface_raw["log_moneyness"], errors="coerce")
elif "forward" in surface_raw.columns:
    surface_raw["log_moneyness"] = np.log(surface_raw["strike"] / surface_raw["forward"])
elif "spot" in surface_raw.columns:
    surface_raw["log_moneyness"] = np.log(surface_raw["strike"] / surface_raw["spot"])

if "total_variance" in surface_raw.columns:
    surface_raw["total_variance"] = pd.to_numeric(surface_raw["total_variance"], errors="coerce")
else:
    surface_raw["total_variance"] = (surface_raw["selected_iv"] ** 2) * surface_raw["tau_years"]

if "selected_price" in surface_raw.columns:
    surface_raw["selected_price"] = pd.to_numeric(surface_raw["selected_price"], errors="coerce")

if "bid" in surface_raw.columns:
    surface_raw["bid"] = pd.to_numeric(surface_raw["bid"], errors="coerce")

if "ask" in surface_raw.columns:
    surface_raw["ask"] = pd.to_numeric(surface_raw["ask"], errors="coerce")

if "rate" in surface_raw.columns:
    surface_raw["rate"] = pd.to_numeric(surface_raw["rate"], errors="coerce")
else:
    surface_raw["rate"] = 0.0

if "dividend_yield" in surface_raw.columns:
    surface_raw["dividend_yield"] = pd.to_numeric(surface_raw["dividend_yield"], errors="coerce")
else:
    surface_raw["dividend_yield"] = 0.0

if "option_type" in surface_raw.columns:
    surface_raw["option_type"] = (
        surface_raw["option_type"]
        .astype(str)
        .str.lower()
        .str.strip()
        .replace(
            {
                "c": "call",
                "call": "call",
                "calls": "call",
                "p": "put",
                "put": "put",
                "puts": "put",
            }
        )
    )


# ------------------------------------------------------------
# Input validation ledger
# ------------------------------------------------------------

def validation_row(check: str, passed: bool, details: str) -> Dict[str, Any]:
    return {
        "check": check,
        "passed": bool(passed),
        "details": details,
    }


validation_rows: List[Dict[str, Any]] = []

validation_rows.append(
    validation_row(
        "handoff_file_exists",
        N09_HANDOFF_PATH.exists(),
        str(N09_HANDOFF_PATH),
    )
)

validation_rows.append(
    validation_row(
        "handoff_non_empty",
        len(surface_raw) > 0,
        f"{len(surface_raw):,} rows",
    )
)

for concept in required_concepts:
    validation_rows.append(
        validation_row(
            f"required_column_resolved__{concept}",
            resolved_columns.get(concept) is not None,
            str(resolved_columns.get(concept)),
        )
    )

validation_rows.append(
    validation_row(
        "expiry_parses",
        surface_raw["expiry"].notna().all(),
        f"{surface_raw['expiry'].isna().sum():,} invalid expiry values",
    )
)

validation_rows.append(
    validation_row(
        "tau_positive",
        np.isfinite(surface_raw["tau_years"]).all() and (surface_raw["tau_years"] > 0).all(),
        f"{((~np.isfinite(surface_raw['tau_years'])) | (surface_raw['tau_years'] <= 0)).sum():,} invalid tau values",
    )
)

validation_rows.append(
    validation_row(
        "strike_positive",
        np.isfinite(surface_raw["strike"]).all() and (surface_raw["strike"] > 0).all(),
        f"{((~np.isfinite(surface_raw['strike'])) | (surface_raw['strike'] <= 0)).sum():,} invalid strike values",
    )
)

validation_rows.append(
    validation_row(
        "iv_positive_finite",
        np.isfinite(surface_raw["selected_iv"]).all() and (surface_raw["selected_iv"] > 0).all(),
        f"{((~np.isfinite(surface_raw['selected_iv'])) | (surface_raw['selected_iv'] <= 0)).sum():,} invalid IV values",
    )
)

validation_rows.append(
    validation_row(
        "total_variance_nonnegative_finite",
        np.isfinite(surface_raw["total_variance"]).all() and (surface_raw["total_variance"] >= 0).all(),
        f"{((~np.isfinite(surface_raw['total_variance'])) | (surface_raw['total_variance'] < 0)).sum():,} invalid total variance values",
    )
)

if "forward" in surface_raw.columns:
    validation_rows.append(
        validation_row(
            "forward_positive_if_available",
            surface_raw["forward"].notna().all() and np.isfinite(surface_raw["forward"]).all() and (surface_raw["forward"] > 0).all(),
            f"{((~np.isfinite(surface_raw['forward'])) | (surface_raw['forward'] <= 0)).sum():,} invalid forward values",
        )
    )

if "spot" in surface_raw.columns:
    validation_rows.append(
        validation_row(
            "spot_positive_if_available",
            surface_raw["spot"].notna().all() and np.isfinite(surface_raw["spot"]).all() and (surface_raw["spot"] > 0).all(),
            f"{((~np.isfinite(surface_raw['spot'])) | (surface_raw['spot'] <= 0)).sum():,} invalid spot values",
        )
    )

if "log_moneyness" in surface_raw.columns:
    validation_rows.append(
        validation_row(
            "log_moneyness_finite_if_available",
            np.isfinite(surface_raw["log_moneyness"]).all(),
            f"{(~np.isfinite(surface_raw['log_moneyness'])).sum():,} invalid log-moneyness values",
        )
    )

duplicate_count = surface_raw.duplicated(subset=["expiry", "strike"], keep=False).sum()

validation_rows.append(
    validation_row(
        "no_duplicate_expiry_strike_rows",
        duplicate_count == 0,
        f"{duplicate_count:,} duplicate expiry-strike rows",
    )
)

expiry_counts = surface_raw.groupby("expiry", dropna=False).size().rename("n_obs").reset_index()
n_expiries = expiry_counts["expiry"].nunique(dropna=True)
n_viable_expiries = int((expiry_counts["n_obs"] >= TOL.min_points_per_expiry).sum())

validation_rows.append(
    validation_row(
        "minimum_number_of_expiries",
        n_expiries >= TOL.min_expiries_for_calendar,
        f"{n_expiries:,} expiries available; threshold={TOL.min_expiries_for_calendar}",
    )
)

validation_rows.append(
    validation_row(
        "minimum_points_per_expiry_available",
        n_viable_expiries >= TOL.min_expiries_for_calendar,
        f"{n_viable_expiries:,} expiries have at least {TOL.min_points_per_expiry} points",
    )
)

input_validation_ledger = pd.DataFrame(validation_rows)

display(input_validation_ledger)

if not input_validation_ledger["passed"].all():
    failed = input_validation_ledger.loc[~input_validation_ledger["passed"]]
    print("Input validation warnings/failures detected:")
    display(failed)
else:
    print("All Notebook 09 handoff input validation checks passed.")


# ------------------------------------------------------------
# Compact input summary
# ------------------------------------------------------------

input_summary = {
    "handoff_path": str(N09_HANDOFF_PATH),
    "rows": int(len(surface_raw)),
    "columns": int(surface_raw.shape[1]),
    "expiry_count": int(n_expiries),
    "viable_expiry_count": int(n_viable_expiries),
    "min_expiry": str(surface_raw["expiry"].min().date()) if surface_raw["expiry"].notna().any() else None,
    "max_expiry": str(surface_raw["expiry"].max().date()) if surface_raw["expiry"].notna().any() else None,
    "min_dte_calendar": float(surface_raw["dte_calendar"].min()) if "dte_calendar" in surface_raw.columns else None,
    "max_dte_calendar": float(surface_raw["dte_calendar"].max()) if "dte_calendar" in surface_raw.columns else None,
    "min_strike": float(surface_raw["strike"].min()),
    "max_strike": float(surface_raw["strike"].max()),
    "min_selected_iv": float(surface_raw["selected_iv"].min()),
    "max_selected_iv": float(surface_raw["selected_iv"].max()),
    "min_total_variance": float(surface_raw["total_variance"].min()),
    "max_total_variance": float(surface_raw["total_variance"].max()),
}

RUN_METADATA["n09_handoff_path"] = str(N09_HANDOFF_PATH)
RUN_METADATA["input_summary"] = input_summary
RUN_METADATA["schema_resolution"] = schema_resolution.to_dict(orient="records")

display(pd.DataFrame([input_summary]))

surface_raw.head()

Candidate Notebook 09 handoff files:


,rank,path,filename,suffix,score,modified_utc,size_mb
0,1,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,spy_n09_n10_handoff_panel_20260705_154512_UTC....,.parquet,98,2026-07-05 17:37:15,0.35294800
1,2,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,spy_n09_n10_handoff_panel_20260705_154512_UTC.csv,.csv,93,2026-07-05 17:37:15,1.12206800
2,3,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,spy_n09_surface_grade_panel_20260705_154512_UT...,.parquet,58,2026-07-05 17:37:15,0.35811600
3,4,D:\Derivative Pricing Project v1.0+\V1.1\data\...,spy_option_surface_grade_panel_20260705_154512...,.parquet,38,2026-07-05 17:37:15,0.35811600


Selected Notebook 09 handoff:
d:\Derivative Pricing Project v1.0+\V1.1\outputs\09_spy_implied_volatility_inversion_and_surface_grade_filtering\spy_n09_n10_handoff_panel_20260705_154512_UTC.parquet
Loaded shape: 1,333 rows x 55 columns


,concept,resolved_column,required
0,expiry,expiry,True
1,tau_years,tau_years,True
2,strike,strike,True
3,selected_iv,mid_iv,True
4,dte_calendar,dte_calendar,False
5,forward,forward,False
6,spot,spot,False
7,log_moneyness,log_moneyness,False
8,total_variance,total_variance,False
9,option_type,selected_option_type,False


,check,passed,details
0,handoff_file_exists,True,d:\Derivative Pricing Project v1.0+\V1.1\outpu...
1,handoff_non_empty,True,"1,333 rows"
2,required_column_resolved__expiry,True,expiry
3,required_column_resolved__tau_years,True,tau_years
4,required_column_resolved__strike,True,strike
5,required_column_resolved__selected_iv,True,mid_iv
6,expiry_parses,True,0 invalid expiry values
7,tau_positive,True,0 invalid tau values
8,strike_positive,True,0 invalid strike values
9,iv_positive_finite,True,0 invalid IV values


All Notebook 09 handoff input validation checks passed.


,handoff_path,rows,columns,expiry_count,viable_expiry_count,min_expiry,max_expiry,min_dte_calendar,max_dte_calendar,min_strike,max_strike,min_selected_iv,max_selected_iv,min_total_variance,max_total_variance
0,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,1333,60,8,8,2026-07-10,2026-09-18,5.00000000,75.00000000,450.00000000,920.00000000,0.10525319,0.63933978,0.00016258,0.03776268


,n10_handoff_row_id,surface_grade_row_id,surface_candidate_row_id,source_n09_row_id,source_row_id,snapshot_ts_utc,ticker,underlying,expiry,dte_calendar,tau_years,selected_option_type,strike,spot,forward,log_moneyness,abs_log_moneyness,mid_iv,total_variance,diagnostic_weight,surface_row_source,moneyness_forward,is_atm_forward,moneyness_bucket,dte_bucket,bid,mid,ask,spread,relative_spread,bid_iv,ask_iv,iv_width_bid_ask,relative_iv_width_bid_ask,total_volatility,vega,price_spread_to_vega,rate_annual,dividend_yield_annual,discount_factor,dividend_discount_factor,bsm_time_value_mid,pair_abs_call_put_iv_diff,pair_relative_call_put_iv_diff,pair_abs_mid_parity_residual_bps_spot,primary_pair_issue,primary_stability_issue,quality_score,final_surface_filter_issue,n09_snapshot_tag,n09_handoff_created_utc,iv_band_weight,spread_weight,vega_weight,diagnostic_weight_raw,selected_iv,option_type,selected_price,rate,dividend_yield
0,0,0,1510,618,931,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,658.00000000,744.78002930,745.03000000,-0.12421955,0.12421955,0.40456494,0.00224209,0.28430707,single_side_stability_clean_fallback,0.88318591,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.39957753,0.40916182,0.00958429,0.02369035,0.04735074,1.04619261,0.00955847,0.04500000,0.02050310,0.99938375,0.99971918,0.04500000,NaN,NaN,NaN,unmatched_single_side,stability_clean,0.67486370,surface_grade,20260705_154512_UTC,2026-07-05T16:51:31+00:00,0.99050670,0.81818182,0.17060725,0.22590303,0.40456494,put,0.04500000,0.00000000,0.00000000
1,1,1,180,619,932,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,659.00000000,744.78002930,745.03000000,-0.12270095,0.12270095,0.40010994,0.00219299,0.28566013,matched_pair_preferred_side,0.88452814,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.39516834,0.40466481,0.00949646,0.02373463,0.04682933,1.05586577,0.00947090,0.04500000,0.02050310,0.99938375,0.99971918,0.04500000,0.04229943,0.10041178,0.84610015,not_both_stability_clean,stability_clean,0.67490358,surface_grade,20260705_154512_UTC,2026-07-05T16:51:31+00:00,0.99059287,0.81818182,0.17218470,0.22697814,0.40010994,put,0.04500000,0.00000000,0.00000000
2,2,2,181,620,933,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,660.00000000,744.78002930,745.03000000,-0.12118465,0.12118465,0.39565753,0.00214445,0.28703299,matched_pair_preferred_side,0.88587037,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.39076178,0.40017035,0.00940857,0.02377958,0.04630821,1.06572725,0.00938326,0.04500000,0.02050310,0.99938375,0.99971918,0.04500000,0.04160668,0.09990537,0.83782592,not_both_stability_clean,stability_clean,0.67494351,surface_grade,20260705_154512_UTC,2026-07-05T16:51:31+00:00,0.99067913,0.81818182,0.17379285,0.22806898,0.39565753,put,0.04500000,0.00000000,0.00000000
3,3,3,182,621,934,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,661.00000000,744.78002930,745.03000000,-0.11967065,0.11967065,0.39120764,0.00209649,0.28842617,matched_pair_preferred_side,0.88721260,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.38635778,0.39567838,0.00932061,0.02382521,0.04578739,1.07578296,0.00929556,0.04500000,0.02050310,0.99938375,0.99971918,0.04500000,0.04091788,0.09939567,0.82955168,not_both_stability_clean,stability_clean,0.67498346,surface_grade,20260705_154512_UTC,2026-07-05T16:51:31+00:00,0.99076547,0.81818182,0.17543268,0.22917596,0.39120764,put,0.04500000,0.00000000,0.00000000
4,4,4,183,622,935,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,662.00000000,744.78002930,745.03000000,-0.11815893,0.11815893,0.38676019,0.00204909,0.28984024,matched_pair_preferred_side,0.88855482,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.

In [3]:
# ============================================================
# Construct normalized surface coordinates and repair canonical economic fields
# ============================================================

surface_base = surface_raw.copy()


# ------------------------------------------------------------
# Repair canonical rate/dividend fields if richer N09 columns exist
# ------------------------------------------------------------
# Cell 3 intentionally created fallback rate/dividend_yield fields when aliases were unresolved.
# The loaded N09 handoff already contains rate_annual and dividend_yield_annual, so use them here.

if "rate_annual" in surface_base.columns:
    surface_base["rate"] = pd.to_numeric(surface_base["rate_annual"], errors="coerce")
else:
    surface_base["rate"] = pd.to_numeric(surface_base.get("rate", 0.0), errors="coerce").fillna(0.0)

if "dividend_yield_annual" in surface_base.columns:
    surface_base["dividend_yield"] = pd.to_numeric(surface_base["dividend_yield_annual"], errors="coerce")
else:
    surface_base["dividend_yield"] = pd.to_numeric(surface_base.get("dividend_yield", 0.0), errors="coerce").fillna(0.0)

surface_base["rate"] = surface_base["rate"].fillna(0.0)
surface_base["dividend_yield"] = surface_base["dividend_yield"].fillna(0.0)


# ------------------------------------------------------------
# Normalize core fields
# ------------------------------------------------------------

surface_base["n10_row_id"] = np.arange(len(surface_base), dtype=int)

surface_base["expiry"] = pd.to_datetime(surface_base["expiry"], errors="coerce")
surface_base["tau_years"] = pd.to_numeric(surface_base["tau_years"], errors="coerce")
surface_base["dte_calendar"] = pd.to_numeric(surface_base["dte_calendar"], errors="coerce")
surface_base["strike"] = pd.to_numeric(surface_base["strike"], errors="coerce")
surface_base["spot"] = pd.to_numeric(surface_base["spot"], errors="coerce")
surface_base["forward"] = pd.to_numeric(surface_base["forward"], errors="coerce")
surface_base["selected_iv"] = pd.to_numeric(surface_base["selected_iv"], errors="coerce")
surface_base["total_variance"] = pd.to_numeric(surface_base["total_variance"], errors="coerce")

if "selected_price" in surface_base.columns:
    surface_base["selected_price"] = pd.to_numeric(surface_base["selected_price"], errors="coerce")

if "bid" in surface_base.columns:
    surface_base["bid"] = pd.to_numeric(surface_base["bid"], errors="coerce")

if "ask" in surface_base.columns:
    surface_base["ask"] = pd.to_numeric(surface_base["ask"], errors="coerce")

if "diagnostic_weight" in surface_base.columns:
    surface_base["diagnostic_weight"] = pd.to_numeric(surface_base["diagnostic_weight"], errors="coerce")
else:
    surface_base["diagnostic_weight"] = 1.0

surface_base["diagnostic_weight"] = surface_base["diagnostic_weight"].clip(lower=0.0, upper=1.0).fillna(0.0)


# ------------------------------------------------------------
# Recompute surface coordinates from canonical fields
# ------------------------------------------------------------

surface_base["moneyness_forward"] = surface_base["strike"] / surface_base["forward"]
surface_base["log_moneyness"] = np.log(surface_base["moneyness_forward"])
surface_base["abs_log_moneyness"] = surface_base["log_moneyness"].abs()
surface_base["total_variance"] = (surface_base["selected_iv"] ** 2) * surface_base["tau_years"]
surface_base["total_volatility"] = np.sqrt(surface_base["total_variance"].clip(lower=0.0))

surface_base["discount_factor"] = np.exp(-surface_base["rate"] * surface_base["tau_years"])
surface_base["dividend_discount_factor"] = np.exp(-surface_base["dividend_yield"] * surface_base["tau_years"])

surface_base["forward_from_spot_carry"] = (
    surface_base["spot"]
    * np.exp((surface_base["rate"] - surface_base["dividend_yield"]) * surface_base["tau_years"])
)

surface_base["forward_carry_residual"] = surface_base["forward"] - surface_base["forward_from_spot_carry"]
surface_base["forward_carry_residual_bps_spot"] = (
    10_000.0 * surface_base["forward_carry_residual"] / surface_base["spot"]
)


# ------------------------------------------------------------
# Option-type normalization
# ------------------------------------------------------------

surface_base["option_type"] = (
    surface_base["option_type"]
    .astype(str)
    .str.lower()
    .str.strip()
    .replace(
        {
            "c": "call",
            "calls": "call",
            "call": "call",
            "p": "put",
            "puts": "put",
            "put": "put",
        }
    )
)

surface_base["is_call"] = surface_base["option_type"].eq("call")
surface_base["is_put"] = surface_base["option_type"].eq("put")


# ------------------------------------------------------------
# Maturity and moneyness buckets
# ------------------------------------------------------------

def assign_dte_bucket(dte: float) -> str:
    if not np.isfinite(dte):
        return "unknown"
    if dte <= 7:
        return "ultra_short"
    if dte <= 21:
        return "short"
    if dte <= 45:
        return "front_intermediate"
    if dte <= 90:
        return "intermediate"
    if dte <= 180:
        return "medium"
    return "long"


def assign_moneyness_bucket(k: float) -> str:
    if not np.isfinite(k):
        return "unknown"
    if k <= -TOL.wing_abs_log_moneyness:
        return "deep_put_wing"
    if k <= -TOL.near_atm_abs_log_moneyness:
        return "put_wing"
    if k <= -TOL.atm_abs_log_moneyness:
        return "near_atm_put"
    if k < TOL.atm_abs_log_moneyness:
        return "atm"
    if k < TOL.near_atm_abs_log_moneyness:
        return "near_atm_call"
    if k < TOL.wing_abs_log_moneyness:
        return "call_wing"
    return "deep_call_wing"


surface_base["maturity_bucket"] = surface_base["dte_calendar"].map(assign_dte_bucket)
surface_base["moneyness_bucket_n10"] = surface_base["log_moneyness"].map(assign_moneyness_bucket)
surface_base["is_atm_n10"] = surface_base["abs_log_moneyness"] <= TOL.atm_abs_log_moneyness
surface_base["is_near_atm_n10"] = surface_base["abs_log_moneyness"] <= TOL.near_atm_abs_log_moneyness
surface_base["is_wing_n10"] = surface_base["abs_log_moneyness"] >= TOL.wing_abs_log_moneyness


# ------------------------------------------------------------
# Expiry-level ranks and within-slice geometry
# ------------------------------------------------------------

surface_base = surface_base.sort_values(["expiry", "strike", "n10_row_id"]).reset_index(drop=True)
surface_base["n10_row_id"] = np.arange(len(surface_base), dtype=int)

expiry_order = (
    surface_base[["expiry", "dte_calendar", "tau_years"]]
    .drop_duplicates()
    .sort_values(["tau_years", "expiry"])
    .reset_index(drop=True)
)

expiry_order["expiry_rank"] = np.arange(len(expiry_order), dtype=int)

surface_base = surface_base.merge(
    expiry_order[["expiry", "expiry_rank"]],
    on="expiry",
    how="left",
    validate="many_to_one",
)

surface_base["strike_rank_within_expiry"] = (
    surface_base.groupby("expiry")["strike"].rank(method="first").astype(int) - 1
)

surface_base["moneyness_rank_within_expiry"] = (
    surface_base.groupby("expiry")["log_moneyness"].rank(method="first").astype(int) - 1
)

surface_base["n_points_expiry"] = surface_base.groupby("expiry")["n10_row_id"].transform("size")
surface_base["expiry_has_min_points"] = surface_base["n_points_expiry"] >= TOL.min_points_per_expiry
surface_base["expiry_has_convexity_points"] = surface_base["n_points_expiry"] >= TOL.min_points_for_convexity


# ------------------------------------------------------------
# Quote uncertainty fields
# ------------------------------------------------------------

if {"bid", "ask"}.issubset(surface_base.columns):
    surface_base["price_spread"] = surface_base["ask"] - surface_base["bid"]
else:
    surface_base["price_spread"] = np.nan

if "selected_price" in surface_base.columns:
    surface_base["relative_price_spread"] = surface_base["price_spread"] / surface_base["selected_price"].abs().clip(lower=TOL.eps)
else:
    surface_base["relative_price_spread"] = np.nan

if "iv_width_bid_ask" in surface_base.columns:
    surface_base["iv_uncertainty_width"] = pd.to_numeric(surface_base["iv_width_bid_ask"], errors="coerce")
elif {"bid_iv", "ask_iv"}.issubset(surface_base.columns):
    surface_base["iv_uncertainty_width"] = (
        pd.to_numeric(surface_base["ask_iv"], errors="coerce")
        - pd.to_numeric(surface_base["bid_iv"], errors="coerce")
    )
else:
    surface_base["iv_uncertainty_width"] = np.nan

surface_base["relative_iv_uncertainty_width"] = (
    surface_base["iv_uncertainty_width"] / surface_base["selected_iv"].abs().clip(lower=TOL.eps)
)


# ------------------------------------------------------------
# Base diagnostic eligibility
# ------------------------------------------------------------

surface_base["eligible_pointwise_diagnostics"] = (
    surface_base["expiry"].notna()
    & np.isfinite(surface_base["tau_years"])
    & (surface_base["tau_years"] > 0)
    & np.isfinite(surface_base["strike"])
    & (surface_base["strike"] > 0)
    & np.isfinite(surface_base["forward"])
    & (surface_base["forward"] > 0)
    & np.isfinite(surface_base["selected_iv"])
    & (surface_base["selected_iv"] > 0)
    & np.isfinite(surface_base["total_variance"])
    & (surface_base["total_variance"] >= 0)
)

surface_base["eligible_cross_section_diagnostics"] = (
    surface_base["eligible_pointwise_diagnostics"]
    & surface_base["expiry_has_min_points"]
)

surface_base["eligible_convexity_diagnostics"] = (
    surface_base["eligible_pointwise_diagnostics"]
    & surface_base["expiry_has_convexity_points"]
)


# ------------------------------------------------------------
# Expiry-level inventory
# ------------------------------------------------------------

expiry_surface_inventory = (
    surface_base
    .groupby(["expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        tau_years=("tau_years", "median"),
        n_obs=("n10_row_id", "size"),
        n_calls=("is_call", "sum"),
        n_puts=("is_put", "sum"),
        min_strike=("strike", "min"),
        max_strike=("strike", "max"),
        min_log_moneyness=("log_moneyness", "min"),
        max_log_moneyness=("log_moneyness", "max"),
        n_atm=("is_atm_n10", "sum"),
        n_near_atm=("is_near_atm_n10", "sum"),
        n_wing=("is_wing_n10", "sum"),
        median_iv=("selected_iv", "median"),
        min_iv=("selected_iv", "min"),
        max_iv=("selected_iv", "max"),
        median_total_variance=("total_variance", "median"),
        min_total_variance=("total_variance", "min"),
        max_total_variance=("total_variance", "max"),
        median_diagnostic_weight=("diagnostic_weight", "median"),
        median_relative_price_spread=("relative_price_spread", "median"),
        median_relative_iv_uncertainty_width=("relative_iv_uncertainty_width", "median"),
        max_abs_forward_carry_residual_bps=("forward_carry_residual_bps_spot", lambda x: np.nanmax(np.abs(x))),
        eligible_pointwise_rows=("eligible_pointwise_diagnostics", "sum"),
        eligible_cross_section_rows=("eligible_cross_section_diagnostics", "sum"),
        eligible_convexity_rows=("eligible_convexity_diagnostics", "sum"),
    )
    .reset_index()
    .sort_values(["expiry_rank", "expiry"])
)

expiry_surface_inventory["has_min_points"] = expiry_surface_inventory["n_obs"] >= TOL.min_points_per_expiry
expiry_surface_inventory["has_convexity_points"] = expiry_surface_inventory["n_obs"] >= TOL.min_points_for_convexity
expiry_surface_inventory["has_atm_support"] = expiry_surface_inventory["n_atm"] > 0


# ------------------------------------------------------------
# Coordinate validation ledger
# ------------------------------------------------------------

coordinate_validation_rows = [
    validation_row(
        "rate_field_uses_annual_column_if_available",
        ("rate_annual" not in surface_base.columns) or np.allclose(surface_base["rate"], surface_base["rate_annual"], equal_nan=True),
        "canonical rate column repaired from rate_annual when available",
    ),
    validation_row(
        "dividend_field_uses_annual_column_if_available",
        ("dividend_yield_annual" not in surface_base.columns)
        or np.allclose(surface_base["dividend_yield"], surface_base["dividend_yield_annual"], equal_nan=True),
        "canonical dividend_yield column repaired from dividend_yield_annual when available",
    ),
    validation_row(
        "all_forward_moneyness_finite",
        np.isfinite(surface_base["moneyness_forward"]).all(),
        f"{(~np.isfinite(surface_base['moneyness_forward'])).sum():,} invalid forward-moneyness values",
    ),
    validation_row(
        "all_log_moneyness_finite",
        np.isfinite(surface_base["log_moneyness"]).all(),
        f"{(~np.isfinite(surface_base['log_moneyness'])).sum():,} invalid log-moneyness values",
    ),
    validation_row(
        "all_total_variance_recomputed_finite",
        np.isfinite(surface_base["total_variance"]).all(),
        f"{(~np.isfinite(surface_base['total_variance'])).sum():,} invalid total-variance values",
    ),
    validation_row(
        "all_rows_pointwise_eligible",
        bool(surface_base["eligible_pointwise_diagnostics"].all()),
        f"{int((~surface_base['eligible_pointwise_diagnostics']).sum()):,} rows fail pointwise eligibility",
    ),
    validation_row(
        "all_expiries_have_min_points",
        bool(expiry_surface_inventory["has_min_points"].all()),
        f"{int((~expiry_surface_inventory['has_min_points']).sum()):,} expiries below min_points_per_expiry",
    ),
    validation_row(
        "all_expiries_have_convexity_points",
        bool(expiry_surface_inventory["has_convexity_points"].all()),
        f"{int((~expiry_surface_inventory['has_convexity_points']).sum()):,} expiries below min_points_for_convexity",
    ),
]

coordinate_validation_ledger = pd.DataFrame(coordinate_validation_rows)

print("=" * 90)
print("Canonical economic field check")
print("=" * 90)
display(
    surface_base[
        [
            "rate",
            "dividend_yield",
            "discount_factor",
            "dividend_discount_factor",
            "forward",
            "forward_from_spot_carry",
            "forward_carry_residual_bps_spot",
        ]
    ].describe().T
)

print("=" * 90)
print("Coordinate validation ledger")
print("=" * 90)
display(coordinate_validation_ledger)

print("=" * 90)
print("Expiry-level raw surface inventory")
print("=" * 90)
display(expiry_surface_inventory)

print("=" * 90)
print("Moneyness bucket inventory")
print("=" * 90)
display(
    surface_base
    .groupby(["maturity_bucket", "moneyness_bucket_n10"], dropna=False)
    .size()
    .rename("n_obs")
    .reset_index()
    .sort_values(["maturity_bucket", "moneyness_bucket_n10"])
)

RUN_METADATA["coordinate_validation"] = coordinate_validation_ledger.to_dict(orient="records")
RUN_METADATA["expiry_surface_inventory"] = expiry_surface_inventory.to_dict(orient="records")

surface_base.head()

Canonical economic field check


,count,mean,std,min,25%,50%,75%,max
rate,"1,333.00000000",0.04500000,0.00000000,0.04500000,0.04500000,0.04500000,0.04500000,0.04500000
dividend_yield,"1,333.00000000",0.01188685,0.00349187,0.00719548,0.00950371,0.01092500,0.01324895,0.02050310
discount_factor,"1,333.00000000",0.99539445,0.00281756,0.99079604,0.99299724,0.99593977,0.99766028,0.99938375
dividend_discount_factor,"1,333.00000000",0.99886670,0.00078543,0.99728132,0.99851696,0.99922208,0.99939159,0.99971918
forward,"1,333.00000000",747.38246999,1.57177146,745.03000000,746.07250000,747.33000000,748.92000000,749.65500000
forward_from_spot_carry,"1,333.00000000",747.38246999,1.57177146,745.03000000,746.07250000,747.33000000,748.92000000,749.65500000
forward_carry_residual_bps_spot,"1,333.00000000",-0.00000000,0.00000000,-0.00000000,-0.00000000,0.00000000,0.00000000,0.00000000


Coordinate validation ledger


,check,passed,details
0,rate_field_uses_annual_column_if_available,True,canonical rate column repaired from rate_annua...
1,dividend_field_uses_annual_column_if_available,True,canonical dividend_yield column repaired from ...
2,all_forward_moneyness_finite,True,0 invalid forward-moneyness values
3,all_log_moneyness_finite,True,0 invalid log-moneyness values
4,all_total_variance_recomputed_finite,True,0 invalid total-variance values
5,all_rows_pointwise_eligible,True,0 rows fail pointwise eligibility
6,all_expiries_have_min_points,True,0 expiries below min_points_per_expiry
7,all_expiries_have_convexity_points,True,0 expiries below min_points_for_convexity


Expiry-level raw surface inventory


,expiry,expiry_rank,maturity_bucket,dte_calendar,tau_years,n_obs,n_calls,n_puts,min_strike,max_strike,min_log_moneyness,max_log_moneyness,n_atm,n_near_atm,n_wing,median_iv,min_iv,max_iv,median_total_variance,min_total_variance,max_total_variance,median_diagnostic_weight,median_relative_price_spread,median_relative_iv_uncertainty_width,max_abs_forward_carry_residual_bps,eligible_pointwise_rows,eligible_cross_section_rows,eligible_convexity_rows,has_min_points,has_convexity_points,has_atm_support
0,2026-07-10 00:00:00+00:00,0,ultra_short,5.00000000,0.01369863,117,29,88,658.00000000,768.00000000,-0.12421955,0.03036525,43,83,0,0.20605265,0.10894070,0.40456494,0.00058161,0.00016258,0.00224209,0.89214865,0.05714286,0.00997783,0.00000000,117,117,117,True,True,True
1,2026-07-17 00:00:00+00:00,1,short,12.00000000,0.03287671,154,39,115,535.00000000,785.00000000,-0.33180851,0.05160846,36,91,20,0.20265289,0.10525319,0.63933978,0.00135022,0.00036422,0.01343853,0.94861885,0.03125763,0.00613005,0.00000000,154,154,154,True,True,True
2,2026-07-24 00:00:00+00:00,2,short,19.00000000,0.05205479,168,53,115,555.00000000,799.00000000,-0.29585467,0.06853817,37,106,18,0.17996124,0.10575389,0.49783264,0.00168588,0.00058217,0.01290112,0.95892869,0.02222497,0.00539110,0.00000000,168,168,168,True,True,True
3,2026-07-31 00:00:00+00:00,3,front_intermediate,26.00000000,0.07123288,183,42,141,500.00000000,810.00000000,-0.40090807,0.08151808,30,90,42,0.21049754,0.11222316,0.55595937,0.00315627,0.00089711,0.02201743,1.00000000,0.01780415,0.00458605,0.00000000,183,183,183,True,True,True
4,2026-08-07 00:00:00+00:00,4,front_intermediate,33.00000000,0.09041096,132,46,86,535.00000000,825.00000000,-0.33424011,0.09887653,36,92,19,0.16941186,0.11417076,0.45492290,0.00259486,0.00117850,0.01871099,0.97851477,0.01171879,0.00440095,0.00000000,132,132,132,True,True,True
5,2026-08-21 00:00:00+00:00,5,intermediate,47.00000000,0.12876712,173,45,128,450.00000000,860.00000000,-0.50848622,0.13919858,31,84,37,0.19436897,0.11698036,0.54153799,0.00486473,0.00176210,0.03776268,1.00000000,0.01052632,0.00352745,0.00000000,173,173,173,True,True,True
6,2026-08-31 00:00:00+00:00,6,intermediate,57.00000000,0.15616438,187,46,141,510.00000000,845.00000000,-0.38422144,0.12070446,29,89,43,0.19574508,0.11866914,0.42709200,0.00598362,0.00219916,0.02848557,1.00000000,0.00951475,0.00323256,0.00000000,187,187,187,True,True,True
7,2026-09-18 00:00:00+00:00,7,intermediate,75.00000000,0.20547945,219,58,161,490.00000000,920.00000000,-0.42520771,0.20476057,22,79,74,0.20538623,0.12254717,0.42842755,0.00866784,0.00308585,0.03771579,0.99365549,0.00706714,0.00230686,0.00000000,219,219,219,True,True,True


Moneyness bucket inventory


,maturity_bucket,moneyness_bucket_n10,n_obs
0,front_intermediate,atm,66
1,front_intermediate,call_wing,6
2,front_intermediate,deep_put_wing,61
3,front_intermediate,near_atm_call,46
4,front_intermediate,near_atm_put,70
5,front_intermediate,put_wing,66
6,intermediate,atm,82
7,intermediate,call_wing,29
8,intermediate,deep_call_wing,6
9,intermediate,deep_put_wing,148


,n10_handoff_row_id,surface_grade_row_id,surface_candidate_row_id,source_n09_row_id,source_row_id,snapshot_ts_utc,ticker,underlying,expiry,dte_calendar,tau_years,selected_option_type,strike,spot,forward,log_moneyness,abs_log_moneyness,mid_iv,total_variance,diagnostic_weight,surface_row_source,moneyness_forward,is_atm_forward,moneyness_bucket,dte_bucket,bid,mid,ask,spread,relative_spread,bid_iv,ask_iv,iv_width_bid_ask,relative_iv_width_bid_ask,total_volatility,vega,price_spread_to_vega,rate_annual,dividend_yield_annual,discount_factor,dividend_discount_factor,bsm_time_value_mid,pair_abs_call_put_iv_diff,pair_relative_call_put_iv_diff,pair_abs_mid_parity_residual_bps_spot,primary_pair_issue,primary_stability_issue,quality_score,final_surface_filter_issue,n09_snapshot_tag,n09_handoff_created_utc,iv_band_weight,spread_weight,vega_weight,diagnostic_weight_raw,selected_iv,option_type,selected_price,rate,dividend_yield,n10_row_id,forward_from_spot_carry,forward_carry_residual,forward_carry_residual_bps_spot,is_call,is_put,maturity_bucket,moneyness_bucket_n10,is_atm_n10,is_near_atm_n10,is_wing_n10,expiry_rank,strike_rank_within_expiry,moneyness_rank_within_expiry,n_points_expiry,expiry_has_min_points,expiry_has_convexity_points,price_spread,relative_price_spread,iv_uncertainty_width,relative_iv_uncertainty_width,eligible_pointwise_diagnostics,eligible_cross_section_diagnostics,eligible_convexity_diagnostics
0,0,0,1510,618,931,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,658.00000000,744.78002930,745.03000000,-0.12421955,0.12421955,0.40456494,0.00224209,0.28430707,single_side_stability_clean_fallback,0.88318591,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.39957753,0.40916182,0.00958429,0.02369035,0.04735074,1.04619261,0.00955847,0.04500000,0.02050310,0.99938375,0.99971918,0.04500000,NaN,NaN,NaN,unmatched_single_side,stability_clean,0.67486370,surface_grade,20260705_154512_UTC,2026-07-05T16:51:31+00:00,0.99050670,0.81818182,0.17060725,0.22590303,0.40456494,put,0.04500000,0.04500000,0.02050310,0,745.03000000,-0.00000000,-0.00000000,False,True,ultra_short,put_wing,False,False,False,0,0,0,117,True,True,0.01000000,0.22222222,0.00958429,0.02369035,True,True,True
1,1,1,180,619,932,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,659.00000000,744.78002930,745.03000000,-0.12270095,0.12270095,0.40010994,0.00219299,0.28566013,matched_pair_preferred_side,0.88452814,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.39516834,0.40466481,0.00949646,0.02373463,0.04682933,1.05586577,0.00947090,0.04500000,0.02050310,0.99938375,0.99971918,0.04500000,0.04229943,0.10041178,0.84610015,not_both_stability_clean,stability_clean,0.67490358,surface_grade,20260705_154512_UTC,2026-07-05T16:51:31+00:00,0.99059287,0.81818182,0.17218470,0.22697814,0.40010994,put,0.04500000,0.04500000,0.02050310,1,745.03000000,-0.00000000,-0.00000000,False,True,ultra_short,put_wing,False,False,False,0,1,1,117,True,True,0.01000000,0.22222222,0.00949646,0.02373463,True,True,True
2,2,2,181,620,933,2026-07-05 15:45:12.973274+00:00,SPY,SPY,2026-07-10 00:00:00+00:00,5.00000000,0.01369863,put,660.00000000,744.78002930,745.03000000,-0.12118465,0.12118465,0.39565753,0.00214445,0.28703299,matched_pair_preferred_side,0.88587037,False,left_wing,short,0.04000000,0.04500000,0.05000000,0.01000000,0.22222222,0.39076178,0.40017035,0.00940857,0.02377958,0.04630821,1.06572725,0.00938326,0.04500000,0.02050310,0.99938375,0.99971918,0.04500000,0.04160668,0.09990537,0.83782592,not_both_stability_clean,stability_clean,0.67494351,surface_grade,20260705_154512_UTC,2026-07-05T16:51:31+00:00,0.99067913,0.81818182,0.17379285,0.22806898,0.39565753,put,0.04500000,0.04500000,0.02050310,2,745.03000000,-0.00000000,-0.00000000,False,True,ultra_short,put_wing,False,False,False,0,2,2,117,True,True,0.01000000,0.22222222,0.00940857,0.02377958,True,True,True
3,3,3,182,621,

In [4]:
# ============================================================
# Reconstruct diagnostic option prices from selected IV
# ============================================================
# Static-arbitrage diagnostics are price-space diagnostics.
# This cell builds two price lenses:
#
# 1. observed selected-side price:
#    the actual N09 selected market mid, call or put depending on selected_option_type;
#
# 2. BSM-implied diagnostic price:
#    the Black-Scholes-Merton price implied by the selected IV, spot, rate, dividend yield, and maturity.
#
# The BSM-implied price is not treated as the true market price.
# It is a consistency reconstruction of the IV handoff.


surface_px = surface_base.copy()


# ------------------------------------------------------------
# Standard normal CDF/PDF helpers
# ------------------------------------------------------------

def norm_cdf(x: np.ndarray | pd.Series | float) -> np.ndarray:
    """
    Vectorized standard normal CDF using scipy if available, otherwise erf fallback.
    """
    x_arr = np.asarray(x, dtype=float)

    try:
        from scipy.stats import norm

        return norm.cdf(x_arr)
    except Exception:
        erf_vec = np.vectorize(math.erf)
        return 0.5 * (1.0 + erf_vec(x_arr / np.sqrt(2.0)))


def norm_pdf(x: np.ndarray | pd.Series | float) -> np.ndarray:
    """
    Vectorized standard normal PDF.
    """
    x_arr = np.asarray(x, dtype=float)
    return np.exp(-0.5 * x_arr * x_arr) / np.sqrt(2.0 * np.pi)


def bsm_d1_d2(
    spot: pd.Series,
    strike: pd.Series,
    tau: pd.Series,
    rate: pd.Series,
    dividend_yield: pd.Series,
    sigma: pd.Series,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute Black-Scholes-Merton d1 and d2 under continuous dividend yield.
    """
    S = pd.to_numeric(spot, errors="coerce").to_numpy(dtype=float)
    K = pd.to_numeric(strike, errors="coerce").to_numpy(dtype=float)
    T = pd.to_numeric(tau, errors="coerce").to_numpy(dtype=float)
    r = pd.to_numeric(rate, errors="coerce").to_numpy(dtype=float)
    q = pd.to_numeric(dividend_yield, errors="coerce").to_numpy(dtype=float)
    vol = pd.to_numeric(sigma, errors="coerce").to_numpy(dtype=float)

    vol_sqrt_T = vol * np.sqrt(np.maximum(T, TOL.eps))

    with np.errstate(divide="ignore", invalid="ignore"):
        d1 = (
            np.log(np.maximum(S, TOL.eps) / np.maximum(K, TOL.eps))
            + (r - q + 0.5 * vol * vol) * T
        ) / np.maximum(vol_sqrt_T, TOL.eps)

    d2 = d1 - vol_sqrt_T

    return d1, d2


def bsm_price_greeks(
    df: pd.DataFrame,
    sigma_col: str = "selected_iv",
) -> pd.DataFrame:
    """
    Add BSM call/put prices and basic diagnostic Greeks to a DataFrame.

    These are diagnostic reconstructions from the selected IV surface.
    """
    out = df.copy()

    d1, d2 = bsm_d1_d2(
        spot=out["spot"],
        strike=out["strike"],
        tau=out["tau_years"],
        rate=out["rate"],
        dividend_yield=out["dividend_yield"],
        sigma=out[sigma_col],
    )

    S = out["spot"].to_numpy(dtype=float)
    K = out["strike"].to_numpy(dtype=float)
    T = out["tau_years"].to_numpy(dtype=float)
    r = out["rate"].to_numpy(dtype=float)
    q = out["dividend_yield"].to_numpy(dtype=float)
    vol = out[sigma_col].to_numpy(dtype=float)

    dq = np.exp(-q * T)
    dr = np.exp(-r * T)

    Nd1 = norm_cdf(d1)
    Nd2 = norm_cdf(d2)
    N_minus_d1 = norm_cdf(-d1)
    N_minus_d2 = norm_cdf(-d2)

    pdf_d1 = norm_pdf(d1)

    out["bsm_d1"] = d1
    out["bsm_d2"] = d2

    out["bsm_call_price"] = S * dq * Nd1 - K * dr * Nd2
    out["bsm_put_price"] = K * dr * N_minus_d2 - S * dq * N_minus_d1

    out["bsm_call_delta"] = dq * Nd1
    out["bsm_put_delta"] = dq * (Nd1 - 1.0)
    out["bsm_gamma"] = dq * pdf_d1 / np.maximum(S * vol * np.sqrt(np.maximum(T, TOL.eps)), TOL.eps)
    out["bsm_vega"] = S * dq * pdf_d1 * np.sqrt(np.maximum(T, TOL.eps))

    out["discounted_spot"] = S * dq
    out["discounted_strike"] = K * dr
    out["bsm_intrinsic_call"] = np.maximum(out["discounted_spot"] - out["discounted_strike"], 0.0)
    out["bsm_intrinsic_put"] = np.maximum(out["discounted_strike"] - out["discounted_spot"], 0.0)

    return out


surface_px = bsm_price_greeks(surface_px, sigma_col="selected_iv")


# ------------------------------------------------------------
# Selected-side BSM reconstruction
# ------------------------------------------------------------

surface_px["bsm_selected_price"] = np.where(
    surface_px["is_call"],
    surface_px["bsm_call_price"],
    np.where(surface_px["is_put"], surface_px["bsm_put_price"], np.nan),
)

surface_px["bsm_selected_delta"] = np.where(
    surface_px["is_call"],
    surface_px["bsm_call_delta"],
    np.where(surface_px["is_put"], surface_px["bsm_put_delta"], np.nan),
)

surface_px["bsm_selected_intrinsic"] = np.where(
    surface_px["is_call"],
    surface_px["bsm_intrinsic_call"],
    np.where(surface_px["is_put"], surface_px["bsm_intrinsic_put"], np.nan),
)

surface_px["bsm_selected_time_value"] = (
    surface_px["bsm_selected_price"] - surface_px["bsm_selected_intrinsic"]
)


# ------------------------------------------------------------
# Market selected-side price and parity-equivalent prices
# ------------------------------------------------------------

surface_px["market_selected_price"] = pd.to_numeric(surface_px["selected_price"], errors="coerce")

# Parity identity under continuous dividend yield:
# C - P = S exp(-qT) - K exp(-rT)
surface_px["parity_forward_value"] = (
    surface_px["discounted_spot"] - surface_px["discounted_strike"]
)

# Convert selected market prices into call-equivalent and put-equivalent values.
# This avoids mixing calls and puts directly in strike-monotonicity / convexity diagnostics.
surface_px["market_call_equiv_price"] = np.where(
    surface_px["is_call"],
    surface_px["market_selected_price"],
    np.where(
        surface_px["is_put"],
        surface_px["market_selected_price"] + surface_px["parity_forward_value"],
        np.nan,
    ),
)

surface_px["market_put_equiv_price"] = np.where(
    surface_px["is_put"],
    surface_px["market_selected_price"],
    np.where(
        surface_px["is_call"],
        surface_px["market_selected_price"] - surface_px["parity_forward_value"],
        np.nan,
    ),
)

# BSM prices already provide both call and put prices directly.
surface_px["bsm_call_equiv_price"] = surface_px["bsm_call_price"]
surface_px["bsm_put_equiv_price"] = surface_px["bsm_put_price"]


# ------------------------------------------------------------
# IV-inversion consistency diagnostics
# ------------------------------------------------------------

surface_px["selected_price_minus_bsm_selected_price"] = (
    surface_px["market_selected_price"] - surface_px["bsm_selected_price"]
)

surface_px["abs_selected_price_minus_bsm"] = surface_px["selected_price_minus_bsm_selected_price"].abs()

surface_px["rel_selected_price_minus_bsm"] = (
    surface_px["selected_price_minus_bsm_selected_price"]
    / surface_px["market_selected_price"].abs().clip(lower=TOL.eps)
)

surface_px["abs_rel_selected_price_minus_bsm"] = surface_px["rel_selected_price_minus_bsm"].abs()

surface_px["selected_price_minus_bsm_in_spread_units"] = (
    surface_px["selected_price_minus_bsm_selected_price"]
    / surface_px["price_spread"].abs().clip(lower=TOL.eps)
)

surface_px["bsm_time_value_nonnegative"] = surface_px["bsm_selected_time_value"] >= -TOL.price_abs_tol


# ------------------------------------------------------------
# Price-space diagnostic eligibility
# ------------------------------------------------------------

surface_px["eligible_market_price_diagnostics"] = (
    surface_px["eligible_pointwise_diagnostics"]
    & np.isfinite(surface_px["market_selected_price"])
    & (surface_px["market_selected_price"] >= 0)
)

surface_px["eligible_market_call_equiv_diagnostics"] = (
    surface_px["eligible_market_price_diagnostics"]
    & np.isfinite(surface_px["market_call_equiv_price"])
)

surface_px["eligible_market_put_equiv_diagnostics"] = (
    surface_px["eligible_market_price_diagnostics"]
    & np.isfinite(surface_px["market_put_equiv_price"])
)

surface_px["eligible_bsm_price_diagnostics"] = (
    surface_px["eligible_pointwise_diagnostics"]
    & np.isfinite(surface_px["bsm_call_price"])
    & np.isfinite(surface_px["bsm_put_price"])
    & (surface_px["bsm_call_price"] >= -TOL.price_abs_tol)
    & (surface_px["bsm_put_price"] >= -TOL.price_abs_tol)
)

surface_px["eligible_price_space_diagnostics"] = (
    surface_px["eligible_market_price_diagnostics"]
    & surface_px["eligible_bsm_price_diagnostics"]
)


# ------------------------------------------------------------
# Validation ledger for reconstructed price layer
# ------------------------------------------------------------

price_reconstruction_validation_rows = [
    validation_row(
        "all_market_selected_prices_finite",
        np.isfinite(surface_px["market_selected_price"]).all(),
        f"{int((~np.isfinite(surface_px['market_selected_price'])).sum()):,} invalid selected market prices",
    ),
    validation_row(
        "all_market_selected_prices_nonnegative",
        bool((surface_px["market_selected_price"] >= 0).all()),
        f"{int((surface_px['market_selected_price'] < 0).sum()):,} negative selected market prices",
    ),
    validation_row(
        "all_bsm_selected_prices_finite",
        np.isfinite(surface_px["bsm_selected_price"]).all(),
        f"{int((~np.isfinite(surface_px['bsm_selected_price'])).sum()):,} invalid BSM selected prices",
    ),
    validation_row(
        "all_bsm_call_equiv_prices_finite",
        np.isfinite(surface_px["bsm_call_equiv_price"]).all(),
        f"{int((~np.isfinite(surface_px['bsm_call_equiv_price'])).sum()):,} invalid BSM call-equivalent prices",
    ),
    validation_row(
        "all_bsm_put_equiv_prices_finite",
        np.isfinite(surface_px["bsm_put_equiv_price"]).all(),
        f"{int((~np.isfinite(surface_px['bsm_put_equiv_price'])).sum()):,} invalid BSM put-equivalent prices",
    ),
    validation_row(
        "all_market_call_equiv_prices_finite",
        np.isfinite(surface_px["market_call_equiv_price"]).all(),
        f"{int((~np.isfinite(surface_px['market_call_equiv_price'])).sum()):,} invalid market call-equivalent prices",
    ),
    validation_row(
        "all_market_put_equiv_prices_finite",
        np.isfinite(surface_px["market_put_equiv_price"]).all(),
        f"{int((~np.isfinite(surface_px['market_put_equiv_price'])).sum()):,} invalid market put-equivalent prices",
    ),
    validation_row(
        "all_bsm_selected_time_value_nonnegative",
        bool(surface_px["bsm_time_value_nonnegative"].all()),
        f"{int((~surface_px['bsm_time_value_nonnegative']).sum()):,} negative BSM selected time values beyond tolerance",
    ),
    validation_row(
        "all_rows_price_space_eligible",
        bool(surface_px["eligible_price_space_diagnostics"].all()),
        f"{int((~surface_px['eligible_price_space_diagnostics']).sum()):,} rows fail price-space eligibility",
    ),
]

price_reconstruction_validation_ledger = pd.DataFrame(price_reconstruction_validation_rows)


# ------------------------------------------------------------
# Compact price reconstruction summaries
# ------------------------------------------------------------

price_reconstruction_summary = pd.DataFrame(
    [
        {
            "rows": len(surface_px),
            "market_price_min": surface_px["market_selected_price"].min(),
            "market_price_median": surface_px["market_selected_price"].median(),
            "market_price_max": surface_px["market_selected_price"].max(),
            "bsm_selected_price_min": surface_px["bsm_selected_price"].min(),
            "bsm_selected_price_median": surface_px["bsm_selected_price"].median(),
            "bsm_selected_price_max": surface_px["bsm_selected_price"].max(),
            "median_abs_price_reconstruction_error": surface_px["abs_selected_price_minus_bsm"].median(),
            "max_abs_price_reconstruction_error": surface_px["abs_selected_price_minus_bsm"].max(),
            "median_abs_rel_price_reconstruction_error": surface_px["abs_rel_selected_price_minus_bsm"].median(),
            "max_abs_rel_price_reconstruction_error": surface_px["abs_rel_selected_price_minus_bsm"].max(),
            "median_error_in_spread_units": surface_px["selected_price_minus_bsm_in_spread_units"].abs().median(),
            "max_error_in_spread_units": surface_px["selected_price_minus_bsm_in_spread_units"].abs().max(),
        }
    ]
)

price_reconstruction_by_expiry = (
    surface_px
    .groupby(["expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        n_obs=("n10_row_id", "size"),
        n_calls=("is_call", "sum"),
        n_puts=("is_put", "sum"),
        median_market_price=("market_selected_price", "median"),
        median_bsm_selected_price=("bsm_selected_price", "median"),
        median_abs_price_reconstruction_error=("abs_selected_price_minus_bsm", "median"),
        max_abs_price_reconstruction_error=("abs_selected_price_minus_bsm", "max"),
        median_abs_rel_price_reconstruction_error=("abs_rel_selected_price_minus_bsm", "median"),
        max_abs_rel_price_reconstruction_error=("abs_rel_selected_price_minus_bsm", "max"),
        median_error_in_spread_units=("selected_price_minus_bsm_in_spread_units", lambda x: np.nanmedian(np.abs(x))),
        max_error_in_spread_units=("selected_price_minus_bsm_in_spread_units", lambda x: np.nanmax(np.abs(x))),
        median_bsm_vega=("bsm_vega", "median"),
        median_selected_time_value=("bsm_selected_time_value", "median"),
    )
    .reset_index()
    .sort_values(["expiry_rank", "expiry"])
)

price_lens_inventory = pd.DataFrame(
    [
        {
            "price_lens": "market_selected_price",
            "description": "N09 selected market mid on selected call/put side",
            "finite_rows": int(np.isfinite(surface_px["market_selected_price"]).sum()),
            "min": surface_px["market_selected_price"].min(),
            "median": surface_px["market_selected_price"].median(),
            "max": surface_px["market_selected_price"].max(),
        },
        {
            "price_lens": "market_call_equiv_price",
            "description": "selected market price converted to call-equivalent value by parity",
            "finite_rows": int(np.isfinite(surface_px["market_call_equiv_price"]).sum()),
            "min": surface_px["market_call_equiv_price"].min(),
            "median": surface_px["market_call_equiv_price"].median(),
            "max": surface_px["market_call_equiv_price"].max(),
        },
        {
            "price_lens": "market_put_equiv_price",
            "description": "selected market price converted to put-equivalent value by parity",
            "finite_rows": int(np.isfinite(surface_px["market_put_equiv_price"]).sum()),
            "min": surface_px["market_put_equiv_price"].min(),
            "median": surface_px["market_put_equiv_price"].median(),
            "max": surface_px["market_put_equiv_price"].max(),
        },
        {
            "price_lens": "bsm_call_equiv_price",
            "description": "BSM call price reconstructed from selected IV",
            "finite_rows": int(np.isfinite(surface_px["bsm_call_equiv_price"]).sum()),
            "min": surface_px["bsm_call_equiv_price"].min(),
            "median": surface_px["bsm_call_equiv_price"].median(),
            "max": surface_px["bsm_call_equiv_price"].max(),
        },
        {
            "price_lens": "bsm_put_equiv_price",
            "description": "BSM put price reconstructed from selected IV",
            "finite_rows": int(np.isfinite(surface_px["bsm_put_equiv_price"]).sum()),
            "min": surface_px["bsm_put_equiv_price"].min(),
            "median": surface_px["bsm_put_equiv_price"].median(),
            "max": surface_px["bsm_put_equiv_price"].max(),
        },
    ]
)


print("=" * 90)
print("Price reconstruction validation ledger")
print("=" * 90)
display(price_reconstruction_validation_ledger)

print("=" * 90)
print("Price lens inventory")
print("=" * 90)
display(price_lens_inventory)

print("=" * 90)
print("Price reconstruction summary")
print("=" * 90)
display(price_reconstruction_summary)

print("=" * 90)
print("Price reconstruction by expiry")
print("=" * 90)
display(price_reconstruction_by_expiry)

RUN_METADATA["price_reconstruction_validation"] = price_reconstruction_validation_ledger.to_dict(orient="records")
RUN_METADATA["price_reconstruction_summary"] = price_reconstruction_summary.to_dict(orient="records")

surface_px[
    [
        "expiry",
        "dte_calendar",
        "option_type",
        "strike",
        "spot",
        "forward",
        "log_moneyness",
        "selected_iv",
        "market_selected_price",
        "bsm_selected_price",
        "selected_price_minus_bsm_selected_price",
        "market_call_equiv_price",
        "market_put_equiv_price",
        "bsm_call_equiv_price",
        "bsm_put_equiv_price",
        "bsm_vega",
        "diagnostic_weight",
    ]
].head()

Price reconstruction validation ledger


,check,passed,details
0,all_market_selected_prices_finite,True,0 invalid selected market prices
1,all_market_selected_prices_nonnegative,True,0 negative selected market prices
2,all_bsm_selected_prices_finite,True,0 invalid BSM selected prices
3,all_bsm_call_equiv_prices_finite,True,0 invalid BSM call-equivalent prices
4,all_bsm_put_equiv_prices_finite,True,0 invalid BSM put-equivalent prices
5,all_market_call_equiv_prices_finite,True,0 invalid market call-equivalent prices
6,all_market_put_equiv_prices_finite,True,0 invalid market put-equivalent prices
7,all_bsm_selected_time_value_nonnegative,True,0 negative BSM selected time values beyond tol...
8,all_rows_price_space_eligible,True,0 rows fail price-space eligibility


Price lens inventory


,price_lens,description,finite_rows,min,median,max
0,market_selected_price,N09 selected market mid on selected call/put side,1333,0.04500000,2.01500000,17.79000000
1,market_call_equiv_price,selected market price converted to call-equiva...,1333,0.04500000,46.12663409,296.67929612
2,market_put_equiv_price,selected market price converted to put-equival...,1333,0.04500000,3.49000000,168.83215191
3,bsm_call_equiv_price,BSM call price reconstructed from selected IV,1333,0.04500000,46.12663409,296.67929612
4,bsm_put_equiv_price,BSM put price reconstructed from selected IV,1333,0.04500000,3.49000000,168.83215191


Price reconstruction summary


,rows,market_price_min,market_price_median,market_price_max,bsm_selected_price_min,bsm_selected_price_median,bsm_selected_price_max,median_abs_price_reconstruction_error,max_abs_price_reconstruction_error,median_abs_rel_price_reconstruction_error,max_abs_rel_price_reconstruction_error,median_error_in_spread_units,max_error_in_spread_units
0,1333,0.04500000,2.01500000,17.79000000,0.04500000,2.01500000,17.79000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000


Price reconstruction by expiry


,expiry,expiry_rank,maturity_bucket,dte_calendar,n_obs,n_calls,n_puts,median_market_price,median_bsm_selected_price,median_abs_price_reconstruction_error,max_abs_price_reconstruction_error,median_abs_rel_price_reconstruction_error,max_abs_rel_price_reconstruction_error,median_error_in_spread_units,max_error_in_spread_units,median_bsm_vega,median_selected_time_value
0,2026-07-10 00:00:00+00:00,0,ultra_short,5.00000000,117,29,88,0.20500000,0.20500000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,6.13216968,0.20500000
1,2026-07-17 00:00:00+00:00,1,short,12.00000000,154,39,115,0.38750000,0.38750000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,10.47641098,0.38750000
2,2026-07-24 00:00:00+00:00,2,short,19.00000000,168,53,115,0.79000000,0.79000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,19.27908690,0.79000000
3,2026-07-31 00:00:00+00:00,3,front_intermediate,26.00000000,183,42,141,1.11000000,1.11000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,24.92564513,1.11000000
4,2026-08-07 00:00:00+00:00,4,front_intermediate,33.00000000,132,46,86,3.15000000,3.15000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,53.93225267,3.15000000
5,2026-08-21 00:00:00+00:00,5,intermediate,47.00000000,173,45,128,3.06000000,3.06000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,52.18020015,3.06000000
6,2026-08-31 00:00:00+00:00,6,intermediate,57.00000000,187,46,141,3.72000000,3.72000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,61.98437939,3.72000000
7,2026-09-18 00:00:00+00:00,7,intermediate,75.00000000,219,58,161,4.23500000,4.23500000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,64.29545869,4.23500000


,expiry,dte_calendar,option_type,strike,spot,forward,log_moneyness,selected_iv,market_selected_price,bsm_selected_price,selected_price_minus_bsm_selected_price,market_call_equiv_price,market_put_equiv_price,bsm_call_equiv_price,bsm_put_equiv_price,bsm_vega,diagnostic_weight
0,2026-07-10 00:00:00+00:00,5.00000000,put,658.00000000,744.78002930,745.03000000,-0.12421955,0.40456494,0.04500000,0.04500000,0.00000000,87.02136790,0.04500000,87.02136790,0.04500000,1.04619261,0.28430707
1,2026-07-10 00:00:00+00:00,5.00000000,put,659.00000000,744.78002930,745.03000000,-0.12270095,0.40010994,0.04500000,0.04500000,0.00000000,86.02198415,0.04500000,86.02198415,0.04500000,1.05586577,0.28566013
2,2026-07-10 00:00:00+00:00,5.00000000,put,660.00000000,744.78002930,745.03000000,-0.12118465,0.39565753,0.04500000,0.04500000,-0.00000000,85.02260040,0.04500000,85.02260040,0.04500000,1.06572725,0.28703299
3,2026-07-10 00:00:00+00:00,5.00000000,put,661.00000000,744.78002930,745.03000000,-0.11967065,0.39120764,0.04500000,0.04500000,0.00000000,84.02321665,0.04500000,84.02321665,0.04500000,1.07578296,0.28842617
4,2026-07-10 00:00:00+00:00,5.00000000,put,662.00000000,744.78002930,745.03000000,-0.11815893,0.38676019,0.04500000,0.04500000,-0.00000000,83.02383290,0.04500000,83.02383290,0.04500000,1.08603907,0.28984024


In [5]:
# ============================================================
# Pointwise no-arbitrage bounds diagnostics
# ============================================================
# This cell checks whether selected market prices, parity-equivalent prices,
# and BSM-reconstructed prices satisfy basic no-arbitrage bounds.
#
# Under continuous dividend yield:
#
#   discounted_spot   = S * exp(-qT)
#   discounted_strike = K * exp(-rT)
#
# Call bounds:
#   max(discounted_spot - discounted_strike, 0) <= C <= discounted_spot
#
# Put bounds:
#   max(discounted_strike - discounted_spot, 0) <= P <= discounted_strike
#
# These are pointwise checks only. Passing this cell does not imply that the
# full surface is free of monotonicity, convexity, or calendar violations.


surface_bounds = surface_px.copy()


# ------------------------------------------------------------
# Pointwise lower and upper bounds
# ------------------------------------------------------------

surface_bounds["call_lower_bound"] = np.maximum(
    surface_bounds["discounted_spot"] - surface_bounds["discounted_strike"],
    0.0,
)

surface_bounds["call_upper_bound"] = surface_bounds["discounted_spot"]

surface_bounds["put_lower_bound"] = np.maximum(
    surface_bounds["discounted_strike"] - surface_bounds["discounted_spot"],
    0.0,
)

surface_bounds["put_upper_bound"] = surface_bounds["discounted_strike"]

surface_bounds["selected_lower_bound"] = np.where(
    surface_bounds["is_call"],
    surface_bounds["call_lower_bound"],
    np.where(surface_bounds["is_put"], surface_bounds["put_lower_bound"], np.nan),
)

surface_bounds["selected_upper_bound"] = np.where(
    surface_bounds["is_call"],
    surface_bounds["call_upper_bound"],
    np.where(surface_bounds["is_put"], surface_bounds["put_upper_bound"], np.nan),
)


# ------------------------------------------------------------
# Generic bound-check helpers
# ------------------------------------------------------------

def bound_tolerance(
    price: pd.Series,
    lower_bound: pd.Series,
    upper_bound: pd.Series,
    abs_tol: float = TOL.price_abs_tol,
    rel_tol: float = TOL.price_rel_tol,
) -> pd.Series:
    """
    Combined absolute-relative tolerance for price-bound diagnostics.
    """
    scale = pd.concat(
        [
            price.abs(),
            lower_bound.abs(),
            upper_bound.abs(),
            pd.Series(1.0, index=price.index),
        ],
        axis=1,
    ).max(axis=1)

    return abs_tol + rel_tol * scale


def add_bounds_diagnostics(
    df: pd.DataFrame,
    price_col: str,
    lower_col: str,
    upper_col: str,
    prefix: str,
) -> pd.DataFrame:
    """
    Add lower/upper bound diagnostics for a chosen price lens.
    """
    out = df.copy()

    price = pd.to_numeric(out[price_col], errors="coerce")
    lower = pd.to_numeric(out[lower_col], errors="coerce")
    upper = pd.to_numeric(out[upper_col], errors="coerce")

    tol = bound_tolerance(price, lower, upper)

    lower_gap = lower - price
    upper_gap = price - upper

    out[f"{prefix}_bound_tol"] = tol
    out[f"{prefix}_lower_gap"] = lower_gap
    out[f"{prefix}_upper_gap"] = upper_gap

    out[f"{prefix}_lower_violation_amount"] = np.maximum(lower_gap - tol, 0.0)
    out[f"{prefix}_upper_violation_amount"] = np.maximum(upper_gap - tol, 0.0)

    out[f"{prefix}_lower_bound_violation"] = out[f"{prefix}_lower_violation_amount"] > 0
    out[f"{prefix}_upper_bound_violation"] = out[f"{prefix}_upper_violation_amount"] > 0

    out[f"{prefix}_bound_violation"] = (
        out[f"{prefix}_lower_bound_violation"]
        | out[f"{prefix}_upper_bound_violation"]
    )

    out[f"{prefix}_max_bound_violation_amount"] = out[
        [
            f"{prefix}_lower_violation_amount",
            f"{prefix}_upper_violation_amount",
        ]
    ].max(axis=1)

    out[f"{prefix}_inside_bounds"] = ~out[f"{prefix}_bound_violation"]

    return out


# ------------------------------------------------------------
# Apply diagnostics to each price lens
# ------------------------------------------------------------

BOUND_LENSES = [
    {
        "prefix": "market_selected",
        "price_col": "market_selected_price",
        "lower_col": "selected_lower_bound",
        "upper_col": "selected_upper_bound",
        "description": "N09 selected market mid on selected call/put side",
    },
    {
        "prefix": "bsm_selected",
        "price_col": "bsm_selected_price",
        "lower_col": "selected_lower_bound",
        "upper_col": "selected_upper_bound",
        "description": "BSM selected-side price reconstructed from selected IV",
    },
    {
        "prefix": "market_call_equiv",
        "price_col": "market_call_equiv_price",
        "lower_col": "call_lower_bound",
        "upper_col": "call_upper_bound",
        "description": "selected market price converted to call-equivalent value by parity",
    },
    {
        "prefix": "market_put_equiv",
        "price_col": "market_put_equiv_price",
        "lower_col": "put_lower_bound",
        "upper_col": "put_upper_bound",
        "description": "selected market price converted to put-equivalent value by parity",
    },
    {
        "prefix": "bsm_call_equiv",
        "price_col": "bsm_call_equiv_price",
        "lower_col": "call_lower_bound",
        "upper_col": "call_upper_bound",
        "description": "BSM call price reconstructed from selected IV",
    },
    {
        "prefix": "bsm_put_equiv",
        "price_col": "bsm_put_equiv_price",
        "lower_col": "put_lower_bound",
        "upper_col": "put_upper_bound",
        "description": "BSM put price reconstructed from selected IV",
    },
]

for lens in BOUND_LENSES:
    surface_bounds = add_bounds_diagnostics(
        df=surface_bounds,
        price_col=lens["price_col"],
        lower_col=lens["lower_col"],
        upper_col=lens["upper_col"],
        prefix=lens["prefix"],
    )


# ------------------------------------------------------------
# Build row-level pointwise violation ledger
# ------------------------------------------------------------

def build_pointwise_violation_ledger(df: pd.DataFrame, lenses: List[Dict[str, str]]) -> pd.DataFrame:
    """
    Convert wide bound-diagnostic flags into a long-form violation ledger.
    """
    rows: List[pd.DataFrame] = []

    base_cols = [
        "n10_row_id",
        "expiry",
        "expiry_rank",
        "dte_calendar",
        "tau_years",
        "maturity_bucket",
        "strike",
        "spot",
        "forward",
        "log_moneyness",
        "abs_log_moneyness",
        "moneyness_bucket_n10",
        "option_type",
        "selected_iv",
        "total_variance",
        "diagnostic_weight",
        "bid",
        "ask",
        "price_spread",
        "relative_price_spread",
        "iv_uncertainty_width",
        "relative_iv_uncertainty_width",
        "surface_row_source",
        "primary_pair_issue",
        "primary_stability_issue",
        "quality_score",
    ]

    available_base_cols = [col for col in base_cols if col in df.columns]

    for lens in lenses:
        prefix = lens["prefix"]

        violation_mask = df[f"{prefix}_bound_violation"]

        if not violation_mask.any():
            continue

        temp = df.loc[violation_mask, available_base_cols].copy()

        temp["diagnostic_type"] = "pointwise_bounds"
        temp["price_lens"] = prefix
        temp["price_lens_description"] = lens["description"]
        temp["tested_price"] = df.loc[violation_mask, lens["price_col"]].to_numpy()
        temp["lower_bound"] = df.loc[violation_mask, lens["lower_col"]].to_numpy()
        temp["upper_bound"] = df.loc[violation_mask, lens["upper_col"]].to_numpy()
        temp["bound_tolerance"] = df.loc[violation_mask, f"{prefix}_bound_tol"].to_numpy()
        temp["lower_gap"] = df.loc[violation_mask, f"{prefix}_lower_gap"].to_numpy()
        temp["upper_gap"] = df.loc[violation_mask, f"{prefix}_upper_gap"].to_numpy()
        temp["lower_violation_amount"] = df.loc[violation_mask, f"{prefix}_lower_violation_amount"].to_numpy()
        temp["upper_violation_amount"] = df.loc[violation_mask, f"{prefix}_upper_violation_amount"].to_numpy()
        temp["max_violation_amount"] = df.loc[violation_mask, f"{prefix}_max_bound_violation_amount"].to_numpy()

        temp["violation_side"] = np.select(
            [
                temp["lower_violation_amount"] > 0,
                temp["upper_violation_amount"] > 0,
            ],
            [
                "below_lower_bound",
                "above_upper_bound",
            ],
            default="none",
        )

        rows.append(temp)

    if not rows:
        return pd.DataFrame(
            columns=[
                *available_base_cols,
                "diagnostic_type",
                "price_lens",
                "price_lens_description",
                "tested_price",
                "lower_bound",
                "upper_bound",
                "bound_tolerance",
                "lower_gap",
                "upper_gap",
                "lower_violation_amount",
                "upper_violation_amount",
                "max_violation_amount",
                "violation_side",
            ]
        )

    return (
        pd.concat(rows, axis=0, ignore_index=True)
        .sort_values(["max_violation_amount", "expiry_rank", "strike"], ascending=[False, True, True])
        .reset_index(drop=True)
    )


pointwise_violation_ledger = build_pointwise_violation_ledger(surface_bounds, BOUND_LENSES)


# ------------------------------------------------------------
# Pointwise summaries
# ------------------------------------------------------------

pointwise_lens_summary_rows: List[Dict[str, Any]] = []

for lens in BOUND_LENSES:
    prefix = lens["prefix"]
    price_col = lens["price_col"]

    pointwise_lens_summary_rows.append(
        {
            "price_lens": prefix,
            "description": lens["description"],
            "tested_rows": int(np.isfinite(surface_bounds[price_col]).sum()),
            "inside_bounds_rows": int(surface_bounds[f"{prefix}_inside_bounds"].sum()),
            "violation_rows": int(surface_bounds[f"{prefix}_bound_violation"].sum()),
            "lower_bound_violations": int(surface_bounds[f"{prefix}_lower_bound_violation"].sum()),
            "upper_bound_violations": int(surface_bounds[f"{prefix}_upper_bound_violation"].sum()),
            "max_violation_amount": float(surface_bounds[f"{prefix}_max_bound_violation_amount"].max()),
            "median_violation_amount_all_rows": float(surface_bounds[f"{prefix}_max_bound_violation_amount"].median()),
            "mean_violation_amount_all_rows": float(surface_bounds[f"{prefix}_max_bound_violation_amount"].mean()),
        }
    )

pointwise_lens_summary = pd.DataFrame(pointwise_lens_summary_rows)

pointwise_expiry_summary_parts: List[pd.DataFrame] = []

for lens in BOUND_LENSES:
    prefix = lens["prefix"]

    temp = (
        surface_bounds
        .groupby(["expiry", "expiry_rank", "maturity_bucket"], dropna=False)
        .agg(
            dte_calendar=("dte_calendar", "median"),
            n_obs=("n10_row_id", "size"),
            violation_rows=(f"{prefix}_bound_violation", "sum"),
            lower_bound_violations=(f"{prefix}_lower_bound_violation", "sum"),
            upper_bound_violations=(f"{prefix}_upper_bound_violation", "sum"),
            max_violation_amount=(f"{prefix}_max_bound_violation_amount", "max"),
            mean_violation_amount=(f"{prefix}_max_bound_violation_amount", "mean"),
            median_violation_amount=(f"{prefix}_max_bound_violation_amount", "median"),
        )
        .reset_index()
    )

    temp["price_lens"] = prefix
    pointwise_expiry_summary_parts.append(temp)

pointwise_expiry_summary = (
    pd.concat(pointwise_expiry_summary_parts, axis=0, ignore_index=True)
    .sort_values(["price_lens", "expiry_rank"])
    .reset_index(drop=True)
)

pointwise_moneyness_summary_parts: List[pd.DataFrame] = []

for lens in BOUND_LENSES:
    prefix = lens["prefix"]

    temp = (
        surface_bounds
        .groupby(["maturity_bucket", "moneyness_bucket_n10"], dropna=False)
        .agg(
            n_obs=("n10_row_id", "size"),
            violation_rows=(f"{prefix}_bound_violation", "sum"),
            lower_bound_violations=(f"{prefix}_lower_bound_violation", "sum"),
            upper_bound_violations=(f"{prefix}_upper_bound_violation", "sum"),
            max_violation_amount=(f"{prefix}_max_bound_violation_amount", "max"),
            mean_violation_amount=(f"{prefix}_max_bound_violation_amount", "mean"),
        )
        .reset_index()
    )

    temp["price_lens"] = prefix
    pointwise_moneyness_summary_parts.append(temp)

pointwise_moneyness_summary = (
    pd.concat(pointwise_moneyness_summary_parts, axis=0, ignore_index=True)
    .sort_values(["price_lens", "maturity_bucket", "moneyness_bucket_n10"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validation ledger
# ------------------------------------------------------------

pointwise_validation_rows = [
    validation_row(
        "pointwise_bounds_computed",
        True,
        f"{len(BOUND_LENSES):,} price lenses tested",
    ),
    validation_row(
        "market_selected_prices_inside_bounds",
        int(surface_bounds["market_selected_bound_violation"].sum()) == 0,
        f"{int(surface_bounds['market_selected_bound_violation'].sum()):,} violations",
    ),
    validation_row(
        "bsm_selected_prices_inside_bounds",
        int(surface_bounds["bsm_selected_bound_violation"].sum()) == 0,
        f"{int(surface_bounds['bsm_selected_bound_violation'].sum()):,} violations",
    ),
    validation_row(
        "market_call_equiv_prices_inside_bounds",
        int(surface_bounds["market_call_equiv_bound_violation"].sum()) == 0,
        f"{int(surface_bounds['market_call_equiv_bound_violation'].sum()):,} violations",
    ),
    validation_row(
        "market_put_equiv_prices_inside_bounds",
        int(surface_bounds["market_put_equiv_bound_violation"].sum()) == 0,
        f"{int(surface_bounds['market_put_equiv_bound_violation'].sum()):,} violations",
    ),
    validation_row(
        "bsm_call_equiv_prices_inside_bounds",
        int(surface_bounds["bsm_call_equiv_bound_violation"].sum()) == 0,
        f"{int(surface_bounds['bsm_call_equiv_bound_violation'].sum()):,} violations",
    ),
    validation_row(
        "bsm_put_equiv_prices_inside_bounds",
        int(surface_bounds["bsm_put_equiv_bound_violation"].sum()) == 0,
        f"{int(surface_bounds['bsm_put_equiv_bound_violation'].sum()):,} violations",
    ),
]

pointwise_validation_ledger = pd.DataFrame(pointwise_validation_rows)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("Pointwise bounds validation ledger")
print("=" * 90)
display(pointwise_validation_ledger)

print("=" * 90)
print("Pointwise bounds summary by price lens")
print("=" * 90)
display(pointwise_lens_summary)

print("=" * 90)
print("Pointwise bounds summary by expiry and price lens")
print("=" * 90)
display(pointwise_expiry_summary)

print("=" * 90)
print("Pointwise bounds summary by maturity / moneyness bucket")
print("=" * 90)
display(pointwise_moneyness_summary)

print("=" * 90)
print("Pointwise violation ledger")
print("=" * 90)

if len(pointwise_violation_ledger) == 0:
    print("No pointwise bound violations detected under configured tolerances.")
else:
    display(pointwise_violation_ledger.head(50))


RUN_METADATA["pointwise_bounds_validation"] = pointwise_validation_ledger.to_dict(orient="records")
RUN_METADATA["pointwise_bounds_lens_summary"] = pointwise_lens_summary.to_dict(orient="records")

surface_bounds[
    [
        "expiry",
        "dte_calendar",
        "option_type",
        "strike",
        "spot",
        "forward",
        "log_moneyness",
        "selected_iv",
        "market_selected_price",
        "selected_lower_bound",
        "selected_upper_bound",
        "market_selected_bound_violation",
        "market_call_equiv_price",
        "call_lower_bound",
        "call_upper_bound",
        "market_call_equiv_bound_violation",
        "market_put_equiv_price",
        "put_lower_bound",
        "put_upper_bound",
        "market_put_equiv_bound_violation",
        "bsm_call_equiv_price",
        "bsm_put_equiv_price",
        "diagnostic_weight",
    ]
].head()

Pointwise bounds validation ledger


,check,passed,details
0,pointwise_bounds_computed,True,6 price lenses tested
1,market_selected_prices_inside_bounds,True,0 violations
2,bsm_selected_prices_inside_bounds,True,0 violations
3,market_call_equiv_prices_inside_bounds,True,0 violations
4,market_put_equiv_prices_inside_bounds,True,0 violations
5,bsm_call_equiv_prices_inside_bounds,True,0 violations
6,bsm_put_equiv_prices_inside_bounds,True,0 violations


Pointwise bounds summary by price lens


,price_lens,description,tested_rows,inside_bounds_rows,violation_rows,lower_bound_violations,upper_bound_violations,max_violation_amount,median_violation_amount_all_rows,mean_violation_amount_all_rows
0,market_selected,N09 selected market mid on selected call/put side,1333,1333,0,0,0,0.00000000,0.00000000,0.00000000
1,bsm_selected,BSM selected-side price reconstructed from sel...,1333,1333,0,0,0,0.00000000,0.00000000,0.00000000
2,market_call_equiv,selected market price converted to call-equiva...,1333,1333,0,0,0,0.00000000,0.00000000,0.00000000
3,market_put_equiv,selected market price converted to put-equival...,1333,1333,0,0,0,0.00000000,0.00000000,0.00000000
4,bsm_call_equiv,BSM call price reconstructed from selected IV,1333,1333,0,0,0,0.00000000,0.00000000,0.00000000
5,bsm_put_equiv,BSM put price reconstructed from selected IV,1333,1333,0,0,0,0.00000000,0.00000000,0.00000000


Pointwise bounds summary by expiry and price lens


,expiry,expiry_rank,maturity_bucket,dte_calendar,n_obs,violation_rows,lower_bound_violations,upper_bound_violations,max_violation_amount,mean_violation_amount,median_violation_amount,price_lens
0,2026-07-10 00:00:00+00:00,0,ultra_short,5.00000000,117,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
1,2026-07-17 00:00:00+00:00,1,short,12.00000000,154,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
2,2026-07-24 00:00:00+00:00,2,short,19.00000000,168,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
3,2026-07-31 00:00:00+00:00,3,front_intermediate,26.00000000,183,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
4,2026-08-07 00:00:00+00:00,4,front_intermediate,33.00000000,132,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
5,2026-08-21 00:00:00+00:00,5,intermediate,47.00000000,173,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
6,2026-08-31 00:00:00+00:00,6,intermediate,57.00000000,187,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
7,2026-09-18 00:00:00+00:00,7,intermediate,75.00000000,219,0,0,0,0.00000000,0.00000000,0.00000000,bsm_call_equiv
8,2026-07-10 00:00:00+00:00,0,ultra_short,5.00000000,117,0,0,0,0.00000000,0.00000000,0.00000000,bsm_put_equiv
9,2026-07-17 00:00:00+00:00,1,short,12.00000000,154,0,0,0,0.00000000,0.00000000,0.00000000,bsm_put_equiv


Pointwise bounds summary by maturity / moneyness bucket


,maturity_bucket,moneyness_bucket_n10,n_obs,violation_rows,lower_bound_violations,upper_bound_violations,max_violation_amount,mean_violation_amount,price_lens
0,front_intermediate,atm,66,0,0,0,0.00000000,0.00000000,bsm_call_equiv
1,front_intermediate,call_wing,6,0,0,0,0.00000000,0.00000000,bsm_call_equiv
2,front_intermediate,deep_put_wing,61,0,0,0,0.00000000,0.00000000,bsm_call_equiv
3,front_intermediate,near_atm_call,46,0,0,0,0.00000000,0.00000000,bsm_call_equiv
4,front_intermediate,near_atm_put,70,0,0,0,0.00000000,0.00000000,bsm_call_equiv
...,...,...,...,...,...,...,...,...,...
127,short,put_wing,87,0,0,0,0.00000000,0.00000000,market_selected
128,ultra_short,atm,43,0,0,0,0.00000000,0.00000000,market_selected
129,ultra_short,near_atm_call,5,0,0,0,0.00000000,0.00000000,market_selected
130,ultra_short,near_atm_put,35,0,0,0,0.00000000,0.00000000,market_selected


Pointwise violation ledger
No pointwise bound violations detected under configured tolerances.


,expiry,dte_calendar,option_type,strike,spot,forward,log_moneyness,selected_iv,market_selected_price,selected_lower_bound,selected_upper_bound,market_selected_bound_violation,market_call_equiv_price,call_lower_bound,call_upper_bound,market_call_equiv_bound_violation,market_put_equiv_price,put_lower_bound,put_upper_bound,market_put_equiv_bound_violation,bsm_call_equiv_price,bsm_put_equiv_price,diagnostic_weight
0,2026-07-10 00:00:00+00:00,5.00000000,put,658.00000000,744.78002930,745.03000000,-0.12421955,0.40456494,0.04500000,0.00000000,657.59450855,False,87.02136790,86.97636790,744.57087646,False,0.04500000,0.00000000,657.59450855,False,87.02136790,0.04500000,0.28430707
1,2026-07-10 00:00:00+00:00,5.00000000,put,659.00000000,744.78002930,745.03000000,-0.12270095,0.40010994,0.04500000,0.00000000,658.59389231,False,86.02198415,85.97698415,744.57087646,False,0.04500000,0.00000000,658.59389231,False,86.02198415,0.04500000,0.28566013
2,2026-07-10 00:00:00+00:00,5.00000000,put,660.00000000,744.78002930,745.03000000,-0.12118465,0.39565753,0.04500000,0.00000000,659.59327606,False,85.02260040,84.97760040,744.57087646,False,0.04500000,0.00000000,659.59327606,False,85.02260040,0.04500000,0.28703299
3,2026-07-10 00:00:00+00:00,5.00000000,put,661.00000000,744.78002930,745.03000000,-0.11967065,0.39120764,0.04500000,0.00000000,660.59265981,False,84.02321665,83.97821665,744.57087646,False,0.04500000,0.00000000,660.59265981,False,84.02321665,0.04500000,0.28842617
4,2026-07-10 00:00:00+00:00,5.00000000,put,662.00000000,744.78002930,745.03000000,-0.11815893,0.38676019,0.04500000,0.00000000,661.59204356,False,83.02383290,82.97883290,744.57087646,False,0.04500000,0.00000000,661.59204356,False,83.02383290,0.04500000,0.28984024


In [6]:
# ============================================================
# Strike monotonicity diagnostics by expiry
# ============================================================
# This cell tests the first cross-sectional static-arbitrage condition.
#
# For a fixed expiry:
#
#   call-equivalent prices should be non-increasing in strike;
#   put-equivalent prices should be non-decreasing in strike.
#
# Important discipline:
# We do NOT test raw selected-side prices directly because Notebook 09 selected
# calls on one side of the surface and puts on the other. Mixing selected calls
# and selected puts across strike would create artificial monotonicity failures.
#
# Instead, this cell tests parity-equivalent call and put price lenses:
#
#   market_call_equiv_price
#   market_put_equiv_price
#   bsm_call_equiv_price
#   bsm_put_equiv_price
#
# The market-equivalent and BSM-equivalent lenses will often be nearly identical
# because Notebook 09 inverted IV from the selected mid price. Keeping both lenses
# is still useful because it preserves the distinction between quote-space and
# IV-reconstructed price-space diagnostics.


surface_monotonicity = surface_bounds.copy()


# ------------------------------------------------------------
# Monotonicity lens configuration
# ------------------------------------------------------------

MONOTONICITY_LENSES = [
    {
        "price_lens": "market_call_equiv",
        "price_col": "market_call_equiv_price",
        "expected_direction": "non_increasing",
        "description": "selected market price converted to call-equivalent value by parity",
    },
    {
        "price_lens": "bsm_call_equiv",
        "price_col": "bsm_call_equiv_price",
        "expected_direction": "non_increasing",
        "description": "BSM call price reconstructed from selected IV",
    },
    {
        "price_lens": "market_put_equiv",
        "price_col": "market_put_equiv_price",
        "expected_direction": "non_decreasing",
        "description": "selected market price converted to put-equivalent value by parity",
    },
    {
        "price_lens": "bsm_put_equiv",
        "price_col": "bsm_put_equiv_price",
        "expected_direction": "non_decreasing",
        "description": "BSM put price reconstructed from selected IV",
    },
]


# ------------------------------------------------------------
# Tolerance and severity helpers
# ------------------------------------------------------------

def monotonicity_tolerance_pair(
    price_left: float,
    price_right: float,
    abs_tol: float = TOL.monotonicity_abs_tol,
    rel_tol: float = TOL.monotonicity_rel_tol,
) -> float:
    """
    Absolute-relative tolerance for adjacent strike monotonicity checks.
    """
    scale = max(abs(price_left), abs(price_right), 1.0)
    return abs_tol + rel_tol * scale


def classify_monotonicity_severity(violation_amount: float) -> str:
    """
    Classify monotonicity violation severity using monotonicity_tolerance_pair(
    price_left: float,
    price_right: float,
    abs_tol: float = TOL.monotonic visible notebook thresholds.
    """
    if not np.isfinite(violation_amount) or violation_amount <= 0:
        return "clean"

    if violation_amount <= TOL.mild_violation_threshold:
        return "micro_noise"

    if violation_amount <= TOL.moderate_violation_threshold:
        return "mild_warning"

    if violation_amount <= TOL.severe_violation_threshold:
        return "moderate_warning"

    return "severe_violation"


def make_adjacent_pair_frame(
    df: pd.DataFrame,
    price_col: str,
    price_lens: str,
    expected_direction: str,
    description: str,
) -> pd.DataFrame:
    """
    Construct adjacent strike-pair diagnostics for one price lens.

    For each expiry, sort by strike and compare neighboring prices.
    """
    rows: List[Dict[str, Any]] = []

    required_cols = [
        "n10_row_id",
        "expiry",
        "expiry_rank",
        "dte_calendar",
        "tau_years",
        "maturity_bucket",
        "strike",
        "spot",
        "forward",
        "log_moneyness",
        "abs_log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "total_variance",
        "diagnostic_weight",
        "price_spread",
        "relative_price_spread",
        "iv_uncertainty_width",
        "relative_iv_uncertainty_width",
        "surface_row_source",
        "primary_pair_issue",
        "primary_stability_issue",
        "quality_score",
        price_col,
    ]

    available_cols = [col for col in required_cols if col in df.columns]

    working = (
        df.loc[df["eligible_cross_section_diagnostics"], available_cols]
        .copy()
        .sort_values(["expiry", "strike", "n10_row_id"])
        .reset_index(drop=True)
    )

    for expiry, group in working.groupby("expiry", sort=True):
        group = group.sort_values(["strike", "n10_row_id"]).reset_index(drop=True)

        if len(group) < 2:
            continue

        for i in range(len(group) - 1):
            left = group.iloc[i]
            right = group.iloc[i + 1]

            strike_left = float(left["strike"])
            strike_right = float(right["strike"])

            if not np.isfinite(strike_left) or not np.isfinite(strike_right):
                continue

            if strike_right <= strike_left:
                continue

            price_left = float(left[price_col])
            price_right = float(right[price_col])

            if not np.isfinite(price_left) or not np.isfinite(price_right):
                continue

            price_change = price_right - price_left
            tolerance = monotonicity_tolerance_pair(price_left, price_right)

            if expected_direction == "non_increasing":
                raw_violation_amount = price_change
                violation_amount = max(raw_violation_amount - tolerance, 0.0)
                expected_relation = "price_right <= price_left"
            elif expected_direction == "non_decreasing":
                raw_violation_amount = -price_change
                violation_amount = max(raw_violation_amount - tolerance, 0.0)
                expected_relation = "price_right >= price_left"
            else:
                raise ValueError(f"Unsupported expected_direction: {expected_direction}")

            is_violation = violation_amount > 0
            severity = classify_monotonicity_severity(violation_amount)

            avg_price = 0.5 * (abs(price_left) + abs(price_right))
            avg_spread = np.nanmean(
                [
                    float(left.get("price_spread", np.nan)),
                    float(right.get("price_spread", np.nan)),
                ]
            )
            avg_iv_uncertainty = np.nanmean(
                [
                    float(left.get("iv_uncertainty_width", np.nan)),
                    float(right.get("iv_uncertainty_width", np.nan)),
                ]
            )

            normalized_violation_by_price = violation_amount / max(avg_price, TOL.eps)
            normalized_violation_by_spread = violation_amount / max(avg_spread, TOL.eps) if np.isfinite(avg_spread) else np.nan
            normalized_violation_by_iv_uncertainty = (
                violation_amount / max(avg_iv_uncertainty, TOL.eps)
                if np.isfinite(avg_iv_uncertainty)
                else np.nan
            )

            rows.append(
                {
                    "diagnostic_type": "strike_monotonicity",
                    "price_lens": price_lens,
                    "price_lens_description": description,
                    "expected_direction": expected_direction,
                    "expected_relation": expected_relation,
                    "expiry": expiry,
                    "expiry_rank": int(left["expiry_rank"]),
                    "dte_calendar": float(left["dte_calendar"]),
                    "tau_years": float(left["tau_years"]),
                    "maturity_bucket": left["maturity_bucket"],
                    "left_n10_row_id": int(left["n10_row_id"]),
                    "right_n10_row_id": int(right["n10_row_id"]),
                    "left_strike": strike_left,
                    "right_strike": strike_right,
                    "strike_gap": strike_right - strike_left,
                    "left_log_moneyness": float(left["log_moneyness"]),
                    "right_log_moneyness": float(right["log_moneyness"]),
                    "mid_log_moneyness": 0.5 * (float(left["log_moneyness"]) + float(right["log_moneyness"])),
                    "left_moneyness_bucket": left["moneyness_bucket_n10"],
                    "right_moneyness_bucket": right["moneyness_bucket_n10"],
                    "left_price": price_left,
                    "right_price": price_right,
                    "price_change_right_minus_left": price_change,
                    "tolerance": tolerance,
                    "raw_violation_amount_before_tolerance": raw_violation_amount,
                    "violation_amount": violation_amount,
                    "is_violation": bool(is_violation),
                    "severity": severity,
                    "normalized_violation_by_price": normalized_violation_by_price,
                    "normalized_violation_by_spread": normalized_violation_by_spread,
                    "normalized_violation_by_iv_uncertainty": normalized_violation_by_iv_uncertainty,
                    "left_selected_iv": float(left["selected_iv"]),
                    "right_selected_iv": float(right["selected_iv"]),
                    "left_total_variance": float(left["total_variance"]),
                    "right_total_variance": float(right["total_variance"]),
                    "left_diagnostic_weight": float(left["diagnostic_weight"]),
                    "right_diagnostic_weight": float(right["diagnostic_weight"]),
                    "avg_diagnostic_weight": np.nanmean(
                        [
                            float(left["diagnostic_weight"]),
                            float(right["diagnostic_weight"]),
                        ]
                    ),
                    "left_price_spread": float(left.get("price_spread", np.nan)),
                    "right_price_spread": float(right.get("price_spread", np.nan)),
                    "avg_price_spread": avg_spread,
                    "left_relative_price_spread": float(left.get("relative_price_spread", np.nan)),
                    "right_relative_price_spread": float(right.get("relative_price_spread", np.nan)),
                    "left_iv_uncertainty_width": float(left.get("iv_uncertainty_width", np.nan)),
                    "right_iv_uncertainty_width": float(right.get("iv_uncertainty_width", np.nan)),
                    "avg_iv_uncertainty_width": avg_iv_uncertainty,
                    "left_surface_row_source": left.get("surface_row_source", None),
                    "right_surface_row_source": right.get("surface_row_source", None),
                    "left_primary_pair_issue": left.get("primary_pair_issue", None),
                    "right_primary_pair_issue": right.get("primary_pair_issue", None),
                    "left_primary_stability_issue": left.get("primary_stability_issue", None),
                    "right_primary_stability_issue": right.get("primary_stability_issue", None),
                    "left_quality_score": float(left.get("quality_score", np.nan)),
                    "right_quality_score": float(right.get("quality_score", np.nan)),
                }
            )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Run adjacent strike-pair diagnostics
# ------------------------------------------------------------

monotonicity_pair_frames: List[pd.DataFrame] = []

for lens in MONOTONICITY_LENSES:
    lens_pairs = make_adjacent_pair_frame(
        df=surface_monotonicity,
        price_col=lens["price_col"],
        price_lens=lens["price_lens"],
        expected_direction=lens["expected_direction"],
        description=lens["description"],
    )
    monotonicity_pair_frames.append(lens_pairs)

monotonicity_pair_diagnostics = pd.concat(
    monotonicity_pair_frames,
    axis=0,
    ignore_index=True,
)

monotonicity_violation_ledger = (
    monotonicity_pair_diagnostics
    .loc[monotonicity_pair_diagnostics["is_violation"]]
    .sort_values(
        [
            "violation_amount",
            "normalized_violation_by_price",
            "expiry_rank",
            "left_strike",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Attach row-level monotonicity exposure flags back to surface
# ------------------------------------------------------------

for lens in MONOTONICITY_LENSES:
    price_lens = lens["price_lens"]

    surface_monotonicity[f"{price_lens}_monotonicity_pair_count"] = 0
    surface_monotonicity[f"{price_lens}_monotonicity_violation_pair_count"] = 0
    surface_monotonicity[f"{price_lens}_max_monotonicity_violation_amount"] = 0.0

for lens in MONOTONICITY_LENSES:
    price_lens = lens["price_lens"]

    lens_pairs = monotonicity_pair_diagnostics.loc[
        monotonicity_pair_diagnostics["price_lens"].eq(price_lens)
    ]

    if lens_pairs.empty:
        continue

    endpoint_records = []

    for side in ["left", "right"]:
        temp = lens_pairs[
            [
                f"{side}_n10_row_id",
                "is_violation",
                "violation_amount",
            ]
        ].rename(columns={f"{side}_n10_row_id": "n10_row_id"})

        endpoint_records.append(temp)

    endpoint_df = pd.concat(endpoint_records, axis=0, ignore_index=True)

    row_summary = (
        endpoint_df
        .groupby("n10_row_id")
        .agg(
            pair_count=("is_violation", "size"),
            violation_pair_count=("is_violation", "sum"),
            max_violation_amount=("violation_amount", "max"),
        )
        .reset_index()
    )

    surface_monotonicity = surface_monotonicity.merge(
        row_summary,
        on="n10_row_id",
        how="left",
    )

    surface_monotonicity[f"{price_lens}_monotonicity_pair_count"] = (
        surface_monotonicity["pair_count"].fillna(0).astype(int)
    )
    surface_monotonicity[f"{price_lens}_monotonicity_violation_pair_count"] = (
        surface_monotonicity["violation_pair_count"].fillna(0).astype(int)
    )
    surface_monotonicity[f"{price_lens}_max_monotonicity_violation_amount"] = (
        surface_monotonicity["max_violation_amount"].fillna(0.0)
    )

    surface_monotonicity = surface_monotonicity.drop(
        columns=["pair_count", "violation_pair_count", "max_violation_amount"],
        errors="ignore",
    )

surface_monotonicity["any_monotonicity_violation_touching_row"] = False

for lens in MONOTONICITY_LENSES:
    price_lens = lens["price_lens"]
    surface_monotonicity["any_monotonicity_violation_touching_row"] = (
        surface_monotonicity["any_monotonicity_violation_touching_row"]
        | (surface_monotonicity[f"{price_lens}_monotonicity_violation_pair_count"] > 0)
    )


# ------------------------------------------------------------
# Monotonicity summaries
# ------------------------------------------------------------

monotonicity_lens_summary = (
    monotonicity_pair_diagnostics
    .groupby(["price_lens", "expected_direction"], dropna=False)
    .agg(
        tested_pairs=("is_violation", "size"),
        violation_pairs=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_normalized_violation_by_price=("normalized_violation_by_price", "max"),
        max_normalized_violation_by_spread=("normalized_violation_by_spread", "max"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
)

monotonicity_lens_summary["violation_rate"] = (
    monotonicity_lens_summary["violation_pairs"]
    / monotonicity_lens_summary["tested_pairs"].clip(lower=1)
)

monotonicity_expiry_summary = (
    monotonicity_pair_diagnostics
    .groupby(["price_lens", "expected_direction", "expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        tested_pairs=("is_violation", "size"),
        violation_pairs=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_normalized_violation_by_price=("normalized_violation_by_price", "max"),
        max_normalized_violation_by_spread=("normalized_violation_by_spread", "max"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["price_lens", "expiry_rank"])
)

monotonicity_expiry_summary["violation_rate"] = (
    monotonicity_expiry_summary["violation_pairs"]
    / monotonicity_expiry_summary["tested_pairs"].clip(lower=1)
)

monotonicity_bucket_summary = (
    monotonicity_pair_diagnostics
    .assign(
        mid_moneyness_bucket=lambda x: x["mid_log_moneyness"].map(assign_moneyness_bucket)
    )
    .groupby(["price_lens", "expected_direction", "maturity_bucket", "mid_moneyness_bucket"], dropna=False)
    .agg(
        tested_pairs=("is_violation", "size"),
        violation_pairs=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_normalized_violation_by_price=("normalized_violation_by_price", "max"),
        max_normalized_violation_by_spread=("normalized_violation_by_spread", "max"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["price_lens", "maturity_bucket", "mid_moneyness_bucket"])
)

monotonicity_bucket_summary["violation_rate"] = (
    monotonicity_bucket_summary["violation_pairs"]
    / monotonicity_bucket_summary["tested_pairs"].clip(lower=1)
)

monotonicity_severity_summary = (
    monotonicity_pair_diagnostics
    .groupby(["price_lens", "severity"], dropna=False)
    .agg(
        pair_count=("is_violation", "size"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
    )
    .reset_index()
)

monotonicity_severity_summary["severity"] = pd.Categorical(
    monotonicity_severity_summary["severity"],
    categories=SEVERITY_ORDER,
    ordered=True,
)

monotonicity_severity_summary = monotonicity_severity_summary.sort_values(
    ["price_lens", "severity"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Validation ledger
# ------------------------------------------------------------

call_market_violations = int(
    monotonicity_lens_summary
    .loc[monotonicity_lens_summary["price_lens"].eq("market_call_equiv"), "violation_pairs"]
    .sum()
)

put_market_violations = int(
    monotonicity_lens_summary
    .loc[monotonicity_lens_summary["price_lens"].eq("market_put_equiv"), "violation_pairs"]
    .sum()
)

call_bsm_violations = int(
    monotonicity_lens_summary
    .loc[monotonicity_lens_summary["price_lens"].eq("bsm_call_equiv"), "violation_pairs"]
    .sum()
)

put_bsm_violations = int(
    monotonicity_lens_summary
    .loc[monotonicity_lens_summary["price_lens"].eq("bsm_put_equiv"), "violation_pairs"]
    .sum()
)

expected_pairs_per_lens = int(
    surface_monotonicity
    .groupby("expiry", dropna=False)
    .size()
    .sub(1)
    .clip(lower=0)
    .sum()
)

monotonicity_validation_rows = [
    validation_row(
        "monotonicity_diagnostics_computed",
        len(monotonicity_pair_diagnostics) > 0,
        f"{len(monotonicity_pair_diagnostics):,} adjacent strike pairs tested across {len(MONOTONICITY_LENSES):,} price lenses",
    ),
    validation_row(
        "expected_pair_count_per_lens_matched",
        bool((monotonicity_lens_summary["tested_pairs"] == expected_pairs_per_lens).all()),
        f"expected {expected_pairs_per_lens:,} adjacent pairs per lens",
    ),
    validation_row(
        "market_call_equiv_monotonicity_clean",
        call_market_violations == 0,
        f"{call_market_violations:,} market call-equivalent violations",
    ),
    validation_row(
        "market_put_equiv_monotonicity_clean",
        put_market_violations == 0,
        f"{put_market_violations:,} market put-equivalent violations",
    ),
    validation_row(
        "bsm_call_equiv_monotonicity_clean",
        call_bsm_violations == 0,
        f"{call_bsm_violations:,} BSM call-equivalent violations",
    ),
    validation_row(
        "bsm_put_equiv_monotonicity_clean",
        put_bsm_violations == 0,
        f"{put_bsm_violations:,} BSM put-equivalent violations",
    ),
    validation_row(
        "row_level_monotonicity_flags_attached",
        "any_monotonicity_violation_touching_row" in surface_monotonicity.columns,
        "row-level endpoint flags attached to surface_monotonicity",
    ),
]

monotonicity_validation_ledger = pd.DataFrame(monotonicity_validation_rows)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("Strike monotonicity validation ledger")
print("=" * 90)
display(monotonicity_validation_ledger)

print("=" * 90)
print("Strike monotonicity summary by price lens")
print("=" * 90)
display(monotonicity_lens_summary)

print("=" * 90)
print("Strike monotonicity summary by expiry")
print("=" * 90)
display(monotonicity_expiry_summary)

print("=" * 90)
print("Strike monotonicity summary by maturity / moneyness bucket")
print("=" * 90)
display(monotonicity_bucket_summary)

print("=" * 90)
print("Strike monotonicity severity summary")
print("=" * 90)
display(monotonicity_severity_summary)

print("=" * 90)
print("Strike monotonicity violation ledger")
print("=" * 90)

if len(monotonicity_violation_ledger) == 0:
    print("No strike monotonicity violations detected under configured tolerances.")
else:
    display(monotonicity_violation_ledger.head(75))


RUN_METADATA["monotonicity_validation"] = monotonicity_validation_ledger.to_dict(orient="records")
RUN_METADATA["monotonicity_lens_summary"] = monotonicity_lens_summary.to_dict(orient="records")

surface_monotonicity[
    [
        "expiry",
        "dte_calendar",
        "strike",
        "log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "market_call_equiv_price",
        "market_put_equiv_price",
        "bsm_call_equiv_price",
        "bsm_put_equiv_price",
        "market_call_equiv_monotonicity_violation_pair_count",
        "market_put_equiv_monotonicity_violation_pair_count",
        "bsm_call_equiv_monotonicity_violation_pair_count",
        "bsm_put_equiv_monotonicity_violation_pair_count",
        "any_monotonicity_violation_touching_row",
        "diagnostic_weight",
    ]
].head()

Strike monotonicity validation ledger


,check,passed,details
0,monotonicity_diagnostics_computed,True,"5,300 adjacent strike pairs tested across 4 pr..."
1,expected_pair_count_per_lens_matched,True,"expected 1,325 adjacent pairs per lens"
2,market_call_equiv_monotonicity_clean,True,0 market call-equivalent violations
3,market_put_equiv_monotonicity_clean,True,0 market put-equivalent violations
4,bsm_call_equiv_monotonicity_clean,True,0 BSM call-equivalent violations
5,bsm_put_equiv_monotonicity_clean,True,0 BSM put-equivalent violations
6,row_level_monotonicity_flags_attached,True,row-level endpoint flags attached to surface_m...


Strike monotonicity summary by price lens


,price_lens,expected_direction,tested_pairs,violation_pairs,max_violation_amount,mean_violation_amount,median_violation_amount,max_normalized_violation_by_price,max_normalized_violation_by_spread,median_avg_diagnostic_weight,violation_rate
0,bsm_call_equiv,non_increasing,1325,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.99060900,0.00000000
1,bsm_put_equiv,non_decreasing,1325,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.99060900,0.00000000
2,market_call_equiv,non_increasing,1325,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.99060900,0.00000000
3,market_put_equiv,non_decreasing,1325,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.99060900,0.00000000


Strike monotonicity summary by expiry


...

,price_lens,expected_direction,maturity_bucket,mid_moneyness_bucket,tested_pairs,violation_pairs,max_violation_amount,mean_violation_amount,median_violation_amount,max_normalized_violation_by_price,max_normalized_violation_by_spread,median_avg_diagnostic_weight,violation_rate
0,bsm_call_equiv,non_increasing,front_intermediate,atm,65,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,1.00000000,0.00000000
1,bsm_call_equiv,non_increasing,front_intermediate,call_wing,4,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.28388049,0.00000000
2,bsm_call_equiv,non_increasing,front_intermediate,deep_put_wing,61,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.42496186,0.00000000
3,bsm_call_equiv,non_increasing,front_intermediate,near_atm_call,48,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.99122037,0.00000000
4,bsm_call_equiv,non_increasing,front_intermediate,near_atm_put,70,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,1.00000000,0.00000000
5,bsm_call_equiv,non_increasing,front_intermediate,put_wing,65,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.91115764,0.00000000
6,bsm_call_equiv,non_increasing,intermediate,atm,83,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,1.00000000,0.00000000
7,bsm_call_equiv,non_increasing,intermediate,call_wing,28,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.59925976,0.00000000
8,bsm_call_equiv,non_increasing,intermediate,deep_call_wing,5,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.18211145,0.00000000
9,bsm_call_equiv,non_increasing,intermediate,deep_put_wing,147,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.64916818,0.00000000


Strike monotonicity severity summary


,price_lens,severity,pair_count,max_violation_amount,mean_violation_amount
0,bsm_call_equiv,clean,1325,0.00000000,0.00000000
1,bsm_put_equiv,clean,1325,0.00000000,0.00000000
2,market_call_equiv,clean,1325,0.00000000,0.00000000
3,market_put_equiv,clean,1325,0.00000000,0.00000000


Strike monotonicity violation ledger
No strike monotonicity violations detected under configured tolerances.


,expiry,dte_calendar,strike,log_moneyness,moneyness_bucket_n10,selected_iv,market_call_equiv_price,market_put_equiv_price,bsm_call_equiv_price,bsm_put_equiv_price,market_call_equiv_monotonicity_violation_pair_count,market_put_equiv_monotonicity_violation_pair_count,bsm_call_equiv_monotonicity_violation_pair_count,bsm_put_equiv_monotonicity_violation_pair_count,any_monotonicity_violation_touching_row,diagnostic_weight
0,2026-07-10 00:00:00+00:00,5.00000000,658.00000000,-0.12421955,put_wing,0.40456494,87.02136790,0.04500000,87.02136790,0.04500000,0,0,0,0,False,0.28430707
1,2026-07-10 00:00:00+00:00,5.00000000,659.00000000,-0.12270095,put_wing,0.40010994,86.02198415,0.04500000,86.02198415,0.04500000,0,0,0,0,False,0.28566013
2,2026-07-10 00:00:00+00:00,5.00000000,660.00000000,-0.12118465,put_wing,0.39565753,85.02260040,0.04500000,85.02260040,0.04500000,0,0,0,0,False,0.28703299
3,2026-07-10 00:00:00+00:00,5.00000000,661.00000000,-0.11967065,put_wing,0.39120764,84.02321665,0.04500000,84.02321665,0.04500000,0,0,0,0,False,0.28842617
4,2026-07-10 00:00:00+00:00,5.00000000,662.00000000,-0.11815893,put_wing,0.38676019,83.02383290,0.04500000,83.02383290,0.04500000,0,0,0,0,False,0.28984024


In [7]:
# ============================================================
# Butterfly convexity diagnostics by expiry
# ============================================================
# This cell tests the second cross-sectional static-arbitrage condition.
#
# For a fixed expiry, option value as a function of strike should be convex.
#
# For three ordered strikes K1 < K2 < K3, convexity requires:
#
#   V(K2) <= w1 * V(K1) + w3 * V(K3)
#
# where:
#
#   w1 = (K3 - K2) / (K3 - K1)
#   w3 = (K2 - K1) / (K3 - K1)
#
# If the middle price is materially above the chord joining the neighboring
# prices, the slice has a butterfly-convexity defect.
#
# Important discipline:
# Convexity is tested in price space, not IV space.
# A jagged implied-volatility smile is not automatically an arbitrage violation.
# The static-arbitrage condition is about option prices across strike.


surface_convexity = surface_monotonicity.copy()


# ------------------------------------------------------------
# Convexity lens configuration
# ------------------------------------------------------------

CONVEXITY_LENSES = [
    {
        "price_lens": "market_call_equiv",
        "price_col": "market_call_equiv_price",
        "description": "selected market price converted to call-equivalent value by parity",
    },
    {
        "price_lens": "bsm_call_equiv",
        "price_col": "bsm_call_equiv_price",
        "description": "BSM call price reconstructed from selected IV",
    },
    {
        "price_lens": "market_put_equiv",
        "price_col": "market_put_equiv_price",
        "description": "selected market price converted to put-equivalent value by parity",
    },
    {
        "price_lens": "bsm_put_equiv",
        "price_col": "bsm_put_equiv_price",
        "description": "BSM put price reconstructed from selected IV",
    },
]


# ------------------------------------------------------------
# Convexity tolerance and severity helpers
# ------------------------------------------------------------

def convexity_tolerance_triplet(
    price_left: float,
    price_mid: float,
    price_right: float,
    abs_tol: float = TOL.convexity_abs_tol,
    rel_tol: float = TOL.convexity_rel_tol,
) -> float:
    """
    Absolute-relative tolerance for three-point convexity checks.
    """
    scale = max(abs(price_left), abs(price_mid), abs(price_right), 1.0)
    return abs_tol + rel_tol * scale


def classify_convexity_severity(violation_amount: float) -> str:
    """
    Classify convexity violation severity using visible notebook thresholds.
    """
    if not np.isfinite(violation_amount) or violation_amount <= 0:
        return "clean"

    if violation_amount <= TOL.mild_violation_threshold:
        return "micro_noise"

    if violation_amount <= TOL.moderate_violation_threshold:
        return "mild_warning"

    if violation_amount <= TOL.severe_violation_threshold:
        return "moderate_warning"

    return "severe_violation"


def make_convexity_triplet_frame(
    df: pd.DataFrame,
    price_col: str,
    price_lens: str,
    description: str,
) -> pd.DataFrame:
    """
    Construct three-strike convexity diagnostics for one price lens.

    The diagnostic handles uneven strike spacing by comparing the middle price
    to the linear interpolation between neighboring strike prices.
    """
    rows: List[Dict[str, Any]] = []

    required_cols = [
        "n10_row_id",
        "expiry",
        "expiry_rank",
        "dte_calendar",
        "tau_years",
        "maturity_bucket",
        "strike",
        "spot",
        "forward",
        "log_moneyness",
        "abs_log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "total_variance",
        "diagnostic_weight",
        "price_spread",
        "relative_price_spread",
        "iv_uncertainty_width",
        "relative_iv_uncertainty_width",
        "surface_row_source",
        "primary_pair_issue",
        "primary_stability_issue",
        "quality_score",
        price_col,
    ]

    available_cols = [col for col in required_cols if col in df.columns]

    working = (
        df.loc[df["eligible_convexity_diagnostics"], available_cols]
        .copy()
        .sort_values(["expiry", "strike", "n10_row_id"])
        .reset_index(drop=True)
    )

    for expiry, group in working.groupby("expiry", sort=True):
        group = group.sort_values(["strike", "n10_row_id"]).reset_index(drop=True)

        if len(group) < 3:
            continue

        for i in range(1, len(group) - 1):
            left = group.iloc[i - 1]
            mid = group.iloc[i]
            right = group.iloc[i + 1]

            k_left = float(left["strike"])
            k_mid = float(mid["strike"])
            k_right = float(right["strike"])

            if not all(np.isfinite([k_left, k_mid, k_right])):
                continue

            if not (k_left < k_mid < k_right):
                continue

            p_left = float(left[price_col])
            p_mid = float(mid[price_col])
            p_right = float(right[price_col])

            if not all(np.isfinite([p_left, p_mid, p_right])):
                continue

            left_width = k_mid - k_left
            right_width = k_right - k_mid
            full_width = k_right - k_left

            w_left = (k_right - k_mid) / full_width
            w_right = (k_mid - k_left) / full_width

            chord_price_mid = w_left * p_left + w_right * p_right

            raw_chord_excess = p_mid - chord_price_mid
            tolerance = convexity_tolerance_triplet(p_left, p_mid, p_right)
            violation_amount = max(raw_chord_excess - tolerance, 0.0)

            is_violation = violation_amount > 0
            severity = classify_convexity_severity(violation_amount)

            left_slope = (p_mid - p_left) / left_width
            right_slope = (p_right - p_mid) / right_width
            slope_increase = right_slope - left_slope

            # For a convex function, slope_increase should be nonnegative.
            # This is a density-like local proxy, not a calibrated density estimate.
            density_proxy = 2.0 * slope_increase / full_width
            negative_density_proxy_amount = max(-density_proxy, 0.0)

            avg_price = np.nanmean([abs(p_left), abs(p_mid), abs(p_right)])
            avg_spread = np.nanmean(
                [
                    float(left.get("price_spread", np.nan)),
                    float(mid.get("price_spread", np.nan)),
                    float(right.get("price_spread", np.nan)),
                ]
            )
            avg_iv_uncertainty = np.nanmean(
                [
                    float(left.get("iv_uncertainty_width", np.nan)),
                    float(mid.get("iv_uncertainty_width", np.nan)),
                    float(right.get("iv_uncertainty_width", np.nan)),
                ]
            )
            avg_diagnostic_weight = np.nanmean(
                [
                    float(left["diagnostic_weight"]),
                    float(mid["diagnostic_weight"]),
                    float(right["diagnostic_weight"]),
                ]
            )

            normalized_violation_by_price = violation_amount / max(avg_price, TOL.eps)
            normalized_violation_by_spread = (
                violation_amount / max(avg_spread, TOL.eps)
                if np.isfinite(avg_spread)
                else np.nan
            )
            normalized_violation_by_iv_uncertainty = (
                violation_amount / max(avg_iv_uncertainty, TOL.eps)
                if np.isfinite(avg_iv_uncertainty)
                else np.nan
            )

            rows.append(
                {
                    "diagnostic_type": "butterfly_convexity",
                    "price_lens": price_lens,
                    "price_lens_description": description,
                    "expiry": expiry,
                    "expiry_rank": int(mid["expiry_rank"]),
                    "dte_calendar": float(mid["dte_calendar"]),
                    "tau_years": float(mid["tau_years"]),
                    "maturity_bucket": mid["maturity_bucket"],
                    "left_n10_row_id": int(left["n10_row_id"]),
                    "mid_n10_row_id": int(mid["n10_row_id"]),
                    "right_n10_row_id": int(right["n10_row_id"]),
                    "left_strike": k_left,
                    "mid_strike": k_mid,
                    "right_strike": k_right,
                    "left_width": left_width,
                    "right_width": right_width,
                    "full_width": full_width,
                    "left_log_moneyness": float(left["log_moneyness"]),
                    "mid_log_moneyness": float(mid["log_moneyness"]),
                    "right_log_moneyness": float(right["log_moneyness"]),
                    "mid_moneyness_bucket": mid["moneyness_bucket_n10"],
                    "left_price": p_left,
                    "mid_price": p_mid,
                    "right_price": p_right,
                    "chord_price_at_mid_strike": chord_price_mid,
                    "raw_chord_excess_before_tolerance": raw_chord_excess,
                    "tolerance": tolerance,
                    "violation_amount": violation_amount,
                    "is_violation": bool(is_violation),
                    "severity": severity,
                    "left_slope": left_slope,
                    "right_slope": right_slope,
                    "slope_increase": slope_increase,
                    "density_proxy": density_proxy,
                    "negative_density_proxy_amount": negative_density_proxy_amount,
                    "normalized_violation_by_price": normalized_violation_by_price,
                    "normalized_violation_by_spread": normalized_violation_by_spread,
                    "normalized_violation_by_iv_uncertainty": normalized_violation_by_iv_uncertainty,
                    "left_selected_iv": float(left["selected_iv"]),
                    "mid_selected_iv": float(mid["selected_iv"]),
                    "right_selected_iv": float(right["selected_iv"]),
                    "left_total_variance": float(left["total_variance"]),
                    "mid_total_variance": float(mid["total_variance"]),
                    "right_total_variance": float(right["total_variance"]),
                    "left_diagnostic_weight": float(left["diagnostic_weight"]),
                    "mid_diagnostic_weight": float(mid["diagnostic_weight"]),
                    "right_diagnostic_weight": float(right["diagnostic_weight"]),
                    "avg_diagnostic_weight": avg_diagnostic_weight,
                    "left_price_spread": float(left.get("price_spread", np.nan)),
                    "mid_price_spread": float(mid.get("price_spread", np.nan)),
                    "right_price_spread": float(right.get("price_spread", np.nan)),
                    "avg_price_spread": avg_spread,
                    "left_relative_price_spread": float(left.get("relative_price_spread", np.nan)),
                    "mid_relative_price_spread": float(mid.get("relative_price_spread", np.nan)),
                    "right_relative_price_spread": float(right.get("relative_price_spread", np.nan)),
                    "left_iv_uncertainty_width": float(left.get("iv_uncertainty_width", np.nan)),
                    "mid_iv_uncertainty_width": float(mid.get("iv_uncertainty_width", np.nan)),
                    "right_iv_uncertainty_width": float(right.get("iv_uncertainty_width", np.nan)),
                    "avg_iv_uncertainty_width": avg_iv_uncertainty,
                    "left_surface_row_source": left.get("surface_row_source", None),
                    "mid_surface_row_source": mid.get("surface_row_source", None),
                    "right_surface_row_source": right.get("surface_row_source", None),
                    "left_primary_pair_issue": left.get("primary_pair_issue", None),
                    "mid_primary_pair_issue": mid.get("primary_pair_issue", None),
                    "right_primary_pair_issue": right.get("primary_pair_issue", None),
                    "left_primary_stability_issue": left.get("primary_stability_issue", None),
                    "mid_primary_stability_issue": mid.get("primary_stability_issue", None),
                    "right_primary_stability_issue": right.get("primary_stability_issue", None),
                    "left_quality_score": float(left.get("quality_score", np.nan)),
                    "mid_quality_score": float(mid.get("quality_score", np.nan)),
                    "right_quality_score": float(right.get("quality_score", np.nan)),
                }
            )

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Run three-strike convexity diagnostics
# ------------------------------------------------------------

convexity_triplet_frames: List[pd.DataFrame] = []

for lens in CONVEXITY_LENSES:
    lens_triplets = make_convexity_triplet_frame(
        df=surface_convexity,
        price_col=lens["price_col"],
        price_lens=lens["price_lens"],
        description=lens["description"],
    )
    convexity_triplet_frames.append(lens_triplets)

convexity_triplet_diagnostics = pd.concat(
    convexity_triplet_frames,
    axis=0,
    ignore_index=True,
)

convexity_violation_ledger = (
    convexity_triplet_diagnostics
    .loc[convexity_triplet_diagnostics["is_violation"]]
    .sort_values(
        [
            "violation_amount",
            "normalized_violation_by_price",
            "expiry_rank",
            "mid_strike",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Attach row-level convexity exposure flags back to surface
# ------------------------------------------------------------

for lens in CONVEXITY_LENSES:
    price_lens = lens["price_lens"]

    surface_convexity[f"{price_lens}_convexity_triplet_count"] = 0
    surface_convexity[f"{price_lens}_convexity_violation_triplet_count"] = 0
    surface_convexity[f"{price_lens}_max_convexity_violation_amount"] = 0.0
    surface_convexity[f"{price_lens}_max_negative_density_proxy_amount"] = 0.0

for lens in CONVEXITY_LENSES:
    price_lens = lens["price_lens"]

    lens_triplets = convexity_triplet_diagnostics.loc[
        convexity_triplet_diagnostics["price_lens"].eq(price_lens)
    ]

    if lens_triplets.empty:
        continue

    endpoint_records = []

    for side in ["left", "mid", "right"]:
        temp = lens_triplets[
            [
                f"{side}_n10_row_id",
                "is_violation",
                "violation_amount",
                "negative_density_proxy_amount",
            ]
        ].rename(columns={f"{side}_n10_row_id": "n10_row_id"})

        endpoint_records.append(temp)

    endpoint_df = pd.concat(endpoint_records, axis=0, ignore_index=True)

    row_summary = (
        endpoint_df
        .groupby("n10_row_id")
        .agg(
            triplet_count=("is_violation", "size"),
            violation_triplet_count=("is_violation", "sum"),
            max_violation_amount=("violation_amount", "max"),
            max_negative_density_proxy_amount=("negative_density_proxy_amount", "max"),
        )
        .reset_index()
    )

    surface_convexity = surface_convexity.merge(
        row_summary,
        on="n10_row_id",
        how="left",
    )

    surface_convexity[f"{price_lens}_convexity_triplet_count"] = (
        surface_convexity["triplet_count"].fillna(0).astype(int)
    )
    surface_convexity[f"{price_lens}_convexity_violation_triplet_count"] = (
        surface_convexity["violation_triplet_count"].fillna(0).astype(int)
    )
    surface_convexity[f"{price_lens}_max_convexity_violation_amount"] = (
        surface_convexity["max_violation_amount"].fillna(0.0)
    )
    surface_convexity[f"{price_lens}_max_negative_density_proxy_amount"] = (
        surface_convexity["max_negative_density_proxy_amount"].fillna(0.0)
    )

    surface_convexity = surface_convexity.drop(
        columns=[
            "triplet_count",
            "violation_triplet_count",
            "max_violation_amount",
            "max_negative_density_proxy_amount",
        ],
        errors="ignore",
    )

surface_convexity["any_convexity_violation_touching_row"] = False

for lens in CONVEXITY_LENSES:
    price_lens = lens["price_lens"]
    surface_convexity["any_convexity_violation_touching_row"] = (
        surface_convexity["any_convexity_violation_touching_row"]
        | (surface_convexity[f"{price_lens}_convexity_violation_triplet_count"] > 0)
    )


# ------------------------------------------------------------
# Convexity summaries
# ------------------------------------------------------------

convexity_lens_summary = (
    convexity_triplet_diagnostics
    .groupby(["price_lens"], dropna=False)
    .agg(
        tested_triplets=("is_violation", "size"),
        violation_triplets=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_raw_chord_excess=("raw_chord_excess_before_tolerance", "max"),
        min_density_proxy=("density_proxy", "min"),
        max_negative_density_proxy_amount=("negative_density_proxy_amount", "max"),
        max_normalized_violation_by_price=("normalized_violation_by_price", "max"),
        max_normalized_violation_by_spread=("normalized_violation_by_spread", "max"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
)

convexity_lens_summary["violation_rate"] = (
    convexity_lens_summary["violation_triplets"]
    / convexity_lens_summary["tested_triplets"].clip(lower=1)
)

convexity_expiry_summary = (
    convexity_triplet_diagnostics
    .groupby(["price_lens", "expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        tested_triplets=("is_violation", "size"),
        violation_triplets=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_raw_chord_excess=("raw_chord_excess_before_tolerance", "max"),
        min_density_proxy=("density_proxy", "min"),
        max_negative_density_proxy_amount=("negative_density_proxy_amount", "max"),
        max_normalized_violation_by_price=("normalized_violation_by_price", "max"),
        max_normalized_violation_by_spread=("normalized_violation_by_spread", "max"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["price_lens", "expiry_rank"])
)

convexity_expiry_summary["violation_rate"] = (
    convexity_expiry_summary["violation_triplets"]
    / convexity_expiry_summary["tested_triplets"].clip(lower=1)
)

convexity_bucket_summary = (
    convexity_triplet_diagnostics
    .groupby(["price_lens", "maturity_bucket", "mid_moneyness_bucket"], dropna=False)
    .agg(
        tested_triplets=("is_violation", "size"),
        violation_triplets=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        min_density_proxy=("density_proxy", "min"),
        max_negative_density_proxy_amount=("negative_density_proxy_amount", "max"),
        max_normalized_violation_by_price=("normalized_violation_by_price", "max"),
        max_normalized_violation_by_spread=("normalized_violation_by_spread", "max"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["price_lens", "maturity_bucket", "mid_moneyness_bucket"])
)

convexity_bucket_summary["violation_rate"] = (
    convexity_bucket_summary["violation_triplets"]
    / convexity_bucket_summary["tested_triplets"].clip(lower=1)
)

convexity_severity_summary = (
    convexity_triplet_diagnostics
    .groupby(["price_lens", "severity"], dropna=False)
    .agg(
        triplet_count=("is_violation", "size"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        min_density_proxy=("density_proxy", "min"),
        max_negative_density_proxy_amount=("negative_density_proxy_amount", "max"),
    )
    .reset_index()
)

convexity_severity_summary["severity"] = pd.Categorical(
    convexity_severity_summary["severity"],
    categories=SEVERITY_ORDER,
    ordered=True,
)

convexity_severity_summary = convexity_severity_summary.sort_values(
    ["price_lens", "severity"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Validation ledger
# ------------------------------------------------------------

market_call_convexity_violations = int(
    convexity_lens_summary
    .loc[convexity_lens_summary["price_lens"].eq("market_call_equiv"), "violation_triplets"]
    .sum()
)

market_put_convexity_violations = int(
    convexity_lens_summary
    .loc[convexity_lens_summary["price_lens"].eq("market_put_equiv"), "violation_triplets"]
    .sum()
)

bsm_call_convexity_violations = int(
    convexity_lens_summary
    .loc[convexity_lens_summary["price_lens"].eq("bsm_call_equiv"), "violation_triplets"]
    .sum()
)

bsm_put_convexity_violations = int(
    convexity_lens_summary
    .loc[convexity_lens_summary["price_lens"].eq("bsm_put_equiv"), "violation_triplets"]
    .sum()
)

expected_triplets_per_lens = int(
    surface_convexity
    .groupby("expiry", dropna=False)
    .size()
    .sub(2)
    .clip(lower=0)
    .sum()
)

convexity_validation_rows = [
    validation_row(
        "convexity_diagnostics_computed",
        len(convexity_triplet_diagnostics) > 0,
        f"{len(convexity_triplet_diagnostics):,} three-strike triplets tested across {len(CONVEXITY_LENSES):,} price lenses",
    ),
    validation_row(
        "expected_triplet_count_per_lens_matched",
        bool((convexity_lens_summary["tested_triplets"] == expected_triplets_per_lens).all()),
        f"expected {expected_triplets_per_lens:,} three-strike triplets per lens",
    ),
    validation_row(
        "market_call_equiv_convexity_clean",
        market_call_convexity_violations == 0,
        f"{market_call_convexity_violations:,} market call-equivalent convexity violations",
    ),
    validation_row(
        "market_put_equiv_convexity_clean",
        market_put_convexity_violations == 0,
        f"{market_put_convexity_violations:,} market put-equivalent convexity violations",
    ),
    validation_row(
        "bsm_call_equiv_convexity_clean",
        bsm_call_convexity_violations == 0,
        f"{bsm_call_convexity_violations:,} BSM call-equivalent convexity violations",
    ),
    validation_row(
        "bsm_put_equiv_convexity_clean",
        bsm_put_convexity_violations == 0,
        f"{bsm_put_convexity_violations:,} BSM put-equivalent convexity violations",
    ),
    validation_row(
        "row_level_convexity_flags_attached",
        "any_convexity_violation_touching_row" in surface_convexity.columns,
        "row-level triplet flags attached to surface_convexity",
    ),
]

convexity_validation_ledger = pd.DataFrame(convexity_validation_rows)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("Butterfly convexity validation ledger")
print("=" * 90)
display(convexity_validation_ledger)

print("=" * 90)
print("Butterfly convexity summary by price lens")
print("=" * 90)
display(convexity_lens_summary)

print("=" * 90)
print("Butterfly convexity summary by expiry")
print("=" * 90)
display(convexity_expiry_summary)

print("=" * 90)
print("Butterfly convexity summary by maturity / moneyness bucket")
print("=" * 90)
display(convexity_bucket_summary)

print("=" * 90)
print("Butterfly convexity severity summary")
print("=" * 90)
display(convexity_severity_summary)

print("=" * 90)
print("Butterfly convexity violation ledger")
print("=" * 90)

if len(convexity_violation_ledger) == 0:
    print("No butterfly convexity violations detected under configured tolerances.")
else:
    display(convexity_violation_ledger.head(75))


RUN_METADATA["convexity_validation"] = convexity_validation_ledger.to_dict(orient="records")
RUN_METADATA["convexity_lens_summary"] = convexity_lens_summary.to_dict(orient="records")

surface_convexity[
    [
        "expiry",
        "dte_calendar",
        "strike",
        "log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "market_call_equiv_price",
        "market_put_equiv_price",
        "bsm_call_equiv_price",
        "bsm_put_equiv_price",
        "market_call_equiv_convexity_violation_triplet_count",
        "market_put_equiv_convexity_violation_triplet_count",
        "bsm_call_equiv_convexity_violation_triplet_count",
        "bsm_put_equiv_convexity_violation_triplet_count",
        "any_convexity_violation_touching_row",
        "diagnostic_weight",
    ]
].head()

Butterfly convexity validation ledger


,check,passed,details
0,convexity_diagnostics_computed,True,"5,268 three-strike triplets tested across 4 pr..."
1,expected_triplet_count_per_lens_matched,True,"expected 1,317 three-strike triplets per lens"
2,market_call_equiv_convexity_clean,False,22 market call-equivalent convexity violations
3,market_put_equiv_convexity_clean,False,141 market put-equivalent convexity violations
4,bsm_call_equiv_convexity_clean,False,22 BSM call-equivalent convexity violations
5,bsm_put_equiv_convexity_clean,False,141 BSM put-equivalent convexity violations
6,row_level_convexity_flags_attached,True,row-level triplet flags attached to surface_co...


Butterfly convexity summary by price lens


,price_lens,tested_triplets,violation_triplets,max_violation_amount,mean_violation_amount,median_violation_amount,max_raw_chord_excess,min_density_proxy,max_negative_density_proxy_amount,max_normalized_violation_by_price,max_normalized_violation_by_spread,median_avg_diagnostic_weight,violation_rate
0,bsm_call_equiv,1317,22,0.01611432,0.00009938,0.00000000,0.01704341,-0.03408682,0.03408682,0.09481935,0.48990000,0.98712225,0.01670463
1,bsm_put_equiv,1317,141,0.01567564,0.00038649,0.00000000,0.01704341,-0.03408682,0.03408682,0.09481935,0.48990000,0.98712225,0.10706150
2,market_call_equiv,1317,22,0.01611432,0.00009938,0.00000000,0.01704341,-0.03408682,0.03408682,0.09481935,0.48990000,0.98712225,0.01670463
3,market_put_equiv,1317,141,0.01567564,0.00038649,0.00000000,0.01704341,-0.03408682,0.03408682,0.09481935,0.48990000,0.98712225,0.10706150


Butterfly convexity summary by expiry


...

Butterfly convexity severity summary


,price_lens,severity,triplet_count,max_violation_amount,mean_violation_amount,min_density_proxy,max_negative_density_proxy_amount
0,bsm_call_equiv,clean,1295,0.00000000,0.00000000,-0.01000000,0.01000000
1,bsm_call_equiv,mild_warning,1,0.00008852,0.00008852,-0.01000000,0.01000000
2,bsm_call_equiv,moderate_warning,2,0.00097675,0.00068154,-0.01000000,0.01000000
3,bsm_call_equiv,severe_violation,19,0.01611432,0.00681243,-0.03408682,0.03408682
4,bsm_put_equiv,clean,1176,0.00000000,0.00000000,-0.01000000,0.01000000
5,bsm_put_equiv,moderate_warning,4,0.00073233,0.00050566,-0.01000000,0.01000000
6,bsm_put_equiv,severe_violation,137,0.01567564,0.00370057,-0.03408682,0.03408682
7,market_call_equiv,clean,1295,0.00000000,0.00000000,-0.01000000,0.01000000
8,market_call_equiv,mild_warning,1,0.00008852,0.00008852,-0.01000000,0.01000000
9,market_call_equiv,moderate_warning,2,0.00097675,0.00068154,-0.01000000,0.01000000


Butterfly convexity violation ledger


,diagnostic_type,price_lens,price_lens_description,expiry,expiry_rank,dte_calendar,tau_years,maturity_bucket,left_n10_row_id,mid_n10_row_id,right_n10_row_id,left_strike,mid_strike,right_strike,left_width,right_width,full_width,left_log_moneyness,mid_log_moneyness,right_log_moneyness,mid_moneyness_bucket,left_price,mid_price,right_price,chord_price_at_mid_strike,raw_chord_excess_before_tolerance,tolerance,violation_amount,is_violation,severity,left_slope,right_slope,slope_increase,density_proxy,negative_density_proxy_amount,normalized_violation_by_price,normalized_violation_by_spread,normalized_violation_by_iv_uncertainty,left_selected_iv,mid_selected_iv,right_selected_iv,left_total_variance,mid_total_variance,right_total_variance,left_diagnostic_weight,mid_diagnostic_weight,right_diagnostic_weight,avg_diagnostic_weight,left_price_spread,mid_price_spread,right_price_spread,avg_price_spread,left_relative_price_spread,mid_relative_price_spread,right_relative_price_spread,left_iv_uncertainty_width,mid_iv_uncertainty_width,right_iv_uncertainty_width,avg_iv_uncertainty_width,left_surface_row_source,mid_surface_row_source,right_surface_row_source,left_primary_pair_issue,mid_primary_pair_issue,right_primary_pair_issue,left_primary_stability_issue,mid_primary_stability_issue,right_primary_stability_issue,left_quality_score,mid_quality_score,right_quality_score
0,butterfly_convexity,market_call_equiv,selected market price converted to call-equiva...,2026-07-31 00:00:00+00:00,3,26.00000000,0.07123288,front_intermediate,579,580,581,750.00000000,751.00000000,752.00000000,1.00000000,1.00000000,2.00000000,0.00455703,0.00588948,0.00722015,atm,9.28091318,8.79500000,8.27500000,8.77795659,0.01704341,0.00092909,0.01611432,True,severe_violation,-0.48591318,-0.52000000,-0.03408682,-0.03408682,0.03408682,0.00183458,0.30214343,23.69139538,0.13720453,0.13655125,0.13527030,0.00134096,0.00132823,0.00130342,1.00000000,1.00000000,1.00000000,1.00000000,0.06000000,0.05000000,0.05000000,0.05333333,0.00473186,0.00568505,0.00604230,0.00076148,0.00063752,0.00064153,0.00068018,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,pair_clean,stability_clean,stability_clean,stability_clean,0.99453667,0.99371716,0.99336055
1,butterfly_convexity,bsm_call_equiv,BSM call price reconstructed from selected IV,2026-07-31 00:00:00+00:00,3,26.00000000,0.07123288,front_intermediate,579,580,581,750.00000000,751.00000000,752.00000000,1.00000000,1.00000000,2.00000000,0.00455703,0.00588948,0.00722015,atm,9.28091318,8.79500000,8.27500000,8.77795659,0.01704341,0.00092909,0.01611432,True,severe_violation,-0.48591318,-0.52000000,-0.03408682,-0.03408682,0.03408682,0.00183458,0.30214343,23.69139538,0.13720453,0.13655125,0.13527030,0.00134096,0.00132823,0.00130342,1.00000000,1.00000000,1.00000000,1.00000000,0.06000000,0.05000000,0.05000000,0.05333333,0.00473186,0.00568505,0.00604230,0.00076148,0.00063752,0.00064153,0.00068018,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,pair_clean,stability_clean,stability_clean,stability_clean,0.99453667,0.99371716,0.99336055
2,butterfly_convexity,market_put_equiv,selected market price converted to put-equival...,2026-07-31 00:00:00+00:00,3,26.00000000,0.07123288,front_intermediate,579,580,581,750.00000000,751.00000000,752.00000000,1.00000000,1.00000000,2.00000000,0.00455703,0.00588948,0.00722015,atm,12.68000000,13.19088647,13.66768612,13.17384306,0.01704341,0.00136777,0.01567564,True,severe_violation,0.51088647,0.47679965,-0.03408682,-0.03408682,0.03408682,0.00118939,0.29391823,23.04644855,0.13720453,0.13655125,0.13527030,0.00134096,0.00132823,0.00130342,1.00000000,1.00000000,1.00000000,1.00000000,0.06000000,0.05000000,0.05000000,0.05333333,0.00473186,0.00568505,0.00604230,0.00076148,0.00063752,0.00064153,0.00068018,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,pair_clean,stabili

,expiry,dte_calendar,strike,log_moneyness,moneyness_bucket_n10,selected_iv,market_call_equiv_price,market_put_equiv_price,bsm_call_equiv_price,bsm_put_equiv_price,market_call_equiv_convexity_violation_triplet_count,market_put_equiv_convexity_violation_triplet_count,bsm_call_equiv_convexity_violation_triplet_count,bsm_put_equiv_convexity_violation_triplet_count,any_convexity_violation_touching_row,diagnostic_weight
0,2026-07-10 00:00:00+00:00,5.00000000,658.00000000,-0.12421955,put_wing,0.40456494,87.02136790,0.04500000,87.02136790,0.04500000,0,0,0,0,False,0.28430707
1,2026-07-10 00:00:00+00:00,5.00000000,659.00000000,-0.12270095,put_wing,0.40010994,86.02198415,0.04500000,86.02198415,0.04500000,0,0,0,0,False,0.28566013
2,2026-07-10 00:00:00+00:00,5.00000000,660.00000000,-0.12118465,put_wing,0.39565753,85.02260040,0.04500000,85.02260040,0.04500000,0,0,0,0,False,0.28703299
3,2026-07-10 00:00:00+00:00,5.00000000,661.00000000,-0.11967065,put_wing,0.39120764,84.02321665,0.04500000,84.02321665,0.04500000,0,0,0,0,False,0.28842617
4,2026-07-10 00:00:00+00:00,5.00000000,662.00000000,-0.11815893,put_wing,0.38676019,83.02383290,0.04500000,83.02383290,0.04500000,0,0,0,0,False,0.28984024


In [8]:
# ============================================================
# Calendar / total-variance diagnostics across expiry
# ============================================================
# This cell tests maturity consistency in the raw empirical surface.
#
# The main diagnostic is:
#
#   At comparable log-moneyness, total variance should not materially decrease
#   as maturity increases.
#
# This is not a full mathematical proof of absence of calendar arbitrage.
# It is a practical raw-surface diagnostic used before SVI/SSVI smoothing.
#
# Why total variance rather than IV alone?
#
#   IV may decline with maturity without creating a calendar problem.
#   Total variance, w(k, T) = sigma_imp(k, T)^2 * T, is the cleaner object
#   for maturity comparisons.
#
# This cell uses two comparison designs:
#
# 1. Interpolated adjacent-expiry grid:
#    Compare total variance between adjacent expiries on their common
#    log-moneyness overlap.
#
# 2. Nearest-neighbor observation pairs:
#    Match real observations across adjacent expiries within a small
#    log-moneyness tolerance.
#
# The interpolated grid gives structured coverage.
# The nearest-neighbor ledger keeps the diagnostic anchored to actual rows.


surface_calendar = surface_convexity.copy()


# ------------------------------------------------------------
# Calendar diagnostic configuration
# ------------------------------------------------------------

CALENDAR_MIN_POINTS_PER_SLICE = TOL.min_points_per_expiry
CALENDAR_GRID_POINTS_PER_PAIR = 101
CALENDAR_MIN_OVERLAP_WIDTH = 0.01

CALENDAR_LENSES = [
    {
        "calendar_lens": "total_variance",
        "value_col": "total_variance",
        "expected_direction": "non_decreasing",
        "description": "selected IV total variance, w = implied_volatility^2 * tau",
    }
]


# ------------------------------------------------------------
# Calendar tolerance and severity helpers
# ------------------------------------------------------------

def calendar_tolerance_pair(
    short_value: float,
    long_value: float,
    abs_tol: float = TOL.total_variance_abs_tol,
    rel_tol: float = TOL.total_variance_rel_tol,
) -> float:
    """
    Absolute-relative tolerance for total-variance calendar comparisons.
    """
    scale = max(abs(short_value), abs(long_value), 1.0)
    return abs_tol + rel_tol * scale


def classify_calendar_severity(violation_amount: float) -> str:
    """
    Classify calendar violation severity using visible notebook thresholds.
    """
    if not np.isfinite(violation_amount) or violation_amount <= 0:
        return "clean"

    if violation_amount <= TOL.mild_violation_threshold:
        return "micro_noise"

    if violation_amount <= TOL.moderate_violation_threshold:
        return "mild_warning"

    if violation_amount <= TOL.severe_violation_threshold:
        return "moderate_warning"

    return "severe_violation"


def safe_interp_1d(x_new: np.ndarray, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Linear interpolation after dropping invalid values and duplicate x values.

    Duplicate log-moneyness coordinates are averaged before interpolation.
    """
    temp = pd.DataFrame({"x": x, "y": y})
    temp = temp.replace([np.inf, -np.inf], np.nan).dropna()

    if temp.empty:
        return np.full_like(x_new, np.nan, dtype=float)

    temp = (
        temp.groupby("x", as_index=False)["y"]
        .mean()
        .sort_values("x")
        .reset_index(drop=True)
    )

    if len(temp) < 2:
        return np.full_like(x_new, np.nan, dtype=float)

    return np.interp(x_new, temp["x"].to_numpy(), temp["y"].to_numpy())


def build_calendar_expiry_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build adjacent expiry-pair metadata using eligible surface rows.
    """
    expiry_meta = (
        df.loc[df["eligible_pointwise_diagnostics"]]
        .groupby(["expiry", "expiry_rank"], dropna=False)
        .agg(
            dte_calendar=("dte_calendar", "median"),
            tau_years=("tau_years", "median"),
            n_obs=("n10_row_id", "size"),
            min_log_moneyness=("log_moneyness", "min"),
            max_log_moneyness=("log_moneyness", "max"),
            min_strike=("strike", "min"),
            max_strike=("strike", "max"),
            median_total_variance=("total_variance", "median"),
            min_total_variance=("total_variance", "min"),
            max_total_variance=("total_variance", "max"),
            median_diagnostic_weight=("diagnostic_weight", "median"),
        )
        .reset_index()
        .sort_values(["tau_years", "expiry"])
        .reset_index(drop=True)
    )

    rows: List[Dict[str, Any]] = []

    for i in range(len(expiry_meta) - 1):
        short = expiry_meta.iloc[i]
        long = expiry_meta.iloc[i + 1]

        overlap_left = max(float(short["min_log_moneyness"]), float(long["min_log_moneyness"]))
        overlap_right = min(float(short["max_log_moneyness"]), float(long["max_log_moneyness"]))
        overlap_width = overlap_right - overlap_left

        rows.append(
            {
                "expiry_pair_id": i,
                "short_expiry": short["expiry"],
                "long_expiry": long["expiry"],
                "short_expiry_rank": int(short["expiry_rank"]),
                "long_expiry_rank": int(long["expiry_rank"]),
                "short_dte_calendar": float(short["dte_calendar"]),
                "long_dte_calendar": float(long["dte_calendar"]),
                "short_tau_years": float(short["tau_years"]),
                "long_tau_years": float(long["tau_years"]),
                "tau_gap_years": float(long["tau_years"] - short["tau_years"]),
                "short_n_obs": int(short["n_obs"]),
                "long_n_obs": int(long["n_obs"]),
                "short_min_log_moneyness": float(short["min_log_moneyness"]),
                "short_max_log_moneyness": float(short["max_log_moneyness"]),
                "long_min_log_moneyness": float(long["min_log_moneyness"]),
                "long_max_log_moneyness": float(long["max_log_moneyness"]),
                "overlap_left_log_moneyness": overlap_left,
                "overlap_right_log_moneyness": overlap_right,
                "overlap_width": overlap_width,
                "has_sufficient_overlap": bool(overlap_width >= CALENDAR_MIN_OVERLAP_WIDTH),
                "short_median_total_variance": float(short["median_total_variance"]),
                "long_median_total_variance": float(long["median_total_variance"]),
                "short_median_diagnostic_weight": float(short["median_diagnostic_weight"]),
                "long_median_diagnostic_weight": float(long["median_diagnostic_weight"]),
            }
        )

    return pd.DataFrame(rows)


expiry_pair_inventory = build_calendar_expiry_pairs(surface_calendar)


# ------------------------------------------------------------
# Interpolated adjacent-expiry total-variance grid diagnostics
# ------------------------------------------------------------

def make_calendar_grid_diagnostics(df: pd.DataFrame, expiry_pairs: pd.DataFrame) -> pd.DataFrame:
    """
    Compare adjacent expiries on a common interpolated log-moneyness grid.
    """
    rows: List[Dict[str, Any]] = []

    for _, pair in expiry_pairs.iterrows():
        if not bool(pair["has_sufficient_overlap"]):
            continue

        short_expiry = pair["short_expiry"]
        long_expiry = pair["long_expiry"]

        short_slice = (
            df.loc[df["expiry"].eq(short_expiry) & df["eligible_pointwise_diagnostics"]]
            .sort_values("log_moneyness")
            .copy()
        )

        long_slice = (
            df.loc[df["expiry"].eq(long_expiry) & df["eligible_pointwise_diagnostics"]]
            .sort_values("log_moneyness")
            .copy()
        )

        if len(short_slice) < CALENDAR_MIN_POINTS_PER_SLICE or len(long_slice) < CALENDAR_MIN_POINTS_PER_SLICE:
            continue

        grid = np.linspace(
            float(pair["overlap_left_log_moneyness"]),
            float(pair["overlap_right_log_moneyness"]),
            CALENDAR_GRID_POINTS_PER_PAIR,
        )

        short_w = safe_interp_1d(
            x_new=grid,
            x=short_slice["log_moneyness"].to_numpy(dtype=float),
            y=short_slice["total_variance"].to_numpy(dtype=float),
        )

        long_w = safe_interp_1d(
            x_new=grid,
            x=long_slice["log_moneyness"].to_numpy(dtype=float),
            y=long_slice["total_variance"].to_numpy(dtype=float),
        )

        short_iv = safe_interp_1d(
            x_new=grid,
            x=short_slice["log_moneyness"].to_numpy(dtype=float),
            y=short_slice["selected_iv"].to_numpy(dtype=float),
        )

        long_iv = safe_interp_1d(
            x_new=grid,
            x=long_slice["log_moneyness"].to_numpy(dtype=float),
            y=long_slice["selected_iv"].to_numpy(dtype=float),
        )

        short_weight = safe_interp_1d(
            x_new=grid,
            x=short_slice["log_moneyness"].to_numpy(dtype=float),
            y=short_slice["diagnostic_weight"].to_numpy(dtype=float),
        )

        long_weight = safe_interp_1d(
            x_new=grid,
            x=long_slice["log_moneyness"].to_numpy(dtype=float),
            y=long_slice["diagnostic_weight"].to_numpy(dtype=float),
        )

        for k, sw, lw, siv, liv, swgt, lwgt in zip(grid, short_w, long_w, short_iv, long_iv, short_weight, long_weight):
            if not all(np.isfinite([k, sw, lw])):
                continue

            tolerance = calendar_tolerance_pair(sw, lw)
            raw_decrease = sw - lw
            violation_amount = max(raw_decrease - tolerance, 0.0)
            is_violation = violation_amount > 0
            severity = classify_calendar_severity(violation_amount)

            avg_w = 0.5 * (abs(sw) + abs(lw))
            avg_weight = np.nanmean([swgt, lwgt])

            rows.append(
                {
                    "diagnostic_type": "calendar_total_variance_grid",
                    "calendar_lens": "total_variance",
                    "expiry_pair_id": int(pair["expiry_pair_id"]),
                    "short_expiry": short_expiry,
                    "long_expiry": long_expiry,
                    "short_expiry_rank": int(pair["short_expiry_rank"]),
                    "long_expiry_rank": int(pair["long_expiry_rank"]),
                    "short_dte_calendar": float(pair["short_dte_calendar"]),
                    "long_dte_calendar": float(pair["long_dte_calendar"]),
                    "short_tau_years": float(pair["short_tau_years"]),
                    "long_tau_years": float(pair["long_tau_years"]),
                    "tau_gap_years": float(pair["tau_gap_years"]),
                    "log_moneyness": float(k),
                    "abs_log_moneyness": abs(float(k)),
                    "moneyness_bucket_n10": assign_moneyness_bucket(float(k)),
                    "short_total_variance_interp": float(sw),
                    "long_total_variance_interp": float(lw),
                    "total_variance_change_long_minus_short": float(lw - sw),
                    "raw_total_variance_decrease_before_tolerance": float(raw_decrease),
                    "tolerance": float(tolerance),
                    "violation_amount": float(violation_amount),
                    "is_violation": bool(is_violation),
                    "severity": severity,
                    "normalized_violation_by_total_variance": float(violation_amount / max(avg_w, TOL.eps)),
                    "short_iv_interp": float(siv) if np.isfinite(siv) else np.nan,
                    "long_iv_interp": float(liv) if np.isfinite(liv) else np.nan,
                    "iv_change_long_minus_short": float(liv - siv) if np.isfinite(siv) and np.isfinite(liv) else np.nan,
                    "short_diagnostic_weight_interp": float(swgt) if np.isfinite(swgt) else np.nan,
                    "long_diagnostic_weight_interp": float(lwgt) if np.isfinite(lwgt) else np.nan,
                    "avg_diagnostic_weight": float(avg_weight) if np.isfinite(avg_weight) else np.nan,
                    "overlap_left_log_moneyness": float(pair["overlap_left_log_moneyness"]),
                    "overlap_right_log_moneyness": float(pair["overlap_right_log_moneyness"]),
                    "overlap_width": float(pair["overlap_width"]),
                }
            )

    return pd.DataFrame(rows)


calendar_grid_diagnostics = make_calendar_grid_diagnostics(
    df=surface_calendar,
    expiry_pairs=expiry_pair_inventory,
)

calendar_grid_violation_ledger = (
    calendar_grid_diagnostics
    .loc[calendar_grid_diagnostics["is_violation"]]
    .sort_values(
        [
            "violation_amount",
            "normalized_violation_by_total_variance",
            "short_expiry_rank",
            "log_moneyness",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Nearest-neighbor adjacent-expiry diagnostics anchored to real rows
# ------------------------------------------------------------

def make_calendar_nearest_neighbor_diagnostics(df: pd.DataFrame, expiry_pairs: pd.DataFrame) -> pd.DataFrame:
    """
    Match real observations across adjacent expiries by nearest log-moneyness.
    """
    rows: List[Dict[str, Any]] = []

    base_cols = [
        "n10_row_id",
        "expiry",
        "expiry_rank",
        "dte_calendar",
        "tau_years",
        "strike",
        "forward",
        "log_moneyness",
        "abs_log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "total_variance",
        "diagnostic_weight",
        "price_spread",
        "relative_price_spread",
        "iv_uncertainty_width",
        "relative_iv_uncertainty_width",
        "surface_row_source",
        "primary_pair_issue",
        "primary_stability_issue",
        "quality_score",
        "any_monotonicity_violation_touching_row",
        "any_convexity_violation_touching_row",
    ]

    available_cols = [col for col in base_cols if col in df.columns]

    for _, pair in expiry_pairs.iterrows():
        if not bool(pair["has_sufficient_overlap"]):
            continue

        short_slice = (
            df.loc[df["expiry"].eq(pair["short_expiry"]) & df["eligible_pointwise_diagnostics"], available_cols]
            .sort_values("log_moneyness")
            .reset_index(drop=True)
        )

        long_slice = (
            df.loc[df["expiry"].eq(pair["long_expiry"]) & df["eligible_pointwise_diagnostics"], available_cols]
            .sort_values("log_moneyness")
            .reset_index(drop=True)
        )

        if short_slice.empty or long_slice.empty:
            continue

        long_k = long_slice["log_moneyness"].to_numpy(dtype=float)

        for _, short_row in short_slice.iterrows():
            k_short = float(short_row["log_moneyness"])

            if not np.isfinite(k_short):
                continue

            nearest_idx = int(np.nanargmin(np.abs(long_k - k_short)))
            long_row = long_slice.iloc[nearest_idx]

            k_long = float(long_row["log_moneyness"])
            k_distance = abs(k_long - k_short)

            if k_distance > TOL.moneyness_match_tol:
                continue

            short_w = float(short_row["total_variance"])
            long_w = float(long_row["total_variance"])

            if not all(np.isfinite([short_w, long_w])):
                continue

            tolerance = calendar_tolerance_pair(short_w, long_w)
            raw_decrease = short_w - long_w
            violation_amount = max(raw_decrease - tolerance, 0.0)
            is_violation = violation_amount > 0
            severity = classify_calendar_severity(violation_amount)

            avg_w = 0.5 * (abs(short_w) + abs(long_w))
            avg_weight = np.nanmean(
                [
                    float(short_row.get("diagnostic_weight", np.nan)),
                    float(long_row.get("diagnostic_weight", np.nan)),
                ]
            )

            rows.append(
                {
                    "diagnostic_type": "calendar_total_variance_nearest_neighbor",
                    "calendar_lens": "total_variance",
                    "expiry_pair_id": int(pair["expiry_pair_id"]),
                    "short_expiry": pair["short_expiry"],
                    "long_expiry": pair["long_expiry"],
                    "short_expiry_rank": int(pair["short_expiry_rank"]),
                    "long_expiry_rank": int(pair["long_expiry_rank"]),
                    "short_dte_calendar": float(pair["short_dte_calendar"]),
                    "long_dte_calendar": float(pair["long_dte_calendar"]),
                    "short_tau_years": float(pair["short_tau_years"]),
                    "long_tau_years": float(pair["long_tau_years"]),
                    "tau_gap_years": float(pair["tau_gap_years"]),
                    "short_n10_row_id": int(short_row["n10_row_id"]),
                    "long_n10_row_id": int(long_row["n10_row_id"]),
                    "short_strike": float(short_row["strike"]),
                    "long_strike": float(long_row["strike"]),
                    "short_forward": float(short_row["forward"]),
                    "long_forward": float(long_row["forward"]),
                    "short_log_moneyness": k_short,
                    "long_log_moneyness": k_long,
                    "matched_log_moneyness_midpoint": 0.5 * (k_short + k_long),
                    "abs_log_moneyness_distance": k_distance,
                    "moneyness_bucket_n10": assign_moneyness_bucket(0.5 * (k_short + k_long)),
                    "short_total_variance": short_w,
                    "long_total_variance": long_w,
                    "total_variance_change_long_minus_short": long_w - short_w,
                    "raw_total_variance_decrease_before_tolerance": raw_decrease,
                    "tolerance": tolerance,
                    "violation_amount": violation_amount,
                    "is_violation": bool(is_violation),
                    "severity": severity,
                    "normalized_violation_by_total_variance": violation_amount / max(avg_w, TOL.eps),
                    "short_selected_iv": float(short_row["selected_iv"]),
                    "long_selected_iv": float(long_row["selected_iv"]),
                    "iv_change_long_minus_short": float(long_row["selected_iv"] - short_row["selected_iv"]),
                    "short_diagnostic_weight": float(short_row.get("diagnostic_weight", np.nan)),
                    "long_diagnostic_weight": float(long_row.get("diagnostic_weight", np.nan)),
                    "avg_diagnostic_weight": avg_weight,
                    "short_price_spread": float(short_row.get("price_spread", np.nan)),
                    "long_price_spread": float(long_row.get("price_spread", np.nan)),
                    "short_relative_price_spread": float(short_row.get("relative_price_spread", np.nan)),
                    "long_relative_price_spread": float(long_row.get("relative_price_spread", np.nan)),
                    "short_iv_uncertainty_width": float(short_row.get("iv_uncertainty_width", np.nan)),
                    "long_iv_uncertainty_width": float(long_row.get("iv_uncertainty_width", np.nan)),
                    "short_surface_row_source": short_row.get("surface_row_source", None),
                    "long_surface_row_source": long_row.get("surface_row_source", None),
                    "short_primary_pair_issue": short_row.get("primary_pair_issue", None),
                    "long_primary_pair_issue": long_row.get("primary_pair_issue", None),
                    "short_primary_stability_issue": short_row.get("primary_stability_issue", None),
                    "long_primary_stability_issue": long_row.get("primary_stability_issue", None),
                    "short_quality_score": float(short_row.get("quality_score", np.nan)),
                    "long_quality_score": float(long_row.get("quality_score", np.nan)),
                    "short_any_monotonicity_violation": bool(short_row.get("any_monotonicity_violation_touching_row", False)),
                    "long_any_monotonicity_violation": bool(long_row.get("any_monotonicity_violation_touching_row", False)),
                    "short_any_convexity_violation": bool(short_row.get("any_convexity_violation_touching_row", False)),
                    "long_any_convexity_violation": bool(long_row.get("any_convexity_violation_touching_row", False)),
                }
            )

    return pd.DataFrame(rows)


calendar_nn_diagnostics = make_calendar_nearest_neighbor_diagnostics(
    df=surface_calendar,
    expiry_pairs=expiry_pair_inventory,
)

calendar_nn_violation_ledger = (
    calendar_nn_diagnostics
    .loc[calendar_nn_diagnostics["is_violation"]]
    .sort_values(
        [
            "violation_amount",
            "normalized_violation_by_total_variance",
            "short_expiry_rank",
            "matched_log_moneyness_midpoint",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Attach row-level calendar exposure flags back to surface
# ------------------------------------------------------------

surface_calendar["calendar_nn_pair_count"] = 0
surface_calendar["calendar_nn_violation_pair_count"] = 0
surface_calendar["max_calendar_violation_amount"] = 0.0
surface_calendar["any_calendar_violation_touching_row"] = False

if len(calendar_nn_diagnostics) > 0:
    endpoint_records = []

    for side in ["short", "long"]:
        temp = calendar_nn_diagnostics[
            [
                f"{side}_n10_row_id",
                "is_violation",
                "violation_amount",
            ]
        ].rename(columns={f"{side}_n10_row_id": "n10_row_id"})

        endpoint_records.append(temp)

    calendar_endpoint_df = pd.concat(endpoint_records, axis=0, ignore_index=True)

    calendar_row_summary = (
        calendar_endpoint_df
        .groupby("n10_row_id")
        .agg(
            calendar_pair_count=("is_violation", "size"),
            calendar_violation_pair_count=("is_violation", "sum"),
            max_calendar_violation_amount_row=("violation_amount", "max"),
        )
        .reset_index()
    )

    surface_calendar = surface_calendar.merge(
        calendar_row_summary,
        on="n10_row_id",
        how="left",
    )

    surface_calendar["calendar_nn_pair_count"] = (
        surface_calendar["calendar_pair_count"].fillna(0).astype(int)
    )

    surface_calendar["calendar_nn_violation_pair_count"] = (
        surface_calendar["calendar_violation_pair_count"].fillna(0).astype(int)
    )

    surface_calendar["max_calendar_violation_amount"] = (
        surface_calendar["max_calendar_violation_amount_row"].fillna(0.0)
    )

    surface_calendar["any_calendar_violation_touching_row"] = (
        surface_calendar["calendar_nn_violation_pair_count"] > 0
    )

    surface_calendar = surface_calendar.drop(
        columns=[
            "calendar_pair_count",
            "calendar_violation_pair_count",
            "max_calendar_violation_amount_row",
        ],
        errors="ignore",
    )


# ------------------------------------------------------------
# Calendar summaries
# ------------------------------------------------------------

calendar_pair_inventory_summary = expiry_pair_inventory.copy()

calendar_grid_pair_summary = (
    calendar_grid_diagnostics
    .groupby(
        [
            "expiry_pair_id",
            "short_expiry",
            "long_expiry",
            "short_expiry_rank",
            "long_expiry_rank",
        ],
        dropna=False,
    )
    .agg(
        short_dte_calendar=("short_dte_calendar", "median"),
        long_dte_calendar=("long_dte_calendar", "median"),
        grid_points_tested=("is_violation", "size"),
        grid_violation_points=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_normalized_violation_by_total_variance=("normalized_violation_by_total_variance", "max"),
        min_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "min"),
        median_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "median"),
        min_iv_change_long_minus_short=("iv_change_long_minus_short", "min"),
        median_iv_change_long_minus_short=("iv_change_long_minus_short", "median"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
        min_log_moneyness=("log_moneyness", "min"),
        max_log_moneyness=("log_moneyness", "max"),
    )
    .reset_index()
    .sort_values(["short_expiry_rank", "long_expiry_rank"])
)

calendar_grid_pair_summary["grid_violation_rate"] = (
    calendar_grid_pair_summary["grid_violation_points"]
    / calendar_grid_pair_summary["grid_points_tested"].clip(lower=1)
)

calendar_nn_pair_summary = (
    calendar_nn_diagnostics
    .groupby(
        [
            "expiry_pair_id",
            "short_expiry",
            "long_expiry",
            "short_expiry_rank",
            "long_expiry_rank",
        ],
        dropna=False,
    )
    .agg(
        short_dte_calendar=("short_dte_calendar", "median"),
        long_dte_calendar=("long_dte_calendar", "median"),
        matched_pairs_tested=("is_violation", "size"),
        matched_violation_pairs=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_normalized_violation_by_total_variance=("normalized_violation_by_total_variance", "max"),
        min_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "min"),
        median_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "median"),
        min_iv_change_long_minus_short=("iv_change_long_minus_short", "min"),
        median_iv_change_long_minus_short=("iv_change_long_minus_short", "median"),
        median_abs_log_moneyness_distance=("abs_log_moneyness_distance", "median"),
        max_abs_log_moneyness_distance=("abs_log_moneyness_distance", "max"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["short_expiry_rank", "long_expiry_rank"])
)

calendar_nn_pair_summary["matched_violation_rate"] = (
    calendar_nn_pair_summary["matched_violation_pairs"]
    / calendar_nn_pair_summary["matched_pairs_tested"].clip(lower=1)
)

calendar_grid_bucket_summary = (
    calendar_grid_diagnostics
    .groupby(["moneyness_bucket_n10"], dropna=False)
    .agg(
        grid_points_tested=("is_violation", "size"),
        grid_violation_points=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_normalized_violation_by_total_variance=("normalized_violation_by_total_variance", "max"),
        min_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "min"),
        median_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "median"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
)

calendar_grid_bucket_summary["grid_violation_rate"] = (
    calendar_grid_bucket_summary["grid_violation_points"]
    / calendar_grid_bucket_summary["grid_points_tested"].clip(lower=1)
)

calendar_nn_bucket_summary = (
    calendar_nn_diagnostics
    .groupby(["moneyness_bucket_n10"], dropna=False)
    .agg(
        matched_pairs_tested=("is_violation", "size"),
        matched_violation_pairs=("is_violation", "sum"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        median_violation_amount=("violation_amount", "median"),
        max_normalized_violation_by_total_variance=("normalized_violation_by_total_variance", "max"),
        min_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "min"),
        median_total_variance_change_long_minus_short=("total_variance_change_long_minus_short", "median"),
        median_abs_log_moneyness_distance=("abs_log_moneyness_distance", "median"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
)

calendar_nn_bucket_summary["matched_violation_rate"] = (
    calendar_nn_bucket_summary["matched_violation_pairs"]
    / calendar_nn_bucket_summary["matched_pairs_tested"].clip(lower=1)
)

calendar_grid_severity_summary = (
    calendar_grid_diagnostics
    .groupby(["severity"], dropna=False)
    .agg(
        grid_point_count=("is_violation", "size"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        max_normalized_violation_by_total_variance=("normalized_violation_by_total_variance", "max"),
    )
    .reset_index()
)

calendar_grid_severity_summary["severity"] = pd.Categorical(
    calendar_grid_severity_summary["severity"],
    categories=SEVERITY_ORDER,
    ordered=True,
)

calendar_grid_severity_summary = calendar_grid_severity_summary.sort_values("severity").reset_index(drop=True)

calendar_nn_severity_summary = (
    calendar_nn_diagnostics
    .groupby(["severity"], dropna=False)
    .agg(
        matched_pair_count=("is_violation", "size"),
        max_violation_amount=("violation_amount", "max"),
        mean_violation_amount=("violation_amount", "mean"),
        max_normalized_violation_by_total_variance=("normalized_violation_by_total_variance", "max"),
    )
    .reset_index()
)

calendar_nn_severity_summary["severity"] = pd.Categorical(
    calendar_nn_severity_summary["severity"],
    categories=SEVERITY_ORDER,
    ordered=True,
)

calendar_nn_severity_summary = calendar_nn_severity_summary.sort_values("severity").reset_index(drop=True)


# ------------------------------------------------------------
# Validation ledger
# ------------------------------------------------------------

n_expiry_pairs = len(expiry_pair_inventory)
n_sufficient_overlap_pairs = int(expiry_pair_inventory["has_sufficient_overlap"].sum()) if n_expiry_pairs > 0 else 0
n_grid_violations = int(calendar_grid_diagnostics["is_violation"].sum()) if len(calendar_grid_diagnostics) else 0
n_nn_violations = int(calendar_nn_diagnostics["is_violation"].sum()) if len(calendar_nn_diagnostics) else 0

calendar_validation_rows = [
    validation_row(
        "calendar_expiry_pairs_constructed",
        n_expiry_pairs > 0,
        f"{n_expiry_pairs:,} adjacent expiry pairs constructed",
    ),
    validation_row(
        "calendar_pairs_have_sufficient_overlap",
        n_sufficient_overlap_pairs > 0,
        f"{n_sufficient_overlap_pairs:,} adjacent expiry pairs have overlap width >= {CALENDAR_MIN_OVERLAP_WIDTH}",
    ),
    validation_row(
        "calendar_grid_diagnostics_computed",
        len(calendar_grid_diagnostics) > 0,
        f"{len(calendar_grid_diagnostics):,} interpolated grid points tested",
    ),
    validation_row(
        "calendar_nearest_neighbor_diagnostics_computed",
        len(calendar_nn_diagnostics) > 0,
        f"{len(calendar_nn_diagnostics):,} nearest-neighbor observation pairs tested",
    ),
    validation_row(
        "calendar_grid_total_variance_clean",
        n_grid_violations == 0,
        f"{n_grid_violations:,} interpolated total-variance calendar violations",
    ),
    validation_row(
        "calendar_nearest_neighbor_total_variance_clean",
        n_nn_violations == 0,
        f"{n_nn_violations:,} nearest-neighbor total-variance calendar violations",
    ),
    validation_row(
        "row_level_calendar_flags_attached",
        "any_calendar_violation_touching_row" in surface_calendar.columns,
        "row-level calendar flags attached to surface_calendar",
    ),
]

calendar_validation_ledger = pd.DataFrame(calendar_validation_rows)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("Calendar / total-variance validation ledger")
print("=" * 90)
display(calendar_validation_ledger)

print("=" * 90)
print("Adjacent expiry-pair inventory")
print("=" * 90)
display(calendar_pair_inventory_summary)

print("=" * 90)
print("Interpolated calendar-grid summary by expiry pair")
print("=" * 90)
display(calendar_grid_pair_summary)

print("=" * 90)
print("Nearest-neighbor calendar summary by expiry pair")
print("=" * 90)
display(calendar_nn_pair_summary)

print("=" * 90)
print("Interpolated calendar-grid summary by moneyness bucket")
print("=" * 90)
display(calendar_grid_bucket_summary)

print("=" * 90)
print("Nearest-neighbor calendar summary by moneyness bucket")
print("=" * 90)
display(calendar_nn_bucket_summary)

print("=" * 90)
print("Calendar-grid severity summary")
print("=" * 90)
display(calendar_grid_severity_summary)

print("=" * 90)
print("Nearest-neighbor calendar severity summary")
print("=" * 90)
display(calendar_nn_severity_summary)

print("=" * 90)
print("Calendar-grid violation ledger")
print("=" * 90)

if len(calendar_grid_violation_ledger) == 0:
    print("No interpolated total-variance calendar violations detected under configured tolerances.")
else:
    display(calendar_grid_violation_ledger.head(75))

print("=" * 90)
print("Nearest-neighbor calendar violation ledger")
print("=" * 90)

if len(calendar_nn_violation_ledger) == 0:
    print("No nearest-neighbor total-variance calendar violations detected under configured tolerances.")
else:
    display(calendar_nn_violation_ledger.head(75))


RUN_METADATA["calendar_validation"] = calendar_validation_ledger.to_dict(orient="records")
RUN_METADATA["calendar_grid_pair_summary"] = calendar_grid_pair_summary.to_dict(orient="records")
RUN_METADATA["calendar_nn_pair_summary"] = calendar_nn_pair_summary.to_dict(orient="records")

surface_calendar[
    [
        "expiry",
        "dte_calendar",
        "strike",
        "log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "total_variance",
        "calendar_nn_pair_count",
        "calendar_nn_violation_pair_count",
        "max_calendar_violation_amount",
        "any_calendar_violation_touching_row",
        "any_monotonicity_violation_touching_row",
        "any_convexity_violation_touching_row",
        "diagnostic_weight",
    ]
].head()

Calendar / total-variance validation ledger


,check,passed,details
0,calendar_expiry_pairs_constructed,True,7 adjacent expiry pairs constructed
1,calendar_pairs_have_sufficient_overlap,True,7 adjacent expiry pairs have overlap width >= ...
2,calendar_grid_diagnostics_computed,True,707 interpolated grid points tested
3,calendar_nearest_neighbor_diagnostics_computed,True,"1,097 nearest-neighbor observation pairs tested"
4,calendar_grid_total_variance_clean,True,0 interpolated total-variance calendar violations
5,calendar_nearest_neighbor_total_variance_clean,True,0 nearest-neighbor total-variance calendar vio...
6,row_level_calendar_flags_attached,True,row-level calendar flags attached to surface_c...


Adjacent expiry-pair inventory


,expiry_pair_id,short_expiry,long_expiry,short_expiry_rank,long_expiry_rank,short_dte_calendar,long_dte_calendar,short_tau_years,long_tau_years,tau_gap_years,short_n_obs,long_n_obs,short_min_log_moneyness,short_max_log_moneyness,long_min_log_moneyness,long_max_log_moneyness,overlap_left_log_moneyness,overlap_right_log_moneyness,overlap_width,has_sufficient_overlap,short_median_total_variance,long_median_total_variance,short_median_diagnostic_weight,long_median_diagnostic_weight
0,0,2026-07-10 00:00:00+00:00,2026-07-17 00:00:00+00:00,0,1,5.00000000,12.00000000,0.01369863,0.03287671,0.01917808,117,154,-0.12421955,0.03036525,-0.33180851,0.05160846,-0.12421955,0.03036525,0.15458480,True,0.00058161,0.00135022,0.89214865,0.94861885
1,1,2026-07-17 00:00:00+00:00,2026-07-24 00:00:00+00:00,1,2,12.00000000,19.00000000,0.03287671,0.05205479,0.01917808,154,168,-0.33180851,0.05160846,-0.29585467,0.06853817,-0.29585467,0.05160846,0.34746313,True,0.00135022,0.00168588,0.94861885,0.95892869
2,2,2026-07-24 00:00:00+00:00,2026-07-31 00:00:00+00:00,2,3,19.00000000,26.00000000,0.05205479,0.07123288,0.01917808,168,183,-0.29585467,0.06853817,-0.40090807,0.08151808,-0.29585467,0.06853817,0.36439283,True,0.00168588,0.00315627,0.95892869,1.00000000
3,3,2026-07-31 00:00:00+00:00,2026-08-07 00:00:00+00:00,3,4,26.00000000,33.00000000,0.07123288,0.09041096,0.01917808,183,132,-0.40090807,0.08151808,-0.33424011,0.09887653,-0.33424011,0.08151808,0.41575818,True,0.00315627,0.00259486,1.00000000,0.97851477
4,4,2026-08-07 00:00:00+00:00,2026-08-21 00:00:00+00:00,4,5,33.00000000,47.00000000,0.09041096,0.12876712,0.03835616,132,173,-0.33424011,0.09887653,-0.50848622,0.13919858,-0.33424011,0.09887653,0.43311664,True,0.00259486,0.00486473,0.97851477,1.00000000
5,5,2026-08-21 00:00:00+00:00,2026-08-31 00:00:00+00:00,5,6,47.00000000,57.00000000,0.12876712,0.15616438,0.02739726,173,187,-0.50848622,0.13919858,-0.38422144,0.12070446,-0.38422144,0.12070446,0.50492590,True,0.00486473,0.00598362,1.00000000,1.00000000
6,6,2026-08-31 00:00:00+00:00,2026-09-18 00:00:00+00:00,6,7,57.00000000,75.00000000,0.15616438,0.20547945,0.04931507,187,219,-0.38422144,0.12070446,-0.42520771,0.20476057,-0.38422144,0.12070446,0.50492590,True,0.00598362,0.00866784,1.00000000,0.99365549


Interpolated calendar-grid summary by expiry pair


...

,severity,grid_point_count,max_violation_amount,mean_violation_amount,max_normalized_violation_by_total_variance
0,clean,707,0.00000000,0.00000000,0.00000000


Nearest-neighbor calendar severity summary


,severity,matched_pair_count,max_violation_amount,mean_violation_amount,max_normalized_violation_by_total_variance
0,clean,1097,0.00000000,0.00000000,0.00000000


Calendar-grid violation ledger
No interpolated total-variance calendar violations detected under configured tolerances.
Nearest-neighbor calendar violation ledger
No nearest-neighbor total-variance calendar violations detected under configured tolerances.


,expiry,dte_calendar,strike,log_moneyness,moneyness_bucket_n10,selected_iv,total_variance,calendar_nn_pair_count,calendar_nn_violation_pair_count,max_calendar_violation_amount,any_calendar_violation_touching_row,any_monotonicity_violation_touching_row,any_convexity_violation_touching_row,diagnostic_weight
0,2026-07-10 00:00:00+00:00,5.00000000,658.00000000,-0.12421955,put_wing,0.40456494,0.00224209,1,0,0.00000000,False,False,False,0.28430707
1,2026-07-10 00:00:00+00:00,5.00000000,659.00000000,-0.12270095,put_wing,0.40010994,0.00219299,1,0,0.00000000,False,False,False,0.28566013
2,2026-07-10 00:00:00+00:00,5.00000000,660.00000000,-0.12118465,put_wing,0.39565753,0.00214445,1,0,0.00000000,False,False,False,0.28703299
3,2026-07-10 00:00:00+00:00,5.00000000,661.00000000,-0.11967065,put_wing,0.39120764,0.00209649,1,0,0.00000000,False,False,False,0.28842617
4,2026-07-10 00:00:00+00:00,5.00000000,662.00000000,-0.11815893,put_wing,0.38676019,0.00204909,1,0,0.00000000,False,False,False,0.28984024


In [9]:
# ============================================================
# IV-space roughness and violation-context diagnostics
# ============================================================
# Static-arbitrage violations are price-space statements.
# This cell does NOT redefine arbitrage in IV space.
#
# Purpose:
#   1. measure raw IV / total-variance roughness across strike;
#   2. identify whether convexity violations are concentrated in noisy regions;
#   3. separate economically meaningful defects from quote-grid / low-price artifacts;
#   4. prepare row-level severity inputs for the later handoff policy.
#
# Key distinction:
#
#   - Price-space convexity failures are the actual static-arbitrage signal.
#   - IV-space roughness is an explanatory diagnostic.
#
# A rough IV smile can still be arbitrage-free.
# A smooth IV smile can still hide arbitrage after conversion to prices.
# Therefore this cell is context, not a replacement for Cells 6-9.


surface_iv_context = surface_calendar.copy()


# ------------------------------------------------------------
# Robust helpers
# ------------------------------------------------------------

def classify_context_severity(value: float) -> str:
    """
    Generic severity classification for positive diagnostic magnitudes.
    """
    if not np.isfinite(value) or value <= 0:
        return "clean"

    if value <= TOL.mild_violation_threshold:
        return "micro_noise"

    if value <= TOL.moderate_violation_threshold:
        return "mild_warning"

    if value <= TOL.severe_violation_threshold:
        return "moderate_warning"

    return "severe_violation"


def safe_ratio(numerator: pd.Series | np.ndarray | float, denominator: pd.Series | np.ndarray | float) -> np.ndarray:
    """
    Stable ratio that avoids division by zero.
    """
    num = np.asarray(numerator, dtype=float)
    den = np.asarray(denominator, dtype=float)
    return num / np.maximum(np.abs(den), TOL.eps)


def extract_unique_violation_row_ids(
    convexity_ledger: pd.DataFrame,
    monotonicity_ledger: pd.DataFrame,
    calendar_nn_ledger: pd.DataFrame,
) -> Dict[str, set]:
    """
    Collect row ids touched by each diagnostic family.
    """
    ids = {
        "convexity": set(),
        "monotonicity": set(),
        "calendar": set(),
    }

    if len(convexity_ledger) > 0:
        for col in ["left_n10_row_id", "mid_n10_row_id", "right_n10_row_id"]:
            if col in convexity_ledger.columns:
                ids["convexity"].update(
                    convexity_ledger[col].dropna().astype(int).tolist()
                )

    if len(monotonicity_ledger) > 0:
        for col in ["left_n10_row_id", "right_n10_row_id"]:
            if col in monotonicity_ledger.columns:
                ids["monotonicity"].update(
                    monotonicity_ledger[col].dropna().astype(int).tolist()
                )

    if len(calendar_nn_ledger) > 0:
        for col in ["short_n10_row_id", "long_n10_row_id"]:
            if col in calendar_nn_ledger.columns:
                ids["calendar"].update(
                    calendar_nn_ledger[col].dropna().astype(int).tolist()
                )

    return ids


violation_row_ids = extract_unique_violation_row_ids(
    convexity_ledger=convexity_violation_ledger,
    monotonicity_ledger=monotonicity_violation_ledger,
    calendar_nn_ledger=calendar_nn_violation_ledger,
)

surface_iv_context["touched_by_convexity_violation"] = surface_iv_context["n10_row_id"].isin(
    violation_row_ids["convexity"]
)

surface_iv_context["touched_by_monotonicity_violation"] = surface_iv_context["n10_row_id"].isin(
    violation_row_ids["monotonicity"]
)

surface_iv_context["touched_by_calendar_violation"] = surface_iv_context["n10_row_id"].isin(
    violation_row_ids["calendar"]
)

surface_iv_context["touched_by_any_static_violation"] = (
    surface_iv_context["touched_by_convexity_violation"]
    | surface_iv_context["touched_by_monotonicity_violation"]
    | surface_iv_context["touched_by_calendar_violation"]
)


# ------------------------------------------------------------
# Adjacent IV and total-variance slope diagnostics
# ------------------------------------------------------------

def make_adjacent_iv_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build adjacent strike-pair diagnostics for IV and total variance.
    """
    rows: List[Dict[str, Any]] = []

    cols = [
        "n10_row_id",
        "expiry",
        "expiry_rank",
        "dte_calendar",
        "tau_years",
        "maturity_bucket",
        "strike",
        "forward",
        "log_moneyness",
        "abs_log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "total_variance",
        "diagnostic_weight",
        "bid",
        "ask",
        "selected_price",
        "price_spread",
        "relative_price_spread",
        "iv_uncertainty_width",
        "relative_iv_uncertainty_width",
        "surface_row_source",
        "primary_pair_issue",
        "primary_stability_issue",
        "quality_score",
        "touched_by_convexity_violation",
        "touched_by_monotonicity_violation",
        "touched_by_calendar_violation",
        "touched_by_any_static_violation",
    ]

    available_cols = [col for col in cols if col in df.columns]

    working = (
        df.loc[df["eligible_pointwise_diagnostics"], available_cols]
        .copy()
        .sort_values(["expiry", "strike", "n10_row_id"])
        .reset_index(drop=True)
    )

    for expiry, group in working.groupby("expiry", sort=True):
        group = group.sort_values(["strike", "n10_row_id"]).reset_index(drop=True)

        if len(group) < 2:
            continue

        for i in range(len(group) - 1):
            left = group.iloc[i]
            right = group.iloc[i + 1]

            k_left = float(left["log_moneyness"])
            k_right = float(right["log_moneyness"])
            strike_left = float(left["strike"])
            strike_right = float(right["strike"])

            if not all(np.isfinite([k_left, k_right, strike_left, strike_right])):
                continue

            if strike_right <= strike_left or k_right <= k_left:
                continue

            dk = k_right - k_left
            dK = strike_right - strike_left

            iv_left = float(left["selected_iv"])
            iv_right = float(right["selected_iv"])
            w_left = float(left["total_variance"])
            w_right = float(right["total_variance"])

            if not all(np.isfinite([iv_left, iv_right, w_left, w_right])):
                continue

            iv_change = iv_right - iv_left
            total_variance_change = w_right - w_left

            iv_slope_logm = iv_change / dk
            total_variance_slope_logm = total_variance_change / dk

            avg_iv_uncertainty = np.nanmean(
                [
                    float(left.get("iv_uncertainty_width", np.nan)),
                    float(right.get("iv_uncertainty_width", np.nan)),
                ]
            )

            avg_price_spread = np.nanmean(
                [
                    float(left.get("price_spread", np.nan)),
                    float(right.get("price_spread", np.nan)),
                ]
            )

            avg_diagnostic_weight = np.nanmean(
                [
                    float(left.get("diagnostic_weight", np.nan)),
                    float(right.get("diagnostic_weight", np.nan)),
                ]
            )

            pair_touched_by_convexity = bool(
                left.get("touched_by_convexity_violation", False)
                or right.get("touched_by_convexity_violation", False)
            )

            pair_touched_by_any_static = bool(
                left.get("touched_by_any_static_violation", False)
                or right.get("touched_by_any_static_violation", False)
            )

            iv_jump_vs_uncertainty = (
                abs(iv_change) / max(avg_iv_uncertainty, TOL.eps)
                if np.isfinite(avg_iv_uncertainty)
                else np.nan
            )

            rows.append(
                {
                    "diagnostic_type": "adjacent_iv_roughness",
                    "expiry": expiry,
                    "expiry_rank": int(left["expiry_rank"]),
                    "dte_calendar": float(left["dte_calendar"]),
                    "tau_years": float(left["tau_years"]),
                    "maturity_bucket": left["maturity_bucket"],
                    "left_n10_row_id": int(left["n10_row_id"]),
                    "right_n10_row_id": int(right["n10_row_id"]),
                    "left_strike": strike_left,
                    "right_strike": strike_right,
                    "strike_gap": dK,
                    "left_log_moneyness": k_left,
                    "right_log_moneyness": k_right,
                    "mid_log_moneyness": 0.5 * (k_left + k_right),
                    "mid_moneyness_bucket": assign_moneyness_bucket(0.5 * (k_left + k_right)),
                    "left_selected_iv": iv_left,
                    "right_selected_iv": iv_right,
                    "iv_change_right_minus_left": iv_change,
                    "abs_iv_change": abs(iv_change),
                    "iv_slope_log_moneyness": iv_slope_logm,
                    "abs_iv_slope_log_moneyness": abs(iv_slope_logm),
                    "left_total_variance": w_left,
                    "right_total_variance": w_right,
                    "total_variance_change_right_minus_left": total_variance_change,
                    "total_variance_slope_log_moneyness": total_variance_slope_logm,
                    "abs_total_variance_slope_log_moneyness": abs(total_variance_slope_logm),
                    "avg_iv_uncertainty_width": avg_iv_uncertainty,
                    "iv_jump_vs_avg_iv_uncertainty": iv_jump_vs_uncertainty,
                    "avg_price_spread": avg_price_spread,
                    "left_relative_price_spread": float(left.get("relative_price_spread", np.nan)),
                    "right_relative_price_spread": float(right.get("relative_price_spread", np.nan)),
                    "avg_diagnostic_weight": avg_diagnostic_weight,
                    "left_surface_row_source": left.get("surface_row_source", None),
                    "right_surface_row_source": right.get("surface_row_source", None),
                    "left_primary_pair_issue": left.get("primary_pair_issue", None),
                    "right_primary_pair_issue": right.get("primary_pair_issue", None),
                    "left_primary_stability_issue": left.get("primary_stability_issue", None),
                    "right_primary_stability_issue": right.get("primary_stability_issue", None),
                    "left_quality_score": float(left.get("quality_score", np.nan)),
                    "right_quality_score": float(right.get("quality_score", np.nan)),
                    "pair_touched_by_convexity_violation": pair_touched_by_convexity,
                    "pair_touched_by_any_static_violation": pair_touched_by_any_static,
                }
            )

    return pd.DataFrame(rows)


adjacent_iv_diagnostics = make_adjacent_iv_diagnostics(surface_iv_context)


# ------------------------------------------------------------
# Three-point IV curvature diagnostics
# ------------------------------------------------------------

def make_iv_curvature_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build three-point IV and total-variance curvature diagnostics.
    """
    rows: List[Dict[str, Any]] = []

    cols = [
        "n10_row_id",
        "expiry",
        "expiry_rank",
        "dte_calendar",
        "tau_years",
        "maturity_bucket",
        "strike",
        "forward",
        "log_moneyness",
        "abs_log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "total_variance",
        "diagnostic_weight",
        "selected_price",
        "price_spread",
        "relative_price_spread",
        "iv_uncertainty_width",
        "relative_iv_uncertainty_width",
        "surface_row_source",
        "primary_pair_issue",
        "primary_stability_issue",
        "quality_score",
        "touched_by_convexity_violation",
        "touched_by_monotonicity_violation",
        "touched_by_calendar_violation",
        "touched_by_any_static_violation",
    ]

    available_cols = [col for col in cols if col in df.columns]

    working = (
        df.loc[df["eligible_pointwise_diagnostics"], available_cols]
        .copy()
        .sort_values(["expiry", "strike", "n10_row_id"])
        .reset_index(drop=True)
    )

    for expiry, group in working.groupby("expiry", sort=True):
        group = group.sort_values(["strike", "n10_row_id"]).reset_index(drop=True)

        if len(group) < 3:
            continue

        for i in range(1, len(group) - 1):
            left = group.iloc[i - 1]
            mid = group.iloc[i]
            right = group.iloc[i + 1]

            k_left = float(left["log_moneyness"])
            k_mid = float(mid["log_moneyness"])
            k_right = float(right["log_moneyness"])

            if not all(np.isfinite([k_left, k_mid, k_right])):
                continue

            if not (k_left < k_mid < k_right):
                continue

            iv_left = float(left["selected_iv"])
            iv_mid = float(mid["selected_iv"])
            iv_right = float(right["selected_iv"])

            w_left = float(left["total_variance"])
            w_mid = float(mid["total_variance"])
            w_right = float(right["total_variance"])

            if not all(np.isfinite([iv_left, iv_mid, iv_right, w_left, w_mid, w_right])):
                continue

            left_width = k_mid - k_left
            right_width = k_right - k_mid
            full_width = k_right - k_left

            left_iv_slope = (iv_mid - iv_left) / left_width
            right_iv_slope = (iv_right - iv_mid) / right_width
            iv_slope_change = right_iv_slope - left_iv_slope
            iv_curvature_proxy = 2.0 * iv_slope_change / full_width

            left_w_slope = (w_mid - w_left) / left_width
            right_w_slope = (w_right - w_mid) / right_width
            w_slope_change = right_w_slope - left_w_slope
            total_variance_curvature_proxy = 2.0 * w_slope_change / full_width

            avg_iv_uncertainty = np.nanmean(
                [
                    float(left.get("iv_uncertainty_width", np.nan)),
                    float(mid.get("iv_uncertainty_width", np.nan)),
                    float(right.get("iv_uncertainty_width", np.nan)),
                ]
            )

            avg_price_spread = np.nanmean(
                [
                    float(left.get("price_spread", np.nan)),
                    float(mid.get("price_spread", np.nan)),
                    float(right.get("price_spread", np.nan)),
                ]
            )

            avg_diagnostic_weight = np.nanmean(
                [
                    float(left.get("diagnostic_weight", np.nan)),
                    float(mid.get("diagnostic_weight", np.nan)),
                    float(right.get("diagnostic_weight", np.nan)),
                ]
            )

            triplet_touched_by_convexity = bool(
                left.get("touched_by_convexity_violation", False)
                or mid.get("touched_by_convexity_violation", False)
                or right.get("touched_by_convexity_violation", False)
            )

            triplet_touched_by_any_static = bool(
                left.get("touched_by_any_static_violation", False)
                or mid.get("touched_by_any_static_violation", False)
                or right.get("touched_by_any_static_violation", False)
            )

            local_iv_range = max(iv_left, iv_mid, iv_right) - min(iv_left, iv_mid, iv_right)
            local_w_range = max(w_left, w_mid, w_right) - min(w_left, w_mid, w_right)

            local_iv_range_vs_uncertainty = (
                local_iv_range / max(avg_iv_uncertainty, TOL.eps)
                if np.isfinite(avg_iv_uncertainty)
                else np.nan
            )

            rows.append(
                {
                    "diagnostic_type": "three_point_iv_curvature",
                    "expiry": expiry,
                    "expiry_rank": int(mid["expiry_rank"]),
                    "dte_calendar": float(mid["dte_calendar"]),
                    "tau_years": float(mid["tau_years"]),
                    "maturity_bucket": mid["maturity_bucket"],
                    "left_n10_row_id": int(left["n10_row_id"]),
                    "mid_n10_row_id": int(mid["n10_row_id"]),
                    "right_n10_row_id": int(right["n10_row_id"]),
                    "left_strike": float(left["strike"]),
                    "mid_strike": float(mid["strike"]),
                    "right_strike": float(right["strike"]),
                    "left_log_moneyness": k_left,
                    "mid_log_moneyness": k_mid,
                    "right_log_moneyness": k_right,
                    "mid_moneyness_bucket": assign_moneyness_bucket(k_mid),
                    "left_width_log_moneyness": left_width,
                    "right_width_log_moneyness": right_width,
                    "full_width_log_moneyness": full_width,
                    "left_selected_iv": iv_left,
                    "mid_selected_iv": iv_mid,
                    "right_selected_iv": iv_right,
                    "local_iv_range": local_iv_range,
                    "left_iv_slope_log_moneyness": left_iv_slope,
                    "right_iv_slope_log_moneyness": right_iv_slope,
                    "iv_slope_change": iv_slope_change,
                    "iv_curvature_proxy": iv_curvature_proxy,
                    "abs_iv_curvature_proxy": abs(iv_curvature_proxy),
                    "left_total_variance": w_left,
                    "mid_total_variance": w_mid,
                    "right_total_variance": w_right,
                    "local_total_variance_range": local_w_range,
                    "left_total_variance_slope_log_moneyness": left_w_slope,
                    "right_total_variance_slope_log_moneyness": right_w_slope,
                    "total_variance_slope_change": w_slope_change,
                    "total_variance_curvature_proxy": total_variance_curvature_proxy,
                    "abs_total_variance_curvature_proxy": abs(total_variance_curvature_proxy),
                    "avg_iv_uncertainty_width": avg_iv_uncertainty,
                    "local_iv_range_vs_avg_iv_uncertainty": local_iv_range_vs_uncertainty,
                    "avg_price_spread": avg_price_spread,
                    "left_relative_price_spread": float(left.get("relative_price_spread", np.nan)),
                    "mid_relative_price_spread": float(mid.get("relative_price_spread", np.nan)),
                    "right_relative_price_spread": float(right.get("relative_price_spread", np.nan)),
                    "avg_diagnostic_weight": avg_diagnostic_weight,
                    "left_surface_row_source": left.get("surface_row_source", None),
                    "mid_surface_row_source": mid.get("surface_row_source", None),
                    "right_surface_row_source": right.get("surface_row_source", None),
                    "left_primary_pair_issue": left.get("primary_pair_issue", None),
                    "mid_primary_pair_issue": mid.get("primary_pair_issue", None),
                    "right_primary_pair_issue": right.get("primary_pair_issue", None),
                    "left_primary_stability_issue": left.get("primary_stability_issue", None),
                    "mid_primary_stability_issue": mid.get("primary_stability_issue", None),
                    "right_primary_stability_issue": right.get("primary_stability_issue", None),
                    "left_quality_score": float(left.get("quality_score", np.nan)),
                    "mid_quality_score": float(mid.get("quality_score", np.nan)),
                    "right_quality_score": float(right.get("quality_score", np.nan)),
                    "triplet_touched_by_convexity_violation": triplet_touched_by_convexity,
                    "triplet_touched_by_any_static_violation": triplet_touched_by_any_static,
                }
            )

    return pd.DataFrame(rows)


iv_curvature_diagnostics = make_iv_curvature_diagnostics(surface_iv_context)


# ------------------------------------------------------------
# Quantile-based roughness flags
# ------------------------------------------------------------
# These are empirical context flags, not theoretical arbitrage thresholds.

def add_quantile_flag(
    df: pd.DataFrame,
    value_col: str,
    flag_col: str,
    q: float = 0.95,
    group_cols: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    Flag observations above a within-group quantile threshold.
    """
    out = df.copy()

    if len(out) == 0:
        out[flag_col] = False
        out[f"{flag_col}_threshold"] = np.nan
        return out

    if group_cols is None:
        threshold = out[value_col].quantile(q)
        out[f"{flag_col}_threshold"] = threshold
        out[flag_col] = out[value_col] >= threshold
        return out

    thresholds = (
        out.groupby(group_cols, dropna=False)[value_col]
        .quantile(q)
        .rename(f"{flag_col}_threshold")
        .reset_index()
    )

    out = out.merge(thresholds, on=group_cols, how="left")
    out[flag_col] = out[value_col] >= out[f"{flag_col}_threshold"]

    return out


adjacent_iv_diagnostics = add_quantile_flag(
    adjacent_iv_diagnostics,
    value_col="abs_iv_slope_log_moneyness",
    flag_col="high_abs_iv_slope_q95_by_expiry",
    q=0.95,
    group_cols=["expiry"],
)

adjacent_iv_diagnostics = add_quantile_flag(
    adjacent_iv_diagnostics,
    value_col="iv_jump_vs_avg_iv_uncertainty",
    flag_col="high_iv_jump_vs_uncertainty_q95_by_expiry",
    q=0.95,
    group_cols=["expiry"],
)

iv_curvature_diagnostics = add_quantile_flag(
    iv_curvature_diagnostics,
    value_col="abs_iv_curvature_proxy",
    flag_col="high_abs_iv_curvature_q95_by_expiry",
    q=0.95,
    group_cols=["expiry"],
)

iv_curvature_diagnostics = add_quantile_flag(
    iv_curvature_diagnostics,
    value_col="local_iv_range_vs_avg_iv_uncertainty",
    flag_col="high_local_iv_range_vs_uncertainty_q95_by_expiry",
    q=0.95,
    group_cols=["expiry"],
)


# ------------------------------------------------------------
# Attach row-level IV roughness exposure flags
# ------------------------------------------------------------

surface_iv_context["iv_adjacent_pair_count"] = 0
surface_iv_context["high_iv_slope_pair_count"] = 0
surface_iv_context["high_iv_jump_vs_uncertainty_pair_count"] = 0
surface_iv_context["iv_curvature_triplet_count"] = 0
surface_iv_context["high_iv_curvature_triplet_count"] = 0
surface_iv_context["high_local_iv_range_vs_uncertainty_triplet_count"] = 0

if len(adjacent_iv_diagnostics) > 0:
    adjacent_endpoint_frames = []

    for side in ["left", "right"]:
        temp = adjacent_iv_diagnostics[
            [
                f"{side}_n10_row_id",
                "high_abs_iv_slope_q95_by_expiry",
                "high_iv_jump_vs_uncertainty_q95_by_expiry",
            ]
        ].rename(columns={f"{side}_n10_row_id": "n10_row_id"})

        adjacent_endpoint_frames.append(temp)

    adjacent_endpoint_df = pd.concat(adjacent_endpoint_frames, axis=0, ignore_index=True)

    adjacent_row_summary = (
        adjacent_endpoint_df
        .groupby("n10_row_id")
        .agg(
            iv_adjacent_pair_count=("high_abs_iv_slope_q95_by_expiry", "size"),
            high_iv_slope_pair_count=("high_abs_iv_slope_q95_by_expiry", "sum"),
            high_iv_jump_vs_uncertainty_pair_count=("high_iv_jump_vs_uncertainty_q95_by_expiry", "sum"),
        )
        .reset_index()
    )

    surface_iv_context = surface_iv_context.merge(
        adjacent_row_summary,
        on="n10_row_id",
        how="left",
        suffixes=("", "_new"),
    )

    for col in [
        "iv_adjacent_pair_count",
        "high_iv_slope_pair_count",
        "high_iv_jump_vs_uncertainty_pair_count",
    ]:
        new_col = f"{col}_new"
        if new_col in surface_iv_context.columns:
            surface_iv_context[col] = surface_iv_context[new_col].fillna(surface_iv_context[col]).fillna(0).astype(int)
            surface_iv_context = surface_iv_context.drop(columns=[new_col])

if len(iv_curvature_diagnostics) > 0:
    curvature_endpoint_frames = []

    for side in ["left", "mid", "right"]:
        temp = iv_curvature_diagnostics[
            [
                f"{side}_n10_row_id",
                "high_abs_iv_curvature_q95_by_expiry",
                "high_local_iv_range_vs_uncertainty_q95_by_expiry",
            ]
        ].rename(columns={f"{side}_n10_row_id": "n10_row_id"})

        curvature_endpoint_frames.append(temp)

    curvature_endpoint_df = pd.concat(curvature_endpoint_frames, axis=0, ignore_index=True)

    curvature_row_summary = (
        curvature_endpoint_df
        .groupby("n10_row_id")
        .agg(
            iv_curvature_triplet_count=("high_abs_iv_curvature_q95_by_expiry", "size"),
            high_iv_curvature_triplet_count=("high_abs_iv_curvature_q95_by_expiry", "sum"),
            high_local_iv_range_vs_uncertainty_triplet_count=("high_local_iv_range_vs_uncertainty_q95_by_expiry", "sum"),
        )
        .reset_index()
    )

    surface_iv_context = surface_iv_context.merge(
        curvature_row_summary,
        on="n10_row_id",
        how="left",
        suffixes=("", "_new"),
    )

    for col in [
        "iv_curvature_triplet_count",
        "high_iv_curvature_triplet_count",
        "high_local_iv_range_vs_uncertainty_triplet_count",
    ]:
        new_col = f"{col}_new"
        if new_col in surface_iv_context.columns:
            surface_iv_context[col] = surface_iv_context[new_col].fillna(surface_iv_context[col]).fillna(0).astype(int)
            surface_iv_context = surface_iv_context.drop(columns=[new_col])

surface_iv_context["touched_by_high_iv_slope"] = surface_iv_context["high_iv_slope_pair_count"] > 0
surface_iv_context["touched_by_high_iv_jump_vs_uncertainty"] = surface_iv_context["high_iv_jump_vs_uncertainty_pair_count"] > 0
surface_iv_context["touched_by_high_iv_curvature"] = surface_iv_context["high_iv_curvature_triplet_count"] > 0
surface_iv_context["touched_by_high_local_iv_range_vs_uncertainty"] = (
    surface_iv_context["high_local_iv_range_vs_uncertainty_triplet_count"] > 0
)

surface_iv_context["touched_by_any_iv_roughness_flag"] = (
    surface_iv_context["touched_by_high_iv_slope"]
    | surface_iv_context["touched_by_high_iv_jump_vs_uncertainty"]
    | surface_iv_context["touched_by_high_iv_curvature"]
    | surface_iv_context["touched_by_high_local_iv_range_vs_uncertainty"]
)


# ------------------------------------------------------------
# Convexity violation context summary
# ------------------------------------------------------------

def summarize_violation_context(convexity_ledger: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize what kind of rows appear in the convexity violation ledger.
    """
    if len(convexity_ledger) == 0:
        return pd.DataFrame(
            [
                {
                    "context": "convexity_violation_ledger",
                    "violation_rows": 0,
                    "median_avg_diagnostic_weight": np.nan,
                    "median_avg_price_spread": np.nan,
                    "median_avg_iv_uncertainty_width": np.nan,
                    "share_with_any_single_side_fallback": np.nan,
                    "share_with_any_pair_issue_not_clean": np.nan,
                    "share_with_any_low_quality_score_below_0_80": np.nan,
                    "share_with_avg_price_spread_above_0_10": np.nan,
                }
            ]
        )

    context = convexity_ledger.copy()

    source_cols = [
        col for col in [
            "left_surface_row_source",
            "mid_surface_row_source",
            "right_surface_row_source",
        ]
        if col in context.columns
    ]

    pair_issue_cols = [
        col for col in [
            "left_primary_pair_issue",
            "mid_primary_pair_issue",
            "right_primary_pair_issue",
        ]
        if col in context.columns
    ]

    quality_cols = [
        col for col in [
            "left_quality_score",
            "mid_quality_score",
            "right_quality_score",
        ]
        if col in context.columns
    ]

    if source_cols:
        any_single_side = context[source_cols].astype(str).apply(
            lambda row: row.str.contains("single_side", case=False, regex=False).any(),
            axis=1,
        )
    else:
        any_single_side = pd.Series(False, index=context.index)

    if pair_issue_cols:
        any_pair_issue_not_clean = context[pair_issue_cols].astype(str).apply(
            lambda row: (~row.eq("pair_clean")).any(),
            axis=1,
        )
    else:
        any_pair_issue_not_clean = pd.Series(False, index=context.index)

    if quality_cols:
        min_quality = context[quality_cols].min(axis=1)
    else:
        min_quality = pd.Series(np.nan, index=context.index)

    return pd.DataFrame(
        [
            {
                "context": "convexity_violation_ledger",
                "violation_rows": int(len(context)),
                "unique_mid_rows": int(context["mid_n10_row_id"].nunique()) if "mid_n10_row_id" in context.columns else np.nan,
                "unique_expiries": int(context["expiry"].nunique()) if "expiry" in context.columns else np.nan,
                "median_violation_amount": float(context["violation_amount"].median()),
                "max_violation_amount": float(context["violation_amount"].max()),
                "median_normalized_violation_by_price": float(context["normalized_violation_by_price"].median()),
                "max_normalized_violation_by_price": float(context["normalized_violation_by_price"].max()),
                "median_normalized_violation_by_spread": float(context["normalized_violation_by_spread"].median()),
                "max_normalized_violation_by_spread": float(context["normalized_violation_by_spread"].max()),
                "median_avg_diagnostic_weight": float(context["avg_diagnostic_weight"].median()),
                "median_avg_price_spread": float(context["avg_price_spread"].median()),
                "median_avg_iv_uncertainty_width": float(context["avg_iv_uncertainty_width"].median()),
                "share_with_any_single_side_fallback": float(any_single_side.mean()),
                "share_with_any_pair_issue_not_clean": float(any_pair_issue_not_clean.mean()),
                "share_with_any_low_quality_score_below_0_80": float((min_quality < 0.80).mean()),
                "share_with_avg_price_spread_above_0_10": float((context["avg_price_spread"] > 0.10).mean()),
            }
        ]
    )


convexity_violation_context_summary = summarize_violation_context(convexity_violation_ledger)


# ------------------------------------------------------------
# IV roughness summaries
# ------------------------------------------------------------

adjacent_iv_summary_by_expiry = (
    adjacent_iv_diagnostics
    .groupby(["expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        adjacent_pairs=("diagnostic_type", "size"),
        median_abs_iv_change=("abs_iv_change", "median"),
        max_abs_iv_change=("abs_iv_change", "max"),
        median_abs_iv_slope=("abs_iv_slope_log_moneyness", "median"),
        q95_abs_iv_slope=("abs_iv_slope_log_moneyness", lambda x: x.quantile(0.95)),
        max_abs_iv_slope=("abs_iv_slope_log_moneyness", "max"),
        median_iv_jump_vs_uncertainty=("iv_jump_vs_avg_iv_uncertainty", "median"),
        q95_iv_jump_vs_uncertainty=("iv_jump_vs_avg_iv_uncertainty", lambda x: x.quantile(0.95)),
        max_iv_jump_vs_uncertainty=("iv_jump_vs_avg_iv_uncertainty", "max"),
        high_iv_slope_pairs=("high_abs_iv_slope_q95_by_expiry", "sum"),
        high_iv_jump_vs_uncertainty_pairs=("high_iv_jump_vs_uncertainty_q95_by_expiry", "sum"),
        pairs_touched_by_convexity_violation=("pair_touched_by_convexity_violation", "sum"),
        pairs_touched_by_any_static_violation=("pair_touched_by_any_static_violation", "sum"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["expiry_rank", "expiry"])
)

iv_curvature_summary_by_expiry = (
    iv_curvature_diagnostics
    .groupby(["expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        triplets=("diagnostic_type", "size"),
        median_abs_iv_curvature=("abs_iv_curvature_proxy", "median"),
        q95_abs_iv_curvature=("abs_iv_curvature_proxy", lambda x: x.quantile(0.95)),
        max_abs_iv_curvature=("abs_iv_curvature_proxy", "max"),
        median_local_iv_range=("local_iv_range", "median"),
        q95_local_iv_range=("local_iv_range", lambda x: x.quantile(0.95)),
        max_local_iv_range=("local_iv_range", "max"),
        median_local_iv_range_vs_uncertainty=("local_iv_range_vs_avg_iv_uncertainty", "median"),
        q95_local_iv_range_vs_uncertainty=("local_iv_range_vs_avg_iv_uncertainty", lambda x: x.quantile(0.95)),
        max_local_iv_range_vs_uncertainty=("local_iv_range_vs_avg_iv_uncertainty", "max"),
        high_iv_curvature_triplets=("high_abs_iv_curvature_q95_by_expiry", "sum"),
        high_local_iv_range_vs_uncertainty_triplets=("high_local_iv_range_vs_uncertainty_q95_by_expiry", "sum"),
        triplets_touched_by_convexity_violation=("triplet_touched_by_convexity_violation", "sum"),
        triplets_touched_by_any_static_violation=("triplet_touched_by_any_static_violation", "sum"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["expiry_rank", "expiry"])
)

adjacent_iv_summary_by_bucket = (
    adjacent_iv_diagnostics
    .groupby(["maturity_bucket", "mid_moneyness_bucket"], dropna=False)
    .agg(
        adjacent_pairs=("diagnostic_type", "size"),
        median_abs_iv_change=("abs_iv_change", "median"),
        max_abs_iv_change=("abs_iv_change", "max"),
        median_abs_iv_slope=("abs_iv_slope_log_moneyness", "median"),
        q95_abs_iv_slope=("abs_iv_slope_log_moneyness", lambda x: x.quantile(0.95)),
        max_abs_iv_slope=("abs_iv_slope_log_moneyness", "max"),
        median_iv_jump_vs_uncertainty=("iv_jump_vs_avg_iv_uncertainty", "median"),
        q95_iv_jump_vs_uncertainty=("iv_jump_vs_avg_iv_uncertainty", lambda x: x.quantile(0.95)),
        max_iv_jump_vs_uncertainty=("iv_jump_vs_avg_iv_uncertainty", "max"),
        pairs_touched_by_convexity_violation=("pair_touched_by_convexity_violation", "sum"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["maturity_bucket", "mid_moneyness_bucket"])
)

iv_curvature_summary_by_bucket = (
    iv_curvature_diagnostics
    .groupby(["maturity_bucket", "mid_moneyness_bucket"], dropna=False)
    .agg(
        triplets=("diagnostic_type", "size"),
        median_abs_iv_curvature=("abs_iv_curvature_proxy", "median"),
        q95_abs_iv_curvature=("abs_iv_curvature_proxy", lambda x: x.quantile(0.95)),
        max_abs_iv_curvature=("abs_iv_curvature_proxy", "max"),
        median_local_iv_range_vs_uncertainty=("local_iv_range_vs_avg_iv_uncertainty", "median"),
        q95_local_iv_range_vs_uncertainty=("local_iv_range_vs_avg_iv_uncertainty", lambda x: x.quantile(0.95)),
        max_local_iv_range_vs_uncertainty=("local_iv_range_vs_avg_iv_uncertainty", "max"),
        triplets_touched_by_convexity_violation=("triplet_touched_by_convexity_violation", "sum"),
        median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
    )
    .reset_index()
    .sort_values(["maturity_bucket", "mid_moneyness_bucket"])
)


# ------------------------------------------------------------
# Top roughness ledgers for inspection
# ------------------------------------------------------------

top_adjacent_iv_roughness_ledger = (
    adjacent_iv_diagnostics
    .sort_values(
        [
            "iv_jump_vs_avg_iv_uncertainty",
            "abs_iv_slope_log_moneyness",
            "expiry_rank",
            "left_strike",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

top_iv_curvature_ledger = (
    iv_curvature_diagnostics
    .sort_values(
        [
            "local_iv_range_vs_avg_iv_uncertainty",
            "abs_iv_curvature_proxy",
            "expiry_rank",
            "mid_strike",
        ],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

convexity_violation_context_by_bucket = pd.DataFrame()

if len(convexity_violation_ledger) > 0:
    convexity_violation_context_by_bucket = (
        convexity_violation_ledger
        .groupby(["price_lens", "maturity_bucket", "mid_moneyness_bucket"], dropna=False)
        .agg(
            violation_triplets=("diagnostic_type", "size"),
            max_violation_amount=("violation_amount", "max"),
            median_violation_amount=("violation_amount", "median"),
            median_normalized_violation_by_price=("normalized_violation_by_price", "median"),
            max_normalized_violation_by_price=("normalized_violation_by_price", "max"),
            median_normalized_violation_by_spread=("normalized_violation_by_spread", "median"),
            max_normalized_violation_by_spread=("normalized_violation_by_spread", "max"),
            median_avg_diagnostic_weight=("avg_diagnostic_weight", "median"),
            median_avg_price_spread=("avg_price_spread", "median"),
            median_avg_iv_uncertainty_width=("avg_iv_uncertainty_width", "median"),
            median_mid_quality_score=("mid_quality_score", "median"),
        )
        .reset_index()
        .sort_values(
            ["violation_triplets", "max_violation_amount"],
            ascending=[False, False],
        )
    )


# ------------------------------------------------------------
# Validation ledger
# ------------------------------------------------------------

iv_context_validation_rows = [
    validation_row(
        "adjacent_iv_diagnostics_computed",
        len(adjacent_iv_diagnostics) > 0,
        f"{len(adjacent_iv_diagnostics):,} adjacent IV pairs computed",
    ),
    validation_row(
        "iv_curvature_diagnostics_computed",
        len(iv_curvature_diagnostics) > 0,
        f"{len(iv_curvature_diagnostics):,} IV curvature triplets computed",
    ),
    validation_row(
        "convexity_violation_context_available",
        len(convexity_violation_context_summary) > 0,
        f"{len(convexity_violation_ledger):,} convexity violation ledger rows summarized",
    ),
    validation_row(
        "row_level_iv_context_flags_attached",
        "touched_by_any_iv_roughness_flag" in surface_iv_context.columns,
        "row-level IV roughness context flags attached",
    ),
    validation_row(
        "static_violation_rows_identified",
        surface_iv_context["touched_by_any_static_violation"].sum() >= 0,
        f"{int(surface_iv_context['touched_by_any_static_violation'].sum()):,} rows touched by at least one static violation family",
    ),
]

iv_context_validation_ledger = pd.DataFrame(iv_context_validation_rows)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("IV-space context validation ledger")
print("=" * 90)
display(iv_context_validation_ledger)

print("=" * 90)
print("Convexity violation context summary")
print("=" * 90)
display(convexity_violation_context_summary)

print("=" * 90)
print("Convexity violation context by bucket")
print("=" * 90)

if len(convexity_violation_context_by_bucket) == 0:
    print("No convexity violations to summarize by bucket.")
else:
    display(convexity_violation_context_by_bucket)

print("=" * 90)
print("Adjacent IV roughness summary by expiry")
print("=" * 90)
display(adjacent_iv_summary_by_expiry)

print("=" * 90)
print("IV curvature summary by expiry")
print("=" * 90)
display(iv_curvature_summary_by_expiry)

print("=" * 90)
print("Adjacent IV roughness summary by maturity / moneyness bucket")
print("=" * 90)
display(adjacent_iv_summary_by_bucket)

print("=" * 90)
print("IV curvature summary by maturity / moneyness bucket")
print("=" * 90)
display(iv_curvature_summary_by_bucket)

print("=" * 90)
print("Top adjacent IV roughness ledger")
print("=" * 90)
display(
    top_adjacent_iv_roughness_ledger[
        [
            "expiry",
            "dte_calendar",
            "left_strike",
            "right_strike",
            "mid_log_moneyness",
            "mid_moneyness_bucket",
            "left_selected_iv",
            "right_selected_iv",
            "abs_iv_change",
            "abs_iv_slope_log_moneyness",
            "avg_iv_uncertainty_width",
            "iv_jump_vs_avg_iv_uncertainty",
            "pair_touched_by_convexity_violation",
            "avg_diagnostic_weight",
            "left_surface_row_source",
            "right_surface_row_source",
            "left_primary_pair_issue",
            "right_primary_pair_issue",
        ]
    ].head(40)
)

print("=" * 90)
print("Top IV curvature roughness ledger")
print("=" * 90)
display(
    top_iv_curvature_ledger[
        [
            "expiry",
            "dte_calendar",
            "left_strike",
            "mid_strike",
            "right_strike",
            "mid_log_moneyness",
            "mid_moneyness_bucket",
            "left_selected_iv",
            "mid_selected_iv",
            "right_selected_iv",
            "local_iv_range",
            "abs_iv_curvature_proxy",
            "avg_iv_uncertainty_width",
            "local_iv_range_vs_avg_iv_uncertainty",
            "triplet_touched_by_convexity_violation",
            "avg_diagnostic_weight",
            "left_surface_row_source",
            "mid_surface_row_source",
            "right_surface_row_source",
            "left_primary_pair_issue",
            "mid_primary_pair_issue",
            "right_primary_pair_issue",
        ]
    ].head(40)
)


RUN_METADATA["iv_context_validation"] = iv_context_validation_ledger.to_dict(orient="records")
RUN_METADATA["convexity_violation_context_summary"] = convexity_violation_context_summary.to_dict(orient="records")
RUN_METADATA["adjacent_iv_summary_by_expiry"] = adjacent_iv_summary_by_expiry.to_dict(orient="records")
RUN_METADATA["iv_curvature_summary_by_expiry"] = iv_curvature_summary_by_expiry.to_dict(orient="records")

surface_iv_context[
    [
        "expiry",
        "dte_calendar",
        "strike",
        "log_moneyness",
        "moneyness_bucket_n10",
        "selected_iv",
        "total_variance",
        "touched_by_convexity_violation",
        "touched_by_monotonicity_violation",
        "touched_by_calendar_violation",
        "touched_by_any_static_violation",
        "touched_by_high_iv_slope",
        "touched_by_high_iv_jump_vs_uncertainty",
        "touched_by_high_iv_curvature",
        "touched_by_high_local_iv_range_vs_uncertainty",
        "touched_by_any_iv_roughness_flag",
        "diagnostic_weight",
    ]
].head()

IV-space context validation ledger


,check,passed,details
0,adjacent_iv_diagnostics_computed,True,"1,325 adjacent IV pairs computed"
1,iv_curvature_diagnostics_computed,True,"1,317 IV curvature triplets computed"
2,convexity_violation_context_available,True,326 convexity violation ledger rows summarized
3,row_level_iv_context_flags_attached,True,row-level IV roughness context flags attached
4,static_violation_rows_identified,True,392 rows touched by at least one static violat...


Convexity violation context summary


,context,violation_rows,unique_mid_rows,unique_expiries,median_violation_amount,max_violation_amount,median_normalized_violation_by_price,max_normalized_violation_by_price,median_normalized_violation_by_spread,max_normalized_violation_by_spread,median_avg_diagnostic_weight,median_avg_price_spread,median_avg_iv_uncertainty_width,share_with_any_single_side_fallback,share_with_any_pair_issue_not_clean,share_with_any_low_quality_score_below_0_80,share_with_avg_price_spread_above_0_10
0,convexity_violation_ledger,326,145,8,0.00239900,0.01611432,0.00211821,0.09481935,0.15781667,0.48990000,0.77466305,0.01666667,0.00097226,0.35582822,0.87730061,0.87730061,0.02453988


Convexity violation context by bucket


...

Top adjacent IV roughness ledger


,expiry,dte_calendar,left_strike,right_strike,mid_log_moneyness,mid_moneyness_bucket,left_selected_iv,right_selected_iv,abs_iv_change,abs_iv_slope_log_moneyness,avg_iv_uncertainty_width,iv_jump_vs_avg_iv_uncertainty,pair_touched_by_convexity_violation,avg_diagnostic_weight,left_surface_row_source,right_surface_row_source,left_primary_pair_issue,right_primary_pair_issue
0,2026-09-18 00:00:00+00:00,75.00000000,744.00000000,757.00000000,0.00108904,atm,0.15300254,0.14556669,0.00743585,0.42926673,0.00045049,16.50599389,False,1.00000000,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean
1,2026-08-31 00:00:00+00:00,57.00000000,580.00000000,590.00000000,-0.24705685,deep_put_wing,0.33359169,0.32056374,0.01302795,0.76211679,0.00091973,14.16491743,False,0.63037442,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean
2,2026-08-31 00:00:00+00:00,57.00000000,525.00000000,535.00000000,-0.34579966,deep_put_wing,0.40685459,0.39308354,0.01377105,0.72984406,0.00111453,12.35593298,False,0.36928036,matched_pair_preferred_side,matched_pair_preferred_side,not_both_stability_clean,not_both_stability_clean
3,2026-08-31 00:00:00+00:00,57.00000000,590.00000000,595.00000000,-0.23429020,deep_put_wing,0.32056374,0.31416530,0.00639844,0.75821091,0.00055028,11.62764575,False,0.66744264,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean
4,2026-08-31 00:00:00+00:00,57.00000000,555.00000000,565.00000000,-0.29073525,deep_put_wing,0.36635941,0.35312624,0.01323317,0.74103771,0.00122433,10.80846701,False,0.54010356,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean
5,2026-07-31 00:00:00+00:00,26.00000000,595.00000000,605.00000000,-0.21862124,deep_put_wing,0.37640971,0.35850817,0.01790154,1.07406735,0.00169338,10.57150442,False,0.46097237,matched_pair_preferred_side,matched_pair_preferred_side,not_both_stability_clean,not_both_stability_clean
6,2026-07-31 00:00:00+00:00,26.00000000,743.00000000,750.00000000,-0.00013155,atm,0.14524391,0.13720453,0.00803938,0.85733619,0.00076327,10.53281907,True,1.00000000,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean
7,2026-08-07 00:00:00+00:00,33.00000000,535.00000000,550.00000000,-0.32041434,deep_put_wing,0.45492290,0.43012836,0.02479455,0.89667895,0.00239965,10.33257726,True,0.25848820,single_side_stability_clean_fallback,single_side_stability_clean_fallback,unmatched_single_side,unmatched_single_side
8,2026-08-31 00:00:00+00:00,57.00000000,744.00000000,752.00000000,-0.00124349,atm,0.14830761,0.14271867,0.00558894,0.52256080,0.00055768,10.02183128,False,1.00000000,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean
9,2026-08-07 00:00:00+00:00,33.00000000,675.00000000,680.00000000,-0.09810411,put_wing,0.23009373,0.22349996,0.00659377,0.89345132,0.00066328,9.94120420,False,0.73625154,single_side_stability_clean_fallback,matched_pair_preferred_side,unmatched_single_side,not_both_stability_clean


Top IV curvature roughness ledger


,expiry,dte_calendar,left_strike,mid_strike,right_strike,mid_log_moneyness,mid_moneyness_bucket,left_selected_iv,mid_selected_iv,right_selected_iv,local_iv_range,abs_iv_curvature_proxy,avg_iv_uncertainty_width,local_iv_range_vs_avg_iv_uncertainty,triplet_touched_by_convexity_violation,avg_diagnostic_weight,left_surface_row_source,mid_surface_row_source,right_surface_row_source,left_primary_pair_issue,mid_primary_pair_issue,right_primary_pair_issue
0,2026-08-31 00:00:00+00:00,57.00000000,580.00000000,590.00000000,595.00000000,-0.23850963,deep_put_wing,0.33359169,0.32056374,0.31416530,0.01942640,0.30594376,0.00079111,24.55575696,False,0.64618831,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,pair_clean
1,2026-09-18 00:00:00+00:00,75.00000000,744.00000000,757.00000000,758.00000000,0.00975015,atm,0.15300254,0.14556669,0.14478118,0.00822136,17.78292619,0.00042556,19.31884173,False,1.00000000,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,pair_clean
2,2026-08-31 00:00:00+00:00,57.00000000,525.00000000,535.00000000,540.00000000,-0.33636542,deep_put_wing,0.40685459,0.39308354,0.38651754,0.02033705,1.70420728,0.00107546,18.91006676,False,0.37666358,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,not_both_stability_clean,not_both_stability_clean,not_both_stability_clean
3,2026-08-31 00:00:00+00:00,57.00000000,555.00000000,565.00000000,570.00000000,-0.28180644,deep_put_wing,0.36635941,0.35312624,0.34645038,0.01990903,1.25000768,0.00105472,18.87621030,False,0.51555152,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,not_both_stability_clean
4,2026-09-18 00:00:00+00:00,75.00000000,530.00000000,535.00000000,540.00000000,-0.33734635,deep_put_wing,0.37900798,0.37268022,0.36686448,0.01214350,5.21213617,0.00064886,18.71507933,True,0.48304809,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,not_both_stability_clean,not_both_stability_clean,not_both_stability_clean
5,2026-08-07 00:00:00+00:00,33.00000000,535.00000000,550.00000000,560.00000000,-0.30658858,deep_put_wing,0.45492290,0.43012836,0.41342764,0.04149526,1.32190771,0.00223918,18.53148928,True,0.26983086,single_side_stability_clean_fallback,single_side_stability_clean_fallback,single_side_stability_clean_fallback,unmatched_single_side,unmatched_single_side,unmatched_single_side
6,2026-08-07 00:00:00+00:00,33.00000000,675.00000000,680.00000000,685.00000000,-0.09441406,put_wing,0.23009373,0.22349996,0.21672647,0.01336726,4.23309420,0.00072807,18.35981064,False,0.75339814,single_side_stability_clean_fallback,matched_pair_preferred_side,single_side_stability_clean_fallback,unmatched_single_side,not_both_stability_clean,unmatched_single_side
7,2026-08-31 00:00:00+00:00,57.00000000,575.00000000,580.00000000,590.00000000,-0.25560407,deep_put_wing,0.33995298,0.33359169,0.32056374,0.01938924,2.12740863,0.00106339,18.23336135,False,0.57685200,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,not_both_stability_clean,pair_clean,pair_clean
8,2026-09-18 00:00:00+00:00,75.00000000,743.00000000,744.00000000,757.00000000,-0.00757207,atm,0.15377328,0.15300254,0.14556669,0.00820659,15.40405039,0.00045121,18.18781373,False,1.00000000,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,pair_clean
9,2026-08-31 00:00:00+00:00,57.00000000,590.00000000,595.00000000,600.00000000,-0.23007076,deep_put_wing,0.32056374,0.31416530,0.30780386,0.01275987,0.23511459,0.00070194,18.17802608,False,0.67369218,matched_pair_preferred_side,matched_pair_preferred_side,matched_pair_preferred_side,pair_clean,pair_clean,pair_clean


,expiry,dte_calendar,strike,log_moneyness,moneyness_bucket_n10,selected_iv,total_variance,touched_by_convexity_violation,touched_by_monotonicity_violation,touched_by_calendar_violation,touched_by_any_static_violation,touched_by_high_iv_slope,touched_by_high_iv_jump_vs_uncertainty,touched_by_high_iv_curvature,touched_by_high_local_iv_range_vs_uncertainty,touched_by_any_iv_roughness_flag,diagnostic_weight
0,2026-07-10 00:00:00+00:00,5.00000000,658.00000000,-0.12421955,put_wing,0.40456494,0.00224209,False,False,False,False,False,False,False,False,False,0.28430707
1,2026-07-10 00:00:00+00:00,5.00000000,659.00000000,-0.12270095,put_wing,0.40010994,0.00219299,False,False,False,False,False,False,False,False,False,0.28566013
2,2026-07-10 00:00:00+00:00,5.00000000,660.00000000,-0.12118465,put_wing,0.39565753,0.00214445,False,False,False,False,False,False,False,False,False,0.28703299
3,2026-07-10 00:00:00+00:00,5.00000000,661.00000000,-0.11967065,put_wing,0.39120764,0.00209649,False,False,False,False,False,False,False,False,False,0.28842617
4,2026-07-10 00:00:00+00:00,5.00000000,662.00000000,-0.11815893,put_wing,0.38676019,0.00204909,False,False,False,False,False,False,False,False,False,0.28984024


In [10]:
# ============================================================
# Integrated static-arbitrage severity scoring
# ============================================================
# This cell combines the raw diagnostics into one observation-level
# and expiry-level severity layer.
#
# The scoring is deliberately diagnostic, not corrective.
#
# It does NOT:
#   - repair the surface;
#   - smooth IVs;
#   - fit SVI/SSVI;
#   - delete observations automatically;
#   - claim that remaining observations are globally arbitrage-free.
#
# It DOES:
#   - preserve all raw observations;
#   - combine pointwise, monotonicity, convexity, and calendar flags;
#   - distinguish hard price-space violations from IV roughness context;
#   - create broad / strict suitability flags for Notebook 11.


surface_scored = surface_iv_context.copy()


# ------------------------------------------------------------
# Defensive input checks
# ------------------------------------------------------------

required_scoring_cols = [
    "n10_row_id",
    "expiry",
    "expiry_rank",
    "dte_calendar",
    "tau_years",
    "strike",
    "forward",
    "log_moneyness",
    "abs_log_moneyness",
    "moneyness_bucket_n10",
    "maturity_bucket",
    "selected_iv",
    "total_variance",
    "diagnostic_weight",
    "quality_score",
    "relative_price_spread",
    "relative_iv_uncertainty_width",
    "touched_by_convexity_violation",
    "touched_by_monotonicity_violation",
    "touched_by_calendar_violation",
    "touched_by_any_static_violation",
    "touched_by_any_iv_roughness_flag",
]

missing_scoring_cols = [col for col in required_scoring_cols if col not in surface_scored.columns]

if missing_scoring_cols:
    raise ValueError(f"Missing required scoring columns: {missing_scoring_cols}")


# ------------------------------------------------------------
# Severity ranking helpers
# ------------------------------------------------------------

STATIC_DIAGNOSTIC_FAMILIES = [
    "pointwise_bounds",
    "strike_monotonicity",
    "butterfly_convexity",
    "calendar_total_variance",
]

STATIC_SEVERITY_RANK = {
    "clean": 0,
    "micro_noise": 1,
    "mild_warning": 2,
    "moderate_warning": 3,
    "severe_violation": 4,
}

STATIC_SEVERITY_LABEL = {
    0: "clean",
    1: "micro_noise",
    2: "mild_warning",
    3: "moderate_warning",
    4: "severe_violation",
}


def severity_rank_from_amount(amount: float) -> int:
    """
    Convert a positive violation amount into an ordered severity rank.
    """
    if not np.isfinite(amount) or amount <= 0:
        return 0

    if amount <= TOL.mild_violation_threshold:
        return 1

    if amount <= TOL.moderate_violation_threshold:
        return 2

    if amount <= TOL.severe_violation_threshold:
        return 3

    return 4


def severity_label_from_rank(rank: int) -> str:
    """
    Convert severity rank to canonical label.
    """
    return STATIC_SEVERITY_LABEL.get(int(rank), "unknown")


def max_or_zero(series: pd.Series) -> float:
    """
    Safe max aggregator that returns zero for empty / all-NaN inputs.
    """
    if len(series) == 0:
        return 0.0

    value = pd.to_numeric(series, errors="coerce").max()

    if not np.isfinite(value):
        return 0.0

    return float(value)


# ------------------------------------------------------------
# Row-level maxima from each diagnostic family
# ------------------------------------------------------------
# The previous cells attached most row-level flags directly.
# This section makes those flags explicit, uniform, and easier to audit.


# ---- Pointwise bounds ------------------------------------------------
# Pointwise violations were zero in the current run, but the code remains
# general in case future snapshots contain violations.

surface_scored["pointwise_violation_count"] = 0
surface_scored["max_pointwise_violation_amount"] = 0.0

if "pointwise_violation_ledger" in globals() and len(pointwise_violation_ledger) > 0:
    pointwise_row_candidates = []

    pointwise_id_cols = [
        col for col in [
            "n10_row_id",
            "row_id",
            "surface_row_id",
            "handoff_row_id",
        ]
        if col in pointwise_violation_ledger.columns
    ]

    if pointwise_id_cols:
        pointwise_id_col = pointwise_id_cols[0]

        amount_cols = [
            col for col in [
                "violation_amount",
                "max_violation_amount",
                "market_selected_bound_violation_amount",
            ]
            if col in pointwise_violation_ledger.columns
        ]

        if amount_cols:
            amount_col = amount_cols[0]
            pointwise_row_summary = (
                pointwise_violation_ledger
                .groupby(pointwise_id_col)
                .agg(
                    pointwise_violation_count=(amount_col, "size"),
                    max_pointwise_violation_amount=(amount_col, "max"),
                )
                .reset_index()
                .rename(columns={pointwise_id_col: "n10_row_id"})
            )

            surface_scored = surface_scored.drop(
                columns=["pointwise_violation_count", "max_pointwise_violation_amount"],
                errors="ignore",
            ).merge(
                pointwise_row_summary,
                on="n10_row_id",
                how="left",
            )

            surface_scored["pointwise_violation_count"] = (
                surface_scored["pointwise_violation_count"].fillna(0).astype(int)
            )
            surface_scored["max_pointwise_violation_amount"] = (
                surface_scored["max_pointwise_violation_amount"].fillna(0.0)
            )


# ---- Strike monotonicity ---------------------------------------------

surface_scored["monotonicity_violation_count"] = 0
surface_scored["max_monotonicity_violation_amount"] = 0.0

monotonicity_count_cols = [
    col for col in surface_scored.columns
    if col.endswith("_monotonicity_violation_pair_count")
]

if monotonicity_count_cols:
    surface_scored["monotonicity_violation_count"] = (
        surface_scored[monotonicity_count_cols].sum(axis=1).fillna(0).astype(int)
    )

if "monotonicity_violation_ledger" in globals() and len(monotonicity_violation_ledger) > 0:
    mono_endpoint_frames = []

    mono_amount_col = (
        "violation_amount"
        if "violation_amount" in monotonicity_violation_ledger.columns
        else None
    )

    if mono_amount_col is not None:
        for side in ["left", "right"]:
            row_col = f"{side}_n10_row_id"
            if row_col in monotonicity_violation_ledger.columns:
                mono_endpoint_frames.append(
                    monotonicity_violation_ledger[[row_col, mono_amount_col]]
                    .rename(columns={row_col: "n10_row_id", mono_amount_col: "violation_amount"})
                )

        if mono_endpoint_frames:
            mono_endpoint_df = pd.concat(mono_endpoint_frames, axis=0, ignore_index=True)

            mono_row_summary = (
                mono_endpoint_df
                .groupby("n10_row_id")
                .agg(
                    monotonicity_violation_count=("violation_amount", "size"),
                    max_monotonicity_violation_amount=("violation_amount", "max"),
                )
                .reset_index()
            )

            surface_scored = surface_scored.drop(
                columns=["monotonicity_violation_count", "max_monotonicity_violation_amount"],
                errors="ignore",
            ).merge(
                mono_row_summary,
                on="n10_row_id",
                how="left",
            )

            surface_scored["monotonicity_violation_count"] = (
                surface_scored["monotonicity_violation_count"].fillna(0).astype(int)
            )
            surface_scored["max_monotonicity_violation_amount"] = (
                surface_scored["max_monotonicity_violation_amount"].fillna(0.0)
            )


# ---- Butterfly convexity ---------------------------------------------

surface_scored["convexity_violation_count"] = 0
surface_scored["max_convexity_violation_amount"] = 0.0
surface_scored["max_negative_density_proxy_amount"] = 0.0

convexity_count_cols = [
    col for col in surface_scored.columns
    if col.endswith("_convexity_violation_triplet_count")
]

convexity_amount_cols = [
    col for col in surface_scored.columns
    if col.endswith("_max_convexity_violation_amount")
]

negative_density_cols = [
    col for col in surface_scored.columns
    if col.endswith("_max_negative_density_proxy_amount")
]

if convexity_count_cols:
    surface_scored["convexity_violation_count"] = (
        surface_scored[convexity_count_cols].sum(axis=1).fillna(0).astype(int)
    )

if convexity_amount_cols:
    surface_scored["max_convexity_violation_amount"] = (
        surface_scored[convexity_amount_cols].max(axis=1).fillna(0.0)
    )

if negative_density_cols:
    surface_scored["max_negative_density_proxy_amount"] = (
        surface_scored[negative_density_cols].max(axis=1).fillna(0.0)
    )

if "convexity_violation_ledger" in globals() and len(convexity_violation_ledger) > 0:
    convexity_endpoint_frames = []

    for side in ["left", "mid", "right"]:
        row_col = f"{side}_n10_row_id"
        if row_col in convexity_violation_ledger.columns:
            convexity_endpoint_frames.append(
                convexity_violation_ledger[
                    [
                        row_col,
                        "violation_amount",
                        "negative_density_proxy_amount",
                    ]
                ].rename(
                    columns={
                        row_col: "n10_row_id",
                        "violation_amount": "violation_amount",
                        "negative_density_proxy_amount": "negative_density_proxy_amount",
                    }
                )
            )

    if convexity_endpoint_frames:
        convexity_endpoint_df = pd.concat(convexity_endpoint_frames, axis=0, ignore_index=True)

        convexity_row_summary = (
            convexity_endpoint_df
            .groupby("n10_row_id")
            .agg(
                convexity_violation_count=("violation_amount", "size"),
                max_convexity_violation_amount=("violation_amount", "max"),
                max_negative_density_proxy_amount=("negative_density_proxy_amount", "max"),
            )
            .reset_index()
        )

        surface_scored = surface_scored.drop(
            columns=[
                "convexity_violation_count",
                "max_convexity_violation_amount",
                "max_negative_density_proxy_amount",
            ],
            errors="ignore",
        ).merge(
            convexity_row_summary,
            on="n10_row_id",
            how="left",
        )

        surface_scored["convexity_violation_count"] = (
            surface_scored["convexity_violation_count"].fillna(0).astype(int)
        )
        surface_scored["max_convexity_violation_amount"] = (
            surface_scored["max_convexity_violation_amount"].fillna(0.0)
        )
        surface_scored["max_negative_density_proxy_amount"] = (
            surface_scored["max_negative_density_proxy_amount"].fillna(0.0)
        )


# ---- Calendar total variance ----------------------------------------

surface_scored["calendar_violation_count"] = (
    surface_scored.get("calendar_nn_violation_pair_count", pd.Series(0, index=surface_scored.index))
    .fillna(0)
    .astype(int)
)

surface_scored["max_calendar_violation_amount"] = (
    surface_scored.get("max_calendar_violation_amount", pd.Series(0.0, index=surface_scored.index))
    .fillna(0.0)
)


# ------------------------------------------------------------
# Unified static-arbitrage counts and severity ranks
# ------------------------------------------------------------

surface_scored["total_static_violation_count"] = (
    surface_scored["pointwise_violation_count"]
    + surface_scored["monotonicity_violation_count"]
    + surface_scored["convexity_violation_count"]
    + surface_scored["calendar_violation_count"]
)

surface_scored["has_pointwise_violation"] = surface_scored["pointwise_violation_count"] > 0
surface_scored["has_monotonicity_violation"] = surface_scored["monotonicity_violation_count"] > 0
surface_scored["has_convexity_violation"] = surface_scored["convexity_violation_count"] > 0
surface_scored["has_calendar_violation"] = surface_scored["calendar_violation_count"] > 0

surface_scored["has_any_static_arbitrage_flag"] = (
    surface_scored["has_pointwise_violation"]
    | surface_scored["has_monotonicity_violation"]
    | surface_scored["has_convexity_violation"]
    | surface_scored["has_calendar_violation"]
)

surface_scored["pointwise_severity_rank"] = surface_scored["max_pointwise_violation_amount"].apply(severity_rank_from_amount)
surface_scored["monotonicity_severity_rank"] = surface_scored["max_monotonicity_violation_amount"].apply(severity_rank_from_amount)
surface_scored["convexity_severity_rank"] = surface_scored["max_convexity_violation_amount"].apply(severity_rank_from_amount)
surface_scored["calendar_severity_rank"] = surface_scored["max_calendar_violation_amount"].apply(severity_rank_from_amount)

surface_scored["max_static_severity_rank"] = surface_scored[
    [
        "pointwise_severity_rank",
        "monotonicity_severity_rank",
        "convexity_severity_rank",
        "calendar_severity_rank",
    ]
].max(axis=1)

surface_scored["max_static_severity"] = surface_scored["max_static_severity_rank"].apply(severity_label_from_rank)


# ------------------------------------------------------------
# Diagnostic score components
# ------------------------------------------------------------
# Score is a handoff suitability score, not a theoretical probability.
# Lower score means more suspicious raw row.
#
# The most important penalty is price-space static-arbitrage evidence.
# IV roughness is only a context penalty.

surface_scored["static_violation_penalty"] = (
    0.15 * surface_scored["has_pointwise_violation"].astype(float)
    + 0.15 * surface_scored["has_monotonicity_violation"].astype(float)
    + 0.35 * surface_scored["has_convexity_violation"].astype(float)
    + 0.25 * surface_scored["has_calendar_violation"].astype(float)
)

surface_scored["static_severity_penalty"] = (
    0.10 * surface_scored["max_static_severity_rank"].clip(lower=0, upper=4) / 4.0
)

surface_scored["iv_roughness_penalty"] = (
    0.05 * surface_scored["touched_by_any_iv_roughness_flag"].astype(float)
)

surface_scored["spread_context_penalty"] = (
    0.05
    * np.clip(
        safe_ratio(surface_scored["relative_price_spread"], 0.10),
        0.0,
        1.0,
    )
)

surface_scored["iv_uncertainty_context_penalty"] = (
    0.05
    * np.clip(
        safe_ratio(surface_scored["relative_iv_uncertainty_width"], 0.05),
        0.0,
        1.0,
    )
)

surface_scored["quality_context_penalty"] = (
    0.10
    * np.clip(
        1.0 - pd.to_numeric(surface_scored["quality_score"], errors="coerce").fillna(0.0),
        0.0,
        1.0,
    )
)

surface_scored["raw_static_arbitrage_score"] = (
    1.0
    - surface_scored[
        [
            "static_violation_penalty",
            "static_severity_penalty",
            "iv_roughness_penalty",
            "spread_context_penalty",
            "iv_uncertainty_context_penalty",
            "quality_context_penalty",
        ]
    ].sum(axis=1)
)

surface_scored["raw_static_arbitrage_score"] = surface_scored["raw_static_arbitrage_score"].clip(0.0, 1.0)


# ------------------------------------------------------------
# Handoff suitability tiers
# ------------------------------------------------------------
# Broad:
#   usable for exploratory repair / smoothing diagnostics.
#
# Strict:
#   cleaner subset for anchor points and less contaminated fitting experiments.
#
# Rejected:
#   kept in the ledger, but should not be treated as clean input.

surface_scored["n11_broad_candidate"] = (
    ~surface_scored["has_pointwise_violation"]
    & ~surface_scored["has_monotonicity_violation"]
    & ~surface_scored["has_calendar_violation"]
    & surface_scored["raw_static_arbitrage_score"].ge(0.45)
)

surface_scored["n11_strict_candidate"] = (
    surface_scored["n11_broad_candidate"]
    & ~surface_scored["has_convexity_violation"]
    & ~surface_scored["touched_by_any_iv_roughness_flag"]
    & surface_scored["diagnostic_weight"].ge(0.50)
    & surface_scored["quality_score"].ge(0.80)
    & surface_scored["raw_static_arbitrage_score"].ge(0.70)
)

surface_scored["n11_anchor_candidate"] = (
    surface_scored["n11_strict_candidate"]
    & surface_scored["abs_log_moneyness"].le(TOL.near_atm_abs_log_moneyness)
    & surface_scored["diagnostic_weight"].ge(0.90)
    & surface_scored["quality_score"].ge(0.90)
)

surface_scored["n11_exclusion_reason"] = "included_broad"

surface_scored.loc[
    surface_scored["has_pointwise_violation"],
    "n11_exclusion_reason",
] = "pointwise_bound_violation"

surface_scored.loc[
    ~surface_scored["has_pointwise_violation"] & surface_scored["has_monotonicity_violation"],
    "n11_exclusion_reason",
] = "strike_monotonicity_violation"

surface_scored.loc[
    ~surface_scored["has_pointwise_violation"]
    & ~surface_scored["has_monotonicity_violation"]
    & surface_scored["has_calendar_violation"],
    "n11_exclusion_reason",
] = "calendar_total_variance_violation"

surface_scored.loc[
    surface_scored["n11_broad_candidate"] & surface_scored["has_convexity_violation"],
    "n11_exclusion_reason",
] = "included_broad_convexity_warning"

surface_scored.loc[
    surface_scored["n11_broad_candidate"]
    & ~surface_scored["has_convexity_violation"]
    & surface_scored["touched_by_any_iv_roughness_flag"],
    "n11_exclusion_reason",
] = "included_broad_iv_roughness_warning"

surface_scored.loc[
    ~surface_scored["n11_broad_candidate"]
    & surface_scored["n11_exclusion_reason"].eq("included_broad"),
    "n11_exclusion_reason",
] = "low_raw_static_arbitrage_score"

surface_scored.loc[
    surface_scored["n11_strict_candidate"],
    "n11_exclusion_reason",
] = "included_strict"

surface_scored.loc[
    surface_scored["n11_anchor_candidate"],
    "n11_exclusion_reason",
] = "included_anchor"


# ------------------------------------------------------------
# Observation-level diagnostic panel for downstream use
# ------------------------------------------------------------

static_diagnostic_panel_cols = [
    "n10_row_id",
    "expiry",
    "expiry_rank",
    "dte_calendar",
    "tau_years",
    "maturity_bucket",
    "strike",
    "forward",
    "log_moneyness",
    "abs_log_moneyness",
    "moneyness_bucket_n10",
    "selected_iv",
    "total_variance",
    "diagnostic_weight",
    "quality_score",
    "relative_price_spread",
    "relative_iv_uncertainty_width",
    "pointwise_violation_count",
    "monotonicity_violation_count",
    "convexity_violation_count",
    "calendar_violation_count",
    "total_static_violation_count",
    "max_pointwise_violation_amount",
    "max_monotonicity_violation_amount",
    "max_convexity_violation_amount",
    "max_calendar_violation_amount",
    "max_negative_density_proxy_amount",
    "has_pointwise_violation",
    "has_monotonicity_violation",
    "has_convexity_violation",
    "has_calendar_violation",
    "has_any_static_arbitrage_flag",
    "max_static_severity",
    "max_static_severity_rank",
    "touched_by_any_iv_roughness_flag",
    "touched_by_high_iv_slope",
    "touched_by_high_iv_jump_vs_uncertainty",
    "touched_by_high_iv_curvature",
    "touched_by_high_local_iv_range_vs_uncertainty",
    "raw_static_arbitrage_score",
    "n11_broad_candidate",
    "n11_strict_candidate",
    "n11_anchor_candidate",
    "n11_exclusion_reason",
]

static_diagnostic_panel_cols = [
    col for col in static_diagnostic_panel_cols
    if col in surface_scored.columns
]

static_diagnostic_panel = surface_scored[static_diagnostic_panel_cols].copy()


# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

static_family_summary = pd.DataFrame(
    [
        {
            "diagnostic_family": "pointwise_bounds",
            "rows_touched": int(surface_scored["has_pointwise_violation"].sum()),
            "max_violation_amount": max_or_zero(surface_scored["max_pointwise_violation_amount"]),
            "max_severity_rank": int(surface_scored["pointwise_severity_rank"].max()),
            "max_severity": severity_label_from_rank(int(surface_scored["pointwise_severity_rank"].max())),
        },
        {
            "diagnostic_family": "strike_monotonicity",
            "rows_touched": int(surface_scored["has_monotonicity_violation"].sum()),
            "max_violation_amount": max_or_zero(surface_scored["max_monotonicity_violation_amount"]),
            "max_severity_rank": int(surface_scored["monotonicity_severity_rank"].max()),
            "max_severity": severity_label_from_rank(int(surface_scored["monotonicity_severity_rank"].max())),
        },
        {
            "diagnostic_family": "butterfly_convexity",
            "rows_touched": int(surface_scored["has_convexity_violation"].sum()),
            "max_violation_amount": max_or_zero(surface_scored["max_convexity_violation_amount"]),
            "max_severity_rank": int(surface_scored["convexity_severity_rank"].max()),
            "max_severity": severity_label_from_rank(int(surface_scored["convexity_severity_rank"].max())),
        },
        {
            "diagnostic_family": "calendar_total_variance",
            "rows_touched": int(surface_scored["has_calendar_violation"].sum()),
            "max_violation_amount": max_or_zero(surface_scored["max_calendar_violation_amount"]),
            "max_severity_rank": int(surface_scored["calendar_severity_rank"].max()),
            "max_severity": severity_label_from_rank(int(surface_scored["calendar_severity_rank"].max())),
        },
    ]
)

static_score_summary = pd.DataFrame(
    [
        {
            "rows": int(len(surface_scored)),
            "rows_with_any_static_flag": int(surface_scored["has_any_static_arbitrage_flag"].sum()),
            "rows_with_iv_roughness_context": int(surface_scored["touched_by_any_iv_roughness_flag"].sum()),
            "broad_candidates": int(surface_scored["n11_broad_candidate"].sum()),
            "strict_candidates": int(surface_scored["n11_strict_candidate"].sum()),
            "anchor_candidates": int(surface_scored["n11_anchor_candidate"].sum()),
            "median_raw_static_arbitrage_score": float(surface_scored["raw_static_arbitrage_score"].median()),
            "min_raw_static_arbitrage_score": float(surface_scored["raw_static_arbitrage_score"].min()),
            "max_raw_static_arbitrage_score": float(surface_scored["raw_static_arbitrage_score"].max()),
            "share_broad_candidates": float(surface_scored["n11_broad_candidate"].mean()),
            "share_strict_candidates": float(surface_scored["n11_strict_candidate"].mean()),
            "share_anchor_candidates": float(surface_scored["n11_anchor_candidate"].mean()),
        }
    ]
)

expiry_static_quality_summary = (
    surface_scored
    .groupby(["expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        rows=("n10_row_id", "size"),
        pointwise_flagged_rows=("has_pointwise_violation", "sum"),
        monotonicity_flagged_rows=("has_monotonicity_violation", "sum"),
        convexity_flagged_rows=("has_convexity_violation", "sum"),
        calendar_flagged_rows=("has_calendar_violation", "sum"),
        any_static_flagged_rows=("has_any_static_arbitrage_flag", "sum"),
        iv_roughness_context_rows=("touched_by_any_iv_roughness_flag", "sum"),
        max_convexity_violation_amount=("max_convexity_violation_amount", "max"),
        max_calendar_violation_amount=("max_calendar_violation_amount", "max"),
        max_static_severity_rank=("max_static_severity_rank", "max"),
        median_raw_static_arbitrage_score=("raw_static_arbitrage_score", "median"),
        min_raw_static_arbitrage_score=("raw_static_arbitrage_score", "min"),
        median_diagnostic_weight=("diagnostic_weight", "median"),
        median_quality_score=("quality_score", "median"),
        broad_candidates=("n11_broad_candidate", "sum"),
        strict_candidates=("n11_strict_candidate", "sum"),
        anchor_candidates=("n11_anchor_candidate", "sum"),
    )
    .reset_index()
    .sort_values(["expiry_rank", "expiry"])
)

expiry_static_quality_summary["any_static_flag_rate"] = (
    expiry_static_quality_summary["any_static_flagged_rows"]
    / expiry_static_quality_summary["rows"].clip(lower=1)
)

expiry_static_quality_summary["strict_candidate_rate"] = (
    expiry_static_quality_summary["strict_candidates"]
    / expiry_static_quality_summary["rows"].clip(lower=1)
)

expiry_static_quality_summary["max_static_severity"] = (
    expiry_static_quality_summary["max_static_severity_rank"].apply(severity_label_from_rank)
)

bucket_static_quality_summary = (
    surface_scored
    .groupby(["maturity_bucket", "moneyness_bucket_n10"], dropna=False)
    .agg(
        rows=("n10_row_id", "size"),
        pointwise_flagged_rows=("has_pointwise_violation", "sum"),
        monotonicity_flagged_rows=("has_monotonicity_violation", "sum"),
        convexity_flagged_rows=("has_convexity_violation", "sum"),
        calendar_flagged_rows=("has_calendar_violation", "sum"),
        any_static_flagged_rows=("has_any_static_arbitrage_flag", "sum"),
        iv_roughness_context_rows=("touched_by_any_iv_roughness_flag", "sum"),
        max_convexity_violation_amount=("max_convexity_violation_amount", "max"),
        max_static_severity_rank=("max_static_severity_rank", "max"),
        median_raw_static_arbitrage_score=("raw_static_arbitrage_score", "median"),
        min_raw_static_arbitrage_score=("raw_static_arbitrage_score", "min"),
        median_diagnostic_weight=("diagnostic_weight", "median"),
        median_quality_score=("quality_score", "median"),
        broad_candidates=("n11_broad_candidate", "sum"),
        strict_candidates=("n11_strict_candidate", "sum"),
        anchor_candidates=("n11_anchor_candidate", "sum"),
    )
    .reset_index()
    .sort_values(["maturity_bucket", "moneyness_bucket_n10"])
)

bucket_static_quality_summary["any_static_flag_rate"] = (
    bucket_static_quality_summary["any_static_flagged_rows"]
    / bucket_static_quality_summary["rows"].clip(lower=1)
)

bucket_static_quality_summary["strict_candidate_rate"] = (
    bucket_static_quality_summary["strict_candidates"]
    / bucket_static_quality_summary["rows"].clip(lower=1)
)

bucket_static_quality_summary["max_static_severity"] = (
    bucket_static_quality_summary["max_static_severity_rank"].apply(severity_label_from_rank)
)

exclusion_reason_summary = (
    surface_scored
    .groupby("n11_exclusion_reason", dropna=False)
    .agg(
        rows=("n10_row_id", "size"),
        median_raw_static_arbitrage_score=("raw_static_arbitrage_score", "median"),
        min_raw_static_arbitrage_score=("raw_static_arbitrage_score", "min"),
        max_static_severity_rank=("max_static_severity_rank", "max"),
        median_diagnostic_weight=("diagnostic_weight", "median"),
        median_quality_score=("quality_score", "median"),
    )
    .reset_index()
    .sort_values(["rows", "n11_exclusion_reason"], ascending=[False, True])
)

exclusion_reason_summary["share"] = (
    exclusion_reason_summary["rows"] / max(len(surface_scored), 1)
)

exclusion_reason_summary["max_static_severity"] = (
    exclusion_reason_summary["max_static_severity_rank"].apply(severity_label_from_rank)
)


# ------------------------------------------------------------
# Severity-ranked observation ledger
# ------------------------------------------------------------

static_severity_ledger = (
    static_diagnostic_panel
    .loc[
        static_diagnostic_panel["has_any_static_arbitrage_flag"]
        | static_diagnostic_panel["touched_by_any_iv_roughness_flag"]
    ]
    .sort_values(
        [
            "max_static_severity_rank",
            "total_static_violation_count",
            "max_convexity_violation_amount",
            "raw_static_arbitrage_score",
            "expiry_rank",
            "abs_log_moneyness",
        ],
        ascending=[False, False, False, True, True, True],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validation ledger
# ------------------------------------------------------------

n_rows = len(surface_scored)
n_broad = int(surface_scored["n11_broad_candidate"].sum())
n_strict = int(surface_scored["n11_strict_candidate"].sum())
n_anchor = int(surface_scored["n11_anchor_candidate"].sum())
n_any_static = int(surface_scored["has_any_static_arbitrage_flag"].sum())

scoring_validation_rows = [
    validation_row(
        "static_scoring_panel_created",
        len(static_diagnostic_panel) == n_rows,
        f"{len(static_diagnostic_panel):,} scored rows",
    ),
    validation_row(
        "static_family_summary_created",
        len(static_family_summary) == len(STATIC_DIAGNOSTIC_FAMILIES),
        f"{len(static_family_summary):,} diagnostic families summarized",
    ),
    validation_row(
        "severity_ranks_are_valid",
        surface_scored["max_static_severity_rank"].between(0, 4).all(),
        "all row-level severity ranks are in [0, 4]",
    ),
    validation_row(
        "raw_static_arbitrage_scores_are_bounded",
        surface_scored["raw_static_arbitrage_score"].between(0, 1).all(),
        "all raw static-arbitrage scores are in [0, 1]",
    ),
    validation_row(
        "broad_candidate_set_non_empty",
        n_broad > 0,
        f"{n_broad:,} broad Notebook 11 candidate rows",
    ),
    validation_row(
        "strict_candidate_set_non_empty",
        n_strict > 0,
        f"{n_strict:,} strict Notebook 11 candidate rows",
    ),
    validation_row(
        "anchor_candidate_set_non_empty",
        n_anchor > 0,
        f"{n_anchor:,} anchor candidate rows",
    ),
    validation_row(
        "static_violation_rows_preserved",
        n_any_static == int(surface_scored["touched_by_any_static_violation"].sum()),
        f"{n_any_static:,} rows have integrated static-arbitrage flags",
    ),
]

scoring_validation_ledger = pd.DataFrame(scoring_validation_rows)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("Integrated static-arbitrage scoring validation ledger")
print("=" * 90)
display(scoring_validation_ledger)

print("=" * 90)
print("Static diagnostic family summary")
print("=" * 90)
display(static_family_summary)

print("=" * 90)
print("Static score summary")
print("=" * 90)
display(static_score_summary)

print("=" * 90)
print("Expiry-level static quality summary")
print("=" * 90)
display(expiry_static_quality_summary)

print("=" * 90)
print("Maturity / moneyness static quality summary")
print("=" * 90)
display(bucket_static_quality_summary)

print("=" * 90)
print("Notebook 11 inclusion / exclusion reason summary")
print("=" * 90)
display(exclusion_reason_summary)

print("=" * 90)
print("Severity-ranked static diagnostic ledger")
print("=" * 90)

if len(static_severity_ledger) == 0:
    print("No static-arbitrage or IV-roughness flags detected.")
else:
    display(static_severity_ledger.head(75))


RUN_METADATA["scoring_validation"] = scoring_validation_ledger.to_dict(orient="records")
RUN_METADATA["static_family_summary"] = static_family_summary.to_dict(orient="records")
RUN_METADATA["static_score_summary"] = static_score_summary.to_dict(orient="records")
RUN_METADATA["expiry_static_quality_summary"] = expiry_static_quality_summary.to_dict(orient="records")
RUN_METADATA["exclusion_reason_summary"] = exclusion_reason_summary.to_dict(orient="records")

static_diagnostic_panel.head()

Integrated static-arbitrage scoring validation ledger


,check,passed,details
0,static_scoring_panel_created,True,"1,333 scored rows"
1,static_family_summary_created,True,4 diagnostic families summarized
2,severity_ranks_are_valid,True,"all row-level severity ranks are in [0, 4]"
3,raw_static_arbitrage_scores_are_bounded,True,"all raw static-arbitrage scores are in [0, 1]"
4,broad_candidate_set_non_empty,True,"1,260 broad Notebook 11 candidate rows"
5,strict_candidate_set_non_empty,True,224 strict Notebook 11 candidate rows
6,anchor_candidate_set_non_empty,True,196 anchor candidate rows
7,static_violation_rows_preserved,True,392 rows have integrated static-arbitrage flags


Static diagnostic family summary


,diagnostic_family,rows_touched,max_violation_amount,max_severity_rank,max_severity
0,pointwise_bounds,0,0.00000000,0,clean
1,strike_monotonicity,0,0.00000000,0,clean
2,butterfly_convexity,392,0.01611432,4,severe_violation
3,calendar_total_variance,0,0.00000000,0,clean


Static score summary


...

Notebook 11 inclusion / exclusion reason summary


,n11_exclusion_reason,rows,median_raw_static_arbitrage_score,min_raw_static_arbitrage_score,max_static_severity_rank,median_diagnostic_weight,median_quality_score,share,max_static_severity
1,included_broad,591,0.96844432,0.88781121,0,1.00000000,0.79123350,0.44336084,clean
2,included_broad_convexity_warning,319,0.49398710,0.45025428,4,0.84814424,0.79256408,0.23930983,severe_violation
0,included_anchor,196,0.99094693,0.96434274,0,1.00000000,0.99237045,0.14703676,clean
3,included_broad_iv_roughness_warning,126,0.91951352,0.84055610,0,0.65823404,0.79061707,0.09452363,clean
5,low_raw_static_arbitrage_score,73,0.41654677,0.37025770,4,0.46036629,0.74089107,0.05476369,severe_violation
4,included_strict,28,0.98656981,0.97213654,0,0.99534413,0.98312282,0.02100525,clean


Severity-ranked static diagnostic ledger


,n10_row_id,expiry,expiry_rank,dte_calendar,tau_years,maturity_bucket,strike,forward,log_moneyness,abs_log_moneyness,moneyness_bucket_n10,selected_iv,total_variance,diagnostic_weight,quality_score,relative_price_spread,relative_iv_uncertainty_width,pointwise_violation_count,monotonicity_violation_count,convexity_violation_count,calendar_violation_count,total_static_violation_count,max_pointwise_violation_amount,max_monotonicity_violation_amount,max_convexity_violation_amount,max_calendar_violation_amount,max_negative_density_proxy_amount,has_pointwise_violation,has_monotonicity_violation,has_convexity_violation,has_calendar_violation,has_any_static_arbitrage_flag,max_static_severity,max_static_severity_rank,touched_by_any_iv_roughness_flag,touched_by_high_iv_slope,touched_by_high_iv_jump_vs_uncertainty,touched_by_high_iv_curvature,touched_by_high_local_iv_range_vs_uncertainty,raw_static_arbitrage_score,n11_broad_candidate,n11_strict_candidate,n11_anchor_candidate,n11_exclusion_reason
0,705,2026-08-07 00:00:00+00:00,4,33.00000000,0.09041096,front_intermediate,741.00000000,747.33000000,-0.00850623,0.00850623,atm,0.14881934,0.00200235,0.95543873,0.70257705,0.14560770,0.18632154,0,0,8,0,8,0.00000000,0.00000000,0.01111681,0.00000000,0.02429876,False,False,True,False,True,severe_violation,4,True,True,False,True,False,0.37025770,False,False,False,low_raw_static_arbitrage_score
1,707,2026-08-07 00:00:00+00:00,4,33.00000000,0.09041096,front_intermediate,743.00000000,747.33000000,-0.00581081,0.00581081,atm,0.14720264,0.00195908,1.00000000,0.99393211,0.00542495,0.00461934,0,0,8,0,8,0.00000000,0.00000000,0.01104238,0.00000000,0.02429876,False,False,True,False,True,severe_violation,4,True,False,False,True,False,0.49206139,True,False,False,included_broad_convexity_warning
2,266,2026-07-17 00:00:00+00:00,1,12.00000000,0.03287671,short,781.00000000,745.51500000,0.04649990,0.04649990,near_atm_call,0.11114022,0.00040610,0.44867573,0.69714686,0.18181818,0.02343110,0,0,8,0,8,0.00000000,0.00000000,0.00489900,0.00000000,0.01000000,False,False,True,False,True,severe_violation,4,True,False,False,True,False,0.39628359,False,False,False,low_raw_static_arbitrage_score
3,431,2026-07-24 00:00:00+00:00,2,19.00000000,0.05205479,short,792.00000000,746.07250000,0.05973861,0.05973861,near_atm_call,0.11217701,0.00065504,0.37725618,0.71122246,0.15384615,0.01948977,0,0,8,0,8,0.00000000,0.00000000,0.00489900,0.00000000,0.01000000,False,False,True,False,True,severe_violation,4,True,True,False,True,False,0.40163247,False,False,False,low_raw_static_arbitrage_score
4,419,2026-07-24 00:00:00+00:00,2,19.00000000,0.05205479,short,780.00000000,746.07250000,0.04447114,0.04447114,near_atm_call,0.10575389,0.00058217,0.75613927,0.77319052,0.04255319,0.00746225,0,0,8,0,8,0.00000000,0.00000000,0.00489900,0.00000000,0.01000000,False,False,True,False,True,severe_violation,4,True,False,False,True,False,0.44858021,False,False,False,low_raw_static_arbitrage_score
5,579,2026-07-31 00:00:00+00:00,3,26.00000000,0.07123288,front_intermediate,750.00000000,746.59000000,0.00455703,0.00455703,atm,0.13720453,0.00134096,1.00000000,0.99453667,0.00473186,0.00554994,0,0,4,0,4,0.00000000,0.00000000,0.01611432,0.00000000,0.03408682,False,False,True,False,True,severe_violation,4,True,False,True,True,True,0.49153779,True,False,False,included_broad_convexity_warning
6,581,2026-07-31 00:00:00+00:00,3,26.00000000,0.07123288,front_intermediate,752.00000000,746.59000000,0.00722015,0.00722015,atm,0.13527030,0.00130342,1.00000000,0.99336055,0.00604230,0.00474258,0,0,4,0,4,0.00000000,0.00000000,0.01611432,0.00000000,0.03408682,False,False,True,False,True,severe_violation,4,True,False,False,True,False,0.49157232,True,False,False,included_broad_convexity_warning
7,580,2026-07-31 00:00:00+00:00,3,26.00000000,0.07123288,front_intermediate,751.00000000,746.59000000,0.00588948,0.00588948,atm,0.13655125,0.00132823,1.00000000,0.99371716,0.00568505,0.00466872,0,0,4,0,4,0.00000000,0.00000000,0.01611432,0.00000000,0.0

,n10_row_id,expiry,expiry_rank,dte_calendar,tau_years,maturity_bucket,strike,forward,log_moneyness,abs_log_moneyness,moneyness_bucket_n10,selected_iv,total_variance,diagnostic_weight,quality_score,relative_price_spread,relative_iv_uncertainty_width,pointwise_violation_count,monotonicity_violation_count,convexity_violation_count,calendar_violation_count,total_static_violation_count,max_pointwise_violation_amount,max_monotonicity_violation_amount,max_convexity_violation_amount,max_calendar_violation_amount,max_negative_density_proxy_amount,has_pointwise_violation,has_monotonicity_violation,has_convexity_violation,has_calendar_violation,has_any_static_arbitrage_flag,max_static_severity,max_static_severity_rank,touched_by_any_iv_roughness_flag,touched_by_high_iv_slope,touched_by_high_iv_jump_vs_uncertainty,touched_by_high_iv_curvature,touched_by_high_local_iv_range_vs_uncertainty,raw_static_arbitrage_score,n11_broad_candidate,n11_strict_candidate,n11_anchor_candidate,n11_exclusion_reason
0,0,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,658.00000000,745.03000000,-0.12421955,0.12421955,put_wing,0.40456494,0.00224209,0.28430707,0.67486370,0.22222222,0.02369035,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.89379602,True,False,False,included_broad
1,1,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,659.00000000,745.03000000,-0.12270095,0.12270095,put_wing,0.40010994,0.00219299,0.28566013,0.67490358,0.22222222,0.02373463,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.89375573,True,False,False,included_broad
2,2,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,660.00000000,745.03000000,-0.12118465,0.12118465,put_wing,0.39565753,0.00214445,0.28703299,0.67494351,0.22222222,0.02377958,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.89371477,True,False,False,included_broad
3,3,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,661.00000000,745.03000000,-0.11967065,0.11967065,put_wing,0.39120764,0.00209649,0.28842617,0.67498346,0.22222222,0.02382521,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.89367313,True,False,False,included_broad
4,4,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,662.00000000,745.03000000,-0.11815893,0.11815893,put_wing,0.38676019,0.00204909,0.28984024,0.67502346,0.22222222,0.02387155,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.89363079,True,False,False,included_broad


In [11]:
# ============================================================
# Notebook 11 handoff construction
# ============================================================
# This cell creates explicit downstream handoffs from the audited raw surface.
#
# Handoff philosophy:
#   - broad handoff: keeps usable raw observations, including convexity warnings;
#   - strict handoff: keeps only cleaner observations with no static flags / roughness flags;
#   - anchor handoff: high-quality near-ATM subset for stabilization checks;
#   - repair queue: observations requiring attention before arbitrage-free calibration;
#   - diagnostic ledger: full scored panel, including exclusions.
#
# No smoothing, fitting, or repair is performed here.


# ------------------------------------------------------------
# Handoff column design
# ------------------------------------------------------------

core_surface_cols = [
    "n10_row_id",
    "expiry",
    "expiry_rank",
    "dte_calendar",
    "tau_years",
    "maturity_bucket",
    "option_type",
    "strike",
    "spot",
    "forward",
    "discount_factor",
    "log_moneyness",
    "abs_log_moneyness",
    "moneyness_bucket_n10",
    "selected_iv",
    "total_variance",
    "market_selected_price",
    "market_call_equiv_price",
    "market_put_equiv_price",
    "bsm_selected_price",
    "bsm_call_equiv_price",
    "bsm_put_equiv_price",
]

quote_quality_cols = [
    "bid_price",
    "ask_price",
    "mid_price",
    "price_spread",
    "relative_price_spread",
    "iv_bid",
    "iv_ask",
    "iv_mid",
    "iv_spread",
    "relative_iv_uncertainty_width",
    "diagnostic_weight",
    "quality_score",
    "surface_row_source",
    "primary_pair_issue",
    "primary_stability_issue",
]

static_score_cols = [
    "pointwise_violation_count",
    "monotonicity_violation_count",
    "convexity_violation_count",
    "calendar_violation_count",
    "total_static_violation_count",
    "max_pointwise_violation_amount",
    "max_monotonicity_violation_amount",
    "max_convexity_violation_amount",
    "max_calendar_violation_amount",
    "max_negative_density_proxy_amount",
    "has_pointwise_violation",
    "has_monotonicity_violation",
    "has_convexity_violation",
    "has_calendar_violation",
    "has_any_static_arbitrage_flag",
    "max_static_severity",
    "max_static_severity_rank",
    "touched_by_any_iv_roughness_flag",
    "touched_by_high_iv_slope",
    "touched_by_high_iv_jump_vs_uncertainty",
    "touched_by_high_iv_curvature",
    "touched_by_high_local_iv_range_vs_uncertainty",
    "raw_static_arbitrage_score",
    "n11_broad_candidate",
    "n11_strict_candidate",
    "n11_anchor_candidate",
    "n11_exclusion_reason",
]

optional_context_cols = [
    "quote_date",
    "snapshot_date",
    "underlying_symbol",
    "contract_symbol",
    "source_row_id",
    "n09_row_id",
    "market_selected_bound_violation",
    "market_call_equiv_bound_violation",
    "market_put_equiv_bound_violation",
    "any_monotonicity_violation_touching_row",
    "any_convexity_violation_touching_row",
    "any_calendar_violation_touching_row",
    "calendar_nn_pair_count",
    "calendar_nn_violation_pair_count",
]

handoff_cols = [
    col for col in (
        core_surface_cols
        + quote_quality_cols
        + static_score_cols
        + optional_context_cols
    )
    if col in surface_scored.columns
]

if "n10_row_id" not in handoff_cols:
    raise ValueError("n10_row_id is required for Notebook 11 handoffs.")

if "selected_iv" not in handoff_cols or "total_variance" not in handoff_cols:
    raise ValueError("selected_iv and total_variance are required for Notebook 11 handoffs.")


# ------------------------------------------------------------
# Create handoff tables
# ------------------------------------------------------------

n10_full_static_diagnostic_handoff = (
    surface_scored[handoff_cols]
    .copy()
    .sort_values(["expiry_rank", "strike", "n10_row_id"])
    .reset_index(drop=True)
)

n10_to_n11_broad_handoff = (
    n10_full_static_diagnostic_handoff
    .loc[n10_full_static_diagnostic_handoff["n11_broad_candidate"]]
    .copy()
    .sort_values(["expiry_rank", "strike", "n10_row_id"])
    .reset_index(drop=True)
)

n10_to_n11_strict_handoff = (
    n10_full_static_diagnostic_handoff
    .loc[n10_full_static_diagnostic_handoff["n11_strict_candidate"]]
    .copy()
    .sort_values(["expiry_rank", "strike", "n10_row_id"])
    .reset_index(drop=True)
)

n10_to_n11_anchor_handoff = (
    n10_full_static_diagnostic_handoff
    .loc[n10_full_static_diagnostic_handoff["n11_anchor_candidate"]]
    .copy()
    .sort_values(["expiry_rank", "strike", "n10_row_id"])
    .reset_index(drop=True)
)

n10_to_n11_repair_queue = (
    n10_full_static_diagnostic_handoff
    .loc[
        n10_full_static_diagnostic_handoff["has_any_static_arbitrage_flag"]
        | n10_full_static_diagnostic_handoff["touched_by_any_iv_roughness_flag"]
        | n10_full_static_diagnostic_handoff["n11_exclusion_reason"].eq("low_raw_static_arbitrage_score")
    ]
    .copy()
    .sort_values(
        [
            "max_static_severity_rank",
            "total_static_violation_count",
            "max_convexity_violation_amount",
            "raw_static_arbitrage_score",
            "expiry_rank",
            "strike",
        ],
        ascending=[False, False, False, True, True, True],
    )
    .reset_index(drop=True)
)

n10_to_n11_excluded_handoff = (
    n10_full_static_diagnostic_handoff
    .loc[~n10_full_static_diagnostic_handoff["n11_broad_candidate"]]
    .copy()
    .sort_values(
        [
            "n11_exclusion_reason",
            "raw_static_arbitrage_score",
            "expiry_rank",
            "strike",
        ],
        ascending=[True, True, True, True],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Add explicit handoff labels
# ------------------------------------------------------------

def add_handoff_metadata(df: pd.DataFrame, handoff_name: str, handoff_role: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "handoff_name", handoff_name)
    out.insert(1, "handoff_role", handoff_role)
    out.insert(2, "source_notebook", NOTEBOOK_NAME)
    out.insert(3, "snapshot_tag", SNAPSHOT_TAG)
    out.insert(4, "created_utc", RUN_TIMESTAMP_UTC)
    return out


n10_full_static_diagnostic_handoff = add_handoff_metadata(
    n10_full_static_diagnostic_handoff,
    "n10_full_static_diagnostic_handoff",
    "complete scored raw surface; includes all rows",
)

n10_to_n11_broad_handoff = add_handoff_metadata(
    n10_to_n11_broad_handoff,
    "n10_to_n11_broad_handoff",
    "broad empirical surface input for Notebook 11; convexity warnings may remain",
)

n10_to_n11_strict_handoff = add_handoff_metadata(
    n10_to_n11_strict_handoff,
    "n10_to_n11_strict_handoff",
    "cleaner subset for first-pass arbitrage-aware fitting",
)

n10_to_n11_anchor_handoff = add_handoff_metadata(
    n10_to_n11_anchor_handoff,
    "n10_to_n11_anchor_handoff",
    "high-quality near-ATM anchor subset",
)

n10_to_n11_repair_queue = add_handoff_metadata(
    n10_to_n11_repair_queue,
    "n10_to_n11_repair_queue",
    "rows requiring repair, downweighting, or diagnostic attention",
)

n10_to_n11_excluded_handoff = add_handoff_metadata(
    n10_to_n11_excluded_handoff,
    "n10_to_n11_excluded_handoff",
    "rows not admitted to the broad Notebook 11 handoff",
)


# ------------------------------------------------------------
# Handoff summary tables
# ------------------------------------------------------------

handoff_inventory = pd.DataFrame(
    [
        {
            "handoff_name": "n10_full_static_diagnostic_handoff",
            "role": "complete scored raw surface",
            "rows": len(n10_full_static_diagnostic_handoff),
            "unique_expiries": n10_full_static_diagnostic_handoff["expiry"].nunique(),
            "min_dte_calendar": n10_full_static_diagnostic_handoff["dte_calendar"].min(),
            "max_dte_calendar": n10_full_static_diagnostic_handoff["dte_calendar"].max(),
            "median_score": n10_full_static_diagnostic_handoff["raw_static_arbitrage_score"].median(),
            "static_flagged_rows": n10_full_static_diagnostic_handoff["has_any_static_arbitrage_flag"].sum(),
            "iv_roughness_rows": n10_full_static_diagnostic_handoff["touched_by_any_iv_roughness_flag"].sum(),
        },
        {
            "handoff_name": "n10_to_n11_broad_handoff",
            "role": "broad Notebook 11 input",
            "rows": len(n10_to_n11_broad_handoff),
            "unique_expiries": n10_to_n11_broad_handoff["expiry"].nunique(),
            "min_dte_calendar": n10_to_n11_broad_handoff["dte_calendar"].min(),
            "max_dte_calendar": n10_to_n11_broad_handoff["dte_calendar"].max(),
            "median_score": n10_to_n11_broad_handoff["raw_static_arbitrage_score"].median(),
            "static_flagged_rows": n10_to_n11_broad_handoff["has_any_static_arbitrage_flag"].sum(),
            "iv_roughness_rows": n10_to_n11_broad_handoff["touched_by_any_iv_roughness_flag"].sum(),
        },
        {
            "handoff_name": "n10_to_n11_strict_handoff",
            "role": "strict Notebook 11 input",
            "rows": len(n10_to_n11_strict_handoff),
            "unique_expiries": n10_to_n11_strict_handoff["expiry"].nunique(),
            "min_dte_calendar": n10_to_n11_strict_handoff["dte_calendar"].min(),
            "max_dte_calendar": n10_to_n11_strict_handoff["dte_calendar"].max(),
            "median_score": n10_to_n11_strict_handoff["raw_static_arbitrage_score"].median(),
            "static_flagged_rows": n10_to_n11_strict_handoff["has_any_static_arbitrage_flag"].sum(),
            "iv_roughness_rows": n10_to_n11_strict_handoff["touched_by_any_iv_roughness_flag"].sum(),
        },
        {
            "handoff_name": "n10_to_n11_anchor_handoff",
            "role": "near-ATM anchor input",
            "rows": len(n10_to_n11_anchor_handoff),
            "unique_expiries": n10_to_n11_anchor_handoff["expiry"].nunique(),
            "min_dte_calendar": n10_to_n11_anchor_handoff["dte_calendar"].min(),
            "max_dte_calendar": n10_to_n11_anchor_handoff["dte_calendar"].max(),
            "median_score": n10_to_n11_anchor_handoff["raw_static_arbitrage_score"].median(),
            "static_flagged_rows": n10_to_n11_anchor_handoff["has_any_static_arbitrage_flag"].sum(),
            "iv_roughness_rows": n10_to_n11_anchor_handoff["touched_by_any_iv_roughness_flag"].sum(),
        },
        {
            "handoff_name": "n10_to_n11_repair_queue",
            "role": "repair / diagnostic queue",
            "rows": len(n10_to_n11_repair_queue),
            "unique_expiries": n10_to_n11_repair_queue["expiry"].nunique(),
            "min_dte_calendar": n10_to_n11_repair_queue["dte_calendar"].min(),
            "max_dte_calendar": n10_to_n11_repair_queue["dte_calendar"].max(),
            "median_score": n10_to_n11_repair_queue["raw_static_arbitrage_score"].median(),
            "static_flagged_rows": n10_to_n11_repair_queue["has_any_static_arbitrage_flag"].sum(),
            "iv_roughness_rows": n10_to_n11_repair_queue["touched_by_any_iv_roughness_flag"].sum(),
        },
        {
            "handoff_name": "n10_to_n11_excluded_handoff",
            "role": "excluded from broad input",
            "rows": len(n10_to_n11_excluded_handoff),
            "unique_expiries": n10_to_n11_excluded_handoff["expiry"].nunique(),
            "min_dte_calendar": n10_to_n11_excluded_handoff["dte_calendar"].min(),
            "max_dte_calendar": n10_to_n11_excluded_handoff["dte_calendar"].max(),
            "median_score": n10_to_n11_excluded_handoff["raw_static_arbitrage_score"].median(),
            "static_flagged_rows": n10_to_n11_excluded_handoff["has_any_static_arbitrage_flag"].sum(),
            "iv_roughness_rows": n10_to_n11_excluded_handoff["touched_by_any_iv_roughness_flag"].sum(),
        },
    ]
)

handoff_by_expiry = (
    n10_full_static_diagnostic_handoff
    .groupby(["expiry", "expiry_rank", "maturity_bucket"], dropna=False)
    .agg(
        dte_calendar=("dte_calendar", "median"),
        full_rows=("n10_row_id", "size"),
        broad_rows=("n11_broad_candidate", "sum"),
        strict_rows=("n11_strict_candidate", "sum"),
        anchor_rows=("n11_anchor_candidate", "sum"),
        static_flagged_rows=("has_any_static_arbitrage_flag", "sum"),
        iv_roughness_rows=("touched_by_any_iv_roughness_flag", "sum"),
        median_score=("raw_static_arbitrage_score", "median"),
        min_score=("raw_static_arbitrage_score", "min"),
        median_diagnostic_weight=("diagnostic_weight", "median"),
        median_quality_score=("quality_score", "median"),
    )
    .reset_index()
    .sort_values(["expiry_rank", "expiry"])
)

handoff_by_expiry["broad_share"] = handoff_by_expiry["broad_rows"] / handoff_by_expiry["full_rows"].clip(lower=1)
handoff_by_expiry["strict_share"] = handoff_by_expiry["strict_rows"] / handoff_by_expiry["full_rows"].clip(lower=1)
handoff_by_expiry["anchor_share"] = handoff_by_expiry["anchor_rows"] / handoff_by_expiry["full_rows"].clip(lower=1)

handoff_by_bucket = (
    n10_full_static_diagnostic_handoff
    .groupby(["maturity_bucket", "moneyness_bucket_n10"], dropna=False)
    .agg(
        full_rows=("n10_row_id", "size"),
        broad_rows=("n11_broad_candidate", "sum"),
        strict_rows=("n11_strict_candidate", "sum"),
        anchor_rows=("n11_anchor_candidate", "sum"),
        static_flagged_rows=("has_any_static_arbitrage_flag", "sum"),
        iv_roughness_rows=("touched_by_any_iv_roughness_flag", "sum"),
        median_score=("raw_static_arbitrage_score", "median"),
        min_score=("raw_static_arbitrage_score", "min"),
        median_diagnostic_weight=("diagnostic_weight", "median"),
        median_quality_score=("quality_score", "median"),
    )
    .reset_index()
    .sort_values(["maturity_bucket", "moneyness_bucket_n10"])
)

handoff_by_bucket["broad_share"] = handoff_by_bucket["broad_rows"] / handoff_by_bucket["full_rows"].clip(lower=1)
handoff_by_bucket["strict_share"] = handoff_by_bucket["strict_rows"] / handoff_by_bucket["full_rows"].clip(lower=1)
handoff_by_bucket["anchor_share"] = handoff_by_bucket["anchor_rows"] / handoff_by_bucket["full_rows"].clip(lower=1)


# ------------------------------------------------------------
# Persist handoffs
# ------------------------------------------------------------

handoff_objects = {
    "n10_full_static_diagnostic_handoff": n10_full_static_diagnostic_handoff,
    "n10_to_n11_broad_handoff": n10_to_n11_broad_handoff,
    "n10_to_n11_strict_handoff": n10_to_n11_strict_handoff,
    "n10_to_n11_anchor_handoff": n10_to_n11_anchor_handoff,
    "n10_to_n11_repair_queue": n10_to_n11_repair_queue,
    "n10_to_n11_excluded_handoff": n10_to_n11_excluded_handoff,
    "handoff_inventory": handoff_inventory,
    "handoff_by_expiry": handoff_by_expiry,
    "handoff_by_bucket": handoff_by_bucket,
}

handoff_file_manifest = []

for object_name, df in handoff_objects.items():
    parquet_path = HANDOFF_DIR / f"{SNAPSHOT_TAG}_{object_name}.parquet"
    csv_path = HANDOFF_DIR / f"{SNAPSHOT_TAG}_{object_name}.csv"

    latest_parquet_path = HANDOFF_DIR / f"latest_{object_name}.parquet"
    latest_csv_path = HANDOFF_DIR / f"latest_{object_name}.csv"

    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)

    df.to_parquet(latest_parquet_path, index=False)
    df.to_csv(latest_csv_path, index=False)

    handoff_file_manifest.extend(
        [
            {
                "object_name": object_name,
                "file_role": "versioned_parquet",
                "path": str(parquet_path),
                "rows": len(df),
                "columns": len(df.columns),
            },
            {
                "object_name": object_name,
                "file_role": "versioned_csv",
                "path": str(csv_path),
                "rows": len(df),
                "columns": len(df.columns),
            },
            {
                "object_name": object_name,
                "file_role": "latest_parquet",
                "path": str(latest_parquet_path),
                "rows": len(df),
                "columns": len(df.columns),
            },
            {
                "object_name": object_name,
                "file_role": "latest_csv",
                "path": str(latest_csv_path),
                "rows": len(df),
                "columns": len(df.columns),
            },
        ]
    )

handoff_file_manifest = pd.DataFrame(handoff_file_manifest)


# ------------------------------------------------------------
# Handoff validation ledger
# ------------------------------------------------------------

handoff_validation_ledger = pd.DataFrame(
    [
        validation_row(
            "full_handoff_preserves_all_rows",
            len(n10_full_static_diagnostic_handoff) == len(surface_scored),
            f"{len(n10_full_static_diagnostic_handoff):,} full rows vs {len(surface_scored):,} scored rows",
        ),
        validation_row(
            "broad_handoff_nonempty",
            len(n10_to_n11_broad_handoff) > 0,
            f"{len(n10_to_n11_broad_handoff):,} broad rows",
        ),
        validation_row(
            "strict_handoff_nonempty",
            len(n10_to_n11_strict_handoff) > 0,
            f"{len(n10_to_n11_strict_handoff):,} strict rows",
        ),
        validation_row(
            "anchor_handoff_nonempty",
            len(n10_to_n11_anchor_handoff) > 0,
            f"{len(n10_to_n11_anchor_handoff):,} anchor rows",
        ),
        validation_row(
            "strict_subset_of_broad",
            set(n10_to_n11_strict_handoff["n10_row_id"]).issubset(set(n10_to_n11_broad_handoff["n10_row_id"])),
            "strict handoff row IDs are contained in broad handoff",
        ),
        validation_row(
            "anchor_subset_of_strict",
            set(n10_to_n11_anchor_handoff["n10_row_id"]).issubset(set(n10_to_n11_strict_handoff["n10_row_id"])),
            "anchor handoff row IDs are contained in strict handoff",
        ),
        validation_row(
            "strict_handoff_has_no_static_flags",
            not n10_to_n11_strict_handoff["has_any_static_arbitrage_flag"].any(),
            "strict handoff has zero pointwise / monotonicity / convexity / calendar flags",
        ),
        validation_row(
            "strict_handoff_has_no_iv_roughness_flags",
            not n10_to_n11_strict_handoff["touched_by_any_iv_roughness_flag"].any(),
            "strict handoff has zero IV roughness context flags",
        ),
        validation_row(
            "broad_handoff_has_no_pointwise_monotonicity_calendar_flags",
            not (
                n10_to_n11_broad_handoff["has_pointwise_violation"].any()
                or n10_to_n11_broad_handoff["has_monotonicity_violation"].any()
                or n10_to_n11_broad_handoff["has_calendar_violation"].any()
            ),
            "broad handoff admits convexity warnings only; no pointwise, monotonicity, or calendar violations",
        ),
        validation_row(
            "all_expiries_represented_in_broad_handoff",
            n10_to_n11_broad_handoff["expiry"].nunique() == surface_scored["expiry"].nunique(),
            f"{n10_to_n11_broad_handoff['expiry'].nunique():,} broad expiries vs {surface_scored['expiry'].nunique():,} full expiries",
        ),
        validation_row(
            "all_expiries_represented_in_strict_handoff",
            n10_to_n11_strict_handoff["expiry"].nunique() == surface_scored["expiry"].nunique(),
            f"{n10_to_n11_strict_handoff['expiry'].nunique():,} strict expiries vs {surface_scored['expiry'].nunique():,} full expiries",
        ),
        validation_row(
            "handoff_files_written",
            len(handoff_file_manifest) == 4 * len(handoff_objects),
            f"{len(handoff_file_manifest):,} files recorded in handoff manifest",
        ),
    ]
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 90)
print("Notebook 11 handoff validation ledger")
print("=" * 90)
display(handoff_validation_ledger)

print("=" * 90)
print("Notebook 11 handoff inventory")
print("=" * 90)
display(handoff_inventory)

print("=" * 90)
print("Notebook 11 handoff summary by expiry")
print("=" * 90)
display(handoff_by_expiry)

print("=" * 90)
print("Notebook 11 handoff summary by maturity / moneyness bucket")
print("=" * 90)
display(handoff_by_bucket)

print("=" * 90)
print("Handoff file manifest")
print("=" * 90)
display(handoff_file_manifest)

print("=" * 90)
print("Broad handoff preview")
print("=" * 90)
display(n10_to_n11_broad_handoff.head())

print("=" * 90)
print("Strict handoff preview")
print("=" * 90)
display(n10_to_n11_strict_handoff.head())

print("=" * 90)
print("Repair queue preview")
print("=" * 90)
display(n10_to_n11_repair_queue.head(25))


RUN_METADATA["handoff_validation"] = handoff_validation_ledger.to_dict(orient="records")
RUN_METADATA["handoff_inventory"] = handoff_inventory.to_dict(orient="records")
RUN_METADATA["handoff_file_manifest"] = handoff_file_manifest.to_dict(orient="records")

handoff_inventory

Notebook 11 handoff validation ledger


,check,passed,details
0,full_handoff_preserves_all_rows,True,"1,333 full rows vs 1,333 scored rows"
1,broad_handoff_nonempty,True,"1,260 broad rows"
2,strict_handoff_nonempty,True,224 strict rows
3,anchor_handoff_nonempty,True,196 anchor rows
4,strict_subset_of_broad,True,strict handoff row IDs are contained in broad ...
5,anchor_subset_of_strict,True,anchor handoff row IDs are contained in strict...
6,strict_handoff_has_no_static_flags,True,strict handoff has zero pointwise / monotonici...
7,strict_handoff_has_no_iv_roughness_flags,True,strict handoff has zero IV roughness context f...
8,broad_handoff_has_no_pointwise_monotonicity_ca...,True,broad handoff admits convexity warnings only; ...
9,all_expiries_represented_in_broad_handoff,True,8 broad expiries vs 8 full expiries


Notebook 11 handoff inventory


,handoff_name,role,rows,unique_expiries,min_dte_calendar,max_dte_calendar,median_score,static_flagged_rows,iv_roughness_rows
0,n10_full_static_diagnostic_handoff,complete scored raw surface,1333,8,5.00000000,75.00000000,0.95922487,392,327
1,n10_to_n11_broad_handoff,broad Notebook 11 input,1260,8,5.00000000,75.00000000,0.96252924,319,258
2,n10_to_n11_strict_handoff,strict Notebook 11 input,224,8,5.00000000,75.00000000,0.99055544,0,0
3,n10_to_n11_anchor_handoff,near-ATM anchor input,196,8,5.00000000,75.00000000,0.99094693,0,0
4,n10_to_n11_repair_queue,repair / diagnostic queue,518,8,5.00000000,75.00000000,0.51113466,392,327
5,n10_to_n11_excluded_handoff,excluded from broad input,73,6,5.00000000,75.00000000,0.41654677,73,69


Notebook 11 handoff summary by expiry


...

Strict handoff preview


,handoff_name,handoff_role,source_notebook,snapshot_tag,created_utc,n10_row_id,expiry,expiry_rank,dte_calendar,tau_years,maturity_bucket,option_type,strike,spot,forward,discount_factor,log_moneyness,abs_log_moneyness,moneyness_bucket_n10,selected_iv,total_variance,market_selected_price,market_call_equiv_price,market_put_equiv_price,bsm_selected_price,bsm_call_equiv_price,bsm_put_equiv_price,price_spread,relative_price_spread,relative_iv_uncertainty_width,diagnostic_weight,quality_score,surface_row_source,primary_pair_issue,primary_stability_issue,pointwise_violation_count,monotonicity_violation_count,convexity_violation_count,calendar_violation_count,total_static_violation_count,max_pointwise_violation_amount,max_monotonicity_violation_amount,max_convexity_violation_amount,max_calendar_violation_amount,max_negative_density_proxy_amount,has_pointwise_violation,has_monotonicity_violation,has_convexity_violation,has_calendar_violation,has_any_static_arbitrage_flag,max_static_severity,max_static_severity_rank,touched_by_any_iv_roughness_flag,touched_by_high_iv_slope,touched_by_high_iv_jump_vs_uncertainty,touched_by_high_iv_curvature,touched_by_high_local_iv_range_vs_uncertainty,raw_static_arbitrage_score,n11_broad_candidate,n11_strict_candidate,n11_anchor_candidate,n11_exclusion_reason,source_row_id,market_selected_bound_violation,market_call_equiv_bound_violation,market_put_equiv_bound_violation,any_monotonicity_violation_touching_row,any_convexity_violation_touching_row,any_calendar_violation_touching_row,calendar_nn_pair_count,calendar_nn_violation_pair_count
0,n10_to_n11_strict_handoff,cleaner subset for first-pass arbitrage-aware ...,10_raw_static_arbitrage_diagnostics_for_spy_iv...,n10_20260705T200149Z,20260705T200149Z,63,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,put,721.00000000,744.78002930,745.03000000,0.99938375,-0.03278535,0.03278535,near_atm_put,0.19373475,0.00051415,0.55000000,24.56519155,0.55000000,0.55000000,24.56519155,0.55000000,0.02000000,0.03636364,0.00858617,1.00000000,0.96336608,matched_pair_preferred_side,pair_clean,stability_clean,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.96956862,True,True,True,included_anchor,994,False,False,False,False,False,False,1,0
1,n10_to_n11_strict_handoff,cleaner subset for first-pass arbitrage-aware ...,10_raw_static_arbitrage_diagnostics_for_spy_iv...,n10_20260705T200149Z,20260705T200149Z,65,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,put,723.00000000,744.78002930,745.03000000,0.99938375,-0.03001526,0.03001526,near_atm_put,0.18872688,0.00048792,0.65000000,22.66642405,0.65000000,0.65000000,22.66642405,0.65000000,0.02000000,0.03076923,0.00779028,1.00000000,0.96876750,matched_pair_preferred_side,pair_clean,stability_clean,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.97370186,True,True,True,included_anchor,996,False,False,False,False,False,False,1,0
2,n10_to_n11_strict_handoff,cleaner subset for first-pass arbitrage-aware ...,10_raw_static_arbitrage_diagnostics_for_spy_iv...,n10_20260705T200149Z,20260705T200149Z,70,2026-07-10 00:00:00+00:00,0,5.00000000,0.01369863,ultra_short,put,728.00000000,744.78002930,745.03000000,0.99938375,-0.02312344,0.02312344,atm,0.17682815,0.00042833,1.01000000,18.02950529,1.01000000,1.01000000,18.02950529,1.01000000,0.02000000,0.01980198,0.00614386,1.00000000,0.97953902,matched_pair_preferred_side,pair_clean,stability_clean,0,0,0,0,0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,False,False,False,False,False,clean,0,False,False,False,False,False,0.98190905,True,True,True,included_anchor,1001,False,False,False,False,False,False,1,0
3,n10_to_n11_strict_handoff,cleaner subset for first-pass arbitrage-aware ...,10_raw_static_arbitrage_diagnostics_for_spy_iv...,n10_20260705T200149Z,20260705T200149Z,71,2026-07-10 00:00:00+00:00,0,5.0000

Repair queue preview


,handoff_name,handoff_role,source_notebook,snapshot_tag,created_utc,n10_row_id,expiry,expiry_rank,dte_calendar,tau_years,maturity_bucket,option_type,strike,spot,forward,discount_factor,log_moneyness,abs_log_moneyness,moneyness_bucket_n10,selected_iv,total_variance,market_selected_price,market_call_equiv_price,market_put_equiv_price,bsm_selected_price,bsm_call_equiv_price,bsm_put_equiv_price,price_spread,relative_price_spread,relative_iv_uncertainty_width,diagnostic_weight,quality_score,surface_row_source,primary_pair_issue,primary_stability_issue,pointwise_violation_count,monotonicity_violation_count,convexity_violation_count,calendar_violation_count,total_static_violation_count,max_pointwise_violation_amount,max_monotonicity_violation_amount,max_convexity_violation_amount,max_calendar_violation_amount,max_negative_density_proxy_amount,has_pointwise_violation,has_monotonicity_violation,has_convexity_violation,has_calendar_violation,has_any_static_arbitrage_flag,max_static_severity,max_static_severity_rank,touched_by_any_iv_roughness_flag,touched_by_high_iv_slope,touched_by_high_iv_jump_vs_uncertainty,touched_by_high_iv_curvature,touched_by_high_local_iv_range_vs_uncertainty,raw_static_arbitrage_score,n11_broad_candidate,n11_strict_candidate,n11_anchor_candidate,n11_exclusion_reason,source_row_id,market_selected_bound_violation,market_call_equiv_bound_violation,market_put_equiv_bound_violation,any_monotonicity_violation_touching_row,any_convexity_violation_touching_row,any_calendar_violation_touching_row,calendar_nn_pair_count,calendar_nn_violation_pair_count
0,n10_to_n11_repair_queue,"rows requiring repair, downweighting, or diagn...",10_raw_static_arbitrage_diagnostics_for_spy_iv...,n10_20260705T200149Z,20260705T200149Z,705,2026-08-07 00:00:00+00:00,4,33.00000000,0.09041096,front_intermediate,call,741.00000000,744.78002930,747.33000000,0.99593977,-0.00850623,0.00850623,atm,0.14881934,0.00200235,16.62000000,16.62000000,10.31570124,16.62000000,16.62000000,10.31570124,2.42000000,0.14560770,0.18632154,0.95543873,0.70257705,single_side_stability_clean_fallback,unmatched_single_side,stability_clean,0,0,8,0,8,0.00000000,0.00000000,0.01111681,0.00000000,0.02429876,False,False,True,False,True,severe_violation,4,True,True,False,True,False,0.37025770,False,False,False,low_raw_static_arbitrage_score,2348,False,False,False,False,True,False,1,0
1,n10_to_n11_repair_queue,"rows requiring repair, downweighting, or diagn...",10_raw_static_arbitrage_diagnostics_for_spy_iv...,n10_20260705T200149Z,20260705T200149Z,707,2026-08-07 00:00:00+00:00,4,33.00000000,0.09041096,front_intermediate,put,743.00000000,744.78002930,747.33000000,0.99593977,-0.00581081,0.00581081,atm,0.14720264,0.00195908,11.06000000,15.37241921,11.06000000,11.06000000,15.37241921,11.06000000,0.06000000,0.00542495,0.00461934,1.00000000,0.99393211,matched_pair_preferred_side,pair_clean,stability_clean,0,0,8,0,8,0.00000000,0.00000000,0.01104238,0.00000000,0.02429876,False,False,True,False,True,severe_violation,4,True,False,False,True,False,0.49206139,True,False,False,included_broad_convexity_warning,2496,False,False,False,False,True,False,2,0
2,n10_to_n11_repair_queue,"rows requiring repair, downweighting, or diagn...",10_raw_static_arbitrage_diagnostics_for_spy_iv...,n10_20260705T200149Z,20260705T200149Z,266,2026-07-17 00:00:00+00:00,1,12.00000000,0.03287671,short,call,781.00000000,744.78002930,745.51500000,0.99852164,0.04649990,0.04649990,near_atm_call,0.11114022,0.00040610,0.05500000,0.05500000,35.48754046,0.05500000,0.05500000,35.48754046,0.01000000,0.18181818,0.02343110,0.44867573,0.69714686,matched_pair_preferred_side,not_both_stability_clean,stability_clean,0,0,8,0,8,0.00000000,0.00000000,0.00489900,0.00000000,0.01000000,False,False,True,False,True,severe_violation,4,True,False,False,True,False,0.39628359,False,False,False,low_raw_static_arbitrage_score,1246,False,False,False,False,True,False,1,0
3,n10_to_n11_repair_queue,"rows requiring repair, downweighting, or diagn...",10

,handoff_name,role,rows,unique_expiries,min_dte_calendar,max_dte_calendar,median_score,static_flagged_rows,iv_roughness_rows
0,n10_full_static_diagnostic_handoff,complete scored raw surface,1333,8,5.00000000,75.00000000,0.95922487,392,327
1,n10_to_n11_broad_handoff,broad Notebook 11 input,1260,8,5.00000000,75.00000000,0.96252924,319,258
2,n10_to_n11_strict_handoff,strict Notebook 11 input,224,8,5.00000000,75.00000000,0.99055544,0,0
3,n10_to_n11_anchor_handoff,near-ATM anchor input,196,8,5.00000000,75.00000000,0.99094693,0,0
4,n10_to_n11_repair_queue,repair / diagnostic queue,518,8,5.00000000,75.00000000,0.51113466,392,327
5,n10_to_n11_excluded_handoff,excluded from broad input,73,6,5.00000000,75.00000000,0.41654677,73,69


In [12]:
# ============================================================
# Final Notebook 10 validation, manifest, and conclusion
# ============================================================
# This cell closes Notebook 10.
#
# It does not alter the surface, repair quotes, fit a model, or change any
# Notebook 11 handoff. It only consolidates the audit trail and records the
# final status of the raw empirical surface.


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def json_safe_value(value: Any) -> Any:
    """
    Convert common pandas / numpy / pathlib objects into JSON-safe values.
    """
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (pd.Timestamp, datetime)):
        if pd.isna(value):
            return None
        return value.isoformat()

    if isinstance(value, np.datetime64):
        if np.isnat(value):
            return None
        return pd.Timestamp(value).isoformat()

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        return value

    if isinstance(value, dict):
        return {str(k): json_safe_value(v) for k, v in value.items()}

    if isinstance(value, (list, tuple, set)):
        return [json_safe_value(v) for v in value]

    return value


def json_safe_records(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """
    Convert a DataFrame to JSON-safe records.
    """
    return [
        {str(k): json_safe_value(v) for k, v in row.items()}
        for row in df.to_dict(orient="records")
    ]


def collect_validation_ledgers() -> pd.DataFrame:
    """
    Collect all validation ledgers created throughout Notebook 10.
    """
    candidate_names = [
        "input_validation_ledger",
        "coordinate_validation_ledger",
        "pricing_validation_ledger",
        "pointwise_validation_ledger",
        "monotonicity_validation_ledger",
        "convexity_validation_ledger",
        "calendar_validation_ledger",
        "iv_context_validation_ledger",
        "static_scoring_validation_ledger",
        "handoff_validation_ledger",
    ]

    frames = []

    for name in candidate_names:
        obj = globals().get(name)

        if isinstance(obj, pd.DataFrame) and {"check", "passed", "details"}.issubset(obj.columns):
            tmp = obj.copy()
            tmp.insert(0, "ledger_name", name)
            frames.append(tmp)

    if not frames:
        return pd.DataFrame(columns=["ledger_name", "check", "passed", "details"])

    out = pd.concat(frames, ignore_index=True)
    out["passed"] = out["passed"].astype(bool)

    return out


def infer_final_n10_status() -> Tuple[str, str]:
    """
    Infer final status from the completed diagnostics and handoff availability.
    """
    has_broad = "n10_to_n11_broad_handoff" in globals() and len(n10_to_n11_broad_handoff) > 0
    has_strict = "n10_to_n11_strict_handoff" in globals() and len(n10_to_n11_strict_handoff) > 0
    has_static_flags = (
        "surface_scored" in globals()
        and "has_any_static_arbitrage_flag" in surface_scored.columns
        and bool(surface_scored["has_any_static_arbitrage_flag"].any())
    )

    has_pointwise_or_mono_or_calendar = False

    if "surface_scored" in globals():
        for col in ["has_pointwise_violation", "has_monotonicity_violation", "has_calendar_violation"]:
            if col in surface_scored.columns and bool(surface_scored[col].any()):
                has_pointwise_or_mono_or_calendar = True

    if has_broad and has_strict and has_static_flags and not has_pointwise_or_mono_or_calendar:
        return (
            "READY_FOR_11_WITH_WARNINGS",
            "Notebook 11 can proceed using the broad handoff, but convexity warnings must be treated as repair/downweighting inputs.",
        )

    if has_broad and has_strict:
        return (
            "READY_FOR_11_STRICT_SUBSET_ONLY",
            "Notebook 11 should begin from the strict handoff because broad-surface diagnostics contain non-convex static-arbitrage risks.",
        )

    if has_broad:
        return (
            "READY_FOR_11_WITH_WARNINGS",
            "Notebook 11 can proceed only with the broad handoff; no strict subset was available.",
        )

    return (
        "NOT_READY_FOR_11",
        "Notebook 11 should not proceed because no usable handoff was produced.",
    )


# ------------------------------------------------------------
# Consolidate validation trail
# ------------------------------------------------------------

combined_validation_ledger = collect_validation_ledgers()

final_status, final_status_reason = infer_final_n10_status()

all_ledgers_passed = (
    len(combined_validation_ledger) > 0
    and combined_validation_ledger["passed"].all()
)

final_validation_ledger = pd.DataFrame(
    [
        validation_row(
            "all_validation_ledgers_collected",
            len(combined_validation_ledger) > 0,
            f"{len(combined_validation_ledger):,} validation checks collected",
        ),
        validation_row(
            "all_collected_validation_checks_passed",
            all_ledgers_passed,
            f"{int(combined_validation_ledger['passed'].sum()):,} / {len(combined_validation_ledger):,} checks passed",
        ),
        validation_row(
            "final_status_is_valid",
            final_status in FINAL_STATUS_OPTIONS,
            final_status,
        ),
        validation_row(
            "notebook_11_broad_handoff_available",
            "n10_to_n11_broad_handoff" in globals() and len(n10_to_n11_broad_handoff) > 0,
            f"{len(n10_to_n11_broad_handoff):,} broad rows" if "n10_to_n11_broad_handoff" in globals() else "missing",
        ),
        validation_row(
            "notebook_11_strict_handoff_available",
            "n10_to_n11_strict_handoff" in globals() and len(n10_to_n11_strict_handoff) > 0,
            f"{len(n10_to_n11_strict_handoff):,} strict rows" if "n10_to_n11_strict_handoff" in globals() else "missing",
        ),
        validation_row(
            "repair_queue_available",
            "n10_to_n11_repair_queue" in globals() and len(n10_to_n11_repair_queue) > 0,
            f"{len(n10_to_n11_repair_queue):,} repair / diagnostic rows" if "n10_to_n11_repair_queue" in globals() else "missing",
        ),
    ]
)

combined_validation_ledger_with_final = pd.concat(
    [
        combined_validation_ledger,
        final_validation_ledger.assign(ledger_name="final_validation_ledger")[
            ["ledger_name", "check", "passed", "details"]
        ],
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Final diagnostic conclusion
# ------------------------------------------------------------

final_static_arbitrage_conclusion = {
    "final_status": final_status,
    "final_status_reason": final_status_reason,
    "raw_surface_rows": int(len(surface_scored)) if "surface_scored" in globals() else None,
    "expiry_count": int(surface_scored["expiry"].nunique()) if "surface_scored" in globals() and "expiry" in surface_scored.columns else None,
    "broad_handoff_rows": int(len(n10_to_n11_broad_handoff)) if "n10_to_n11_broad_handoff" in globals() else None,
    "strict_handoff_rows": int(len(n10_to_n11_strict_handoff)) if "n10_to_n11_strict_handoff" in globals() else None,
    "anchor_handoff_rows": int(len(n10_to_n11_anchor_handoff)) if "n10_to_n11_anchor_handoff" in globals() else None,
    "repair_queue_rows": int(len(n10_to_n11_repair_queue)) if "n10_to_n11_repair_queue" in globals() else None,
    "excluded_rows": int(len(n10_to_n11_excluded_handoff)) if "n10_to_n11_excluded_handoff" in globals() else None,
    "pointwise_flagged_rows": int(surface_scored["has_pointwise_violation"].sum()) if "surface_scored" in globals() and "has_pointwise_violation" in surface_scored.columns else None,
    "monotonicity_flagged_rows": int(surface_scored["has_monotonicity_violation"].sum()) if "surface_scored" in globals() and "has_monotonicity_violation" in surface_scored.columns else None,
    "convexity_flagged_rows": int(surface_scored["has_convexity_violation"].sum()) if "surface_scored" in globals() and "has_convexity_violation" in surface_scored.columns else None,
    "calendar_flagged_rows": int(surface_scored["has_calendar_violation"].sum()) if "surface_scored" in globals() and "has_calendar_violation" in surface_scored.columns else None,
    "iv_roughness_context_rows": int(surface_scored["touched_by_any_iv_roughness_flag"].sum()) if "surface_scored" in globals() and "touched_by_any_iv_roughness_flag" in surface_scored.columns else None,
    "median_raw_static_arbitrage_score": float(surface_scored["raw_static_arbitrage_score"].median()) if "surface_scored" in globals() and "raw_static_arbitrage_score" in surface_scored.columns else None,
    "minimum_raw_static_arbitrage_score": float(surface_scored["raw_static_arbitrage_score"].min()) if "surface_scored" in globals() and "raw_static_arbitrage_score" in surface_scored.columns else None,
    "maximum_convexity_violation_amount": float(surface_scored["max_convexity_violation_amount"].max()) if "surface_scored" in globals() and "max_convexity_violation_amount" in surface_scored.columns else None,
    "main_empirical_finding": (
        "The filtered SPY IV observations pass pointwise, strike-monotonicity, and total-variance calendar diagnostics, "
        "but the raw surface still contains butterfly-convexity violations. Notebook 11 should therefore treat the broad "
        "handoff as an empirical candidate requiring arbitrage-aware fitting or repair, and use the strict handoff as the "
        "first clean calibration subset."
    ),
}


final_status_table = pd.DataFrame([final_static_arbitrage_conclusion])


# ------------------------------------------------------------
# Persist final tables and manifest
# ------------------------------------------------------------

final_table_objects = {
    "combined_validation_ledger": combined_validation_ledger_with_final,
    "final_validation_ledger": final_validation_ledger,
    "final_status_table": final_status_table,
}

final_table_manifest_rows = []

for object_name, df in final_table_objects.items():
    versioned_csv_path = TABLE_DIR / f"{SNAPSHOT_TAG}_{object_name}.csv"
    latest_csv_path = TABLE_DIR / f"latest_{object_name}.csv"

    df.to_csv(versioned_csv_path, index=False)
    df.to_csv(latest_csv_path, index=False)

    final_table_manifest_rows.extend(
        [
            {
                "object_name": object_name,
                "file_role": "versioned_csv",
                "path": str(versioned_csv_path),
                "rows": len(df),
                "columns": len(df.columns),
            },
            {
                "object_name": object_name,
                "file_role": "latest_csv",
                "path": str(latest_csv_path),
                "rows": len(df),
                "columns": len(df.columns),
            },
        ]
    )

final_table_manifest = pd.DataFrame(final_table_manifest_rows)

RUN_METADATA["final_status"] = final_status
RUN_METADATA["final_status_reason"] = final_status_reason
RUN_METADATA["final_static_arbitrage_conclusion"] = final_static_arbitrage_conclusion
RUN_METADATA["combined_validation_summary"] = {
    "checks_collected": int(len(combined_validation_ledger_with_final)),
    "checks_passed": int(combined_validation_ledger_with_final["passed"].sum()),
    "checks_failed": int((~combined_validation_ledger_with_final["passed"]).sum()),
}
RUN_METADATA["final_table_manifest"] = final_table_manifest.to_dict(orient="records")

manifest_payload = {
    "run_metadata": json_safe_value(RUN_METADATA),
    "final_status": final_status,
    "final_status_reason": final_status_reason,
    "final_static_arbitrage_conclusion": json_safe_value(final_static_arbitrage_conclusion),
    "handoff_inventory": json_safe_records(handoff_inventory) if "handoff_inventory" in globals() else [],
    "handoff_file_manifest": json_safe_records(handoff_file_manifest) if "handoff_file_manifest" in globals() else [],
    "final_table_manifest": json_safe_records(final_table_manifest),
    "validation_summary": json_safe_value(RUN_METADATA["combined_validation_summary"]),
    "failed_validation_checks": json_safe_records(
        combined_validation_ledger_with_final.loc[~combined_validation_ledger_with_final["passed"]]
    ),
}

manifest_path = MANIFEST_DIR / f"{SNAPSHOT_TAG}_notebook_10_manifest.json"
latest_manifest_path = MANIFEST_DIR / "latest_notebook_10_manifest.json"

manifest_path.write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")
latest_manifest_path.write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")


# ------------------------------------------------------------
# Display closeout
# ------------------------------------------------------------

print("=" * 90)
print("Notebook 10 final validation ledger")
print("=" * 90)
display(final_validation_ledger)

print("=" * 90)
print("Combined validation summary")
print("=" * 90)
display(
    pd.DataFrame(
        [
            {
                "checks_collected": len(combined_validation_ledger_with_final),
                "checks_passed": int(combined_validation_ledger_with_final["passed"].sum()),
                "checks_failed": int((~combined_validation_ledger_with_final["passed"]).sum()),
                "all_checks_passed": bool(combined_validation_ledger_with_final["passed"].all()),
            }
        ]
    )
)

if (~combined_validation_ledger_with_final["passed"]).any():
    print("=" * 90)
    print("Failed validation checks")
    print("=" * 90)
    display(combined_validation_ledger_with_final.loc[~combined_validation_ledger_with_final["passed"]])

print("=" * 90)
print("Notebook 10 final status")
print("=" * 90)
display(final_status_table)

print("=" * 90)
print("Final table manifest")
print("=" * 90)
display(final_table_manifest)

print("=" * 90)
print("Notebook 10 manifest paths")
print("=" * 90)
display(
    pd.DataFrame(
        [
            {
                "manifest_role": "versioned_manifest",
                "path": str(manifest_path),
            },
            {
                "manifest_role": "latest_manifest",
                "path": str(latest_manifest_path),
            },
        ]
    )
)

print("=" * 90)
print("Notebook 10 conclusion")
print("=" * 90)
print(final_static_arbitrage_conclusion["main_empirical_finding"])
print(f"\nFinal status: {final_status}")
print(f"Reason: {final_status_reason}")

final_status_table

Notebook 10 final validation ledger


,check,passed,details
0,all_validation_ledgers_collected,True,70 validation checks collected
1,all_collected_validation_checks_passed,False,66 / 70 checks passed
2,final_status_is_valid,True,READY_FOR_11_WITH_WARNINGS
3,notebook_11_broad_handoff_available,True,"1,260 broad rows"
4,notebook_11_strict_handoff_available,True,224 strict rows
5,repair_queue_available,True,518 repair / diagnostic rows


Combined validation summary


,checks_collected,checks_passed,checks_failed,all_checks_passed
0,76,71,5,False


Failed validation checks


...

,object_name,file_role,path,rows,columns
0,combined_validation_ledger,versioned_csv,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,76,4
1,combined_validation_ledger,latest_csv,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,76,4
2,final_validation_ledger,versioned_csv,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,6,3
3,final_validation_ledger,latest_csv,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,6,3
4,final_status_table,versioned_csv,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,1,18
5,final_status_table,latest_csv,d:\Derivative Pricing Project v1.0+\V1.1\outpu...,1,18


Notebook 10 manifest paths


,manifest_role,path
0,versioned_manifest,d:\Derivative Pricing Project v1.0+\V1.1\outpu...
1,latest_manifest,d:\Derivative Pricing Project v1.0+\V1.1\outpu...


Notebook 10 conclusion
The filtered SPY IV observations pass pointwise, strike-monotonicity, and total-variance calendar diagnostics, but the raw surface still contains butterfly-convexity violations. Notebook 11 should therefore treat the broad handoff as an empirical candidate requiring arbitrage-aware fitting or repair, and use the strict handoff as the first clean calibration subset.

Final status: READY_FOR_11_WITH_WARNINGS
Reason: Notebook 11 can proceed using the broad handoff, but convexity warnings must be treated as repair/downweighting inputs.


,final_status,final_status_reason,raw_surface_rows,expiry_count,broad_handoff_rows,strict_handoff_rows,anchor_handoff_rows,repair_queue_rows,excluded_rows,pointwise_flagged_rows,monotonicity_flagged_rows,convexity_flagged_rows,calendar_flagged_rows,iv_roughness_context_rows,median_raw_static_arbitrage_score,minimum_raw_static_arbitrage_score,maximum_convexity_violation_amount,main_empirical_finding
0,READY_FOR_11_WITH_WARNINGS,Notebook 11 can proceed using the broad handof...,1333,8,1260,224,196,518,73,0,0,392,0,327,0.95922487,0.37025770,0.01611432,The filtered SPY IV observations pass pointwis...
